# Figure 5 and Supplementary Figures S14–S15: RBP-associated RNA processing

This notebook reproduces the RBP analysis underlying Figure 5A–N, the Reactome overlap-class context in S14A–B, and the CPTAC counterparts in S15A–H. All RBP analysis and plotting functions are embedded below; running it does not import project scripts or execute another notebook.

## Inputs and reproducibility

The starting inputs are the processed expression rhythmicity/phase tables, APA phase and host-membership tables, filtered skipped-exon events and their rhythmicity results, and TPM/PDUI/PSI matrices used by the original analysis. These are downstream of RNA, DaPars2, and rMATS processing; `process.smk` does not itself produce every input to this notebook. Earlier CHIRAL and general APA/SE preprocessing are outside this notebook's input boundary. Input paths, file hashes, parameters, and software versions are recorded alongside the new results.

Consensus CLIP links are treated as a processed binding-evidence snapshot by default. Their fingerprint records which snapshot was used; it does not establish that the historical cache matches a changed raw POSTAR3 file. The embedded CLIP functions also support rebuilding those links from the raw reference. Reactome and CLIP are external annotations, and the regulatory-site proxy uses the existing TargetScan-derived miRNA burden table. Supply the files documented in the companion README when distributing this notebook.

## Analysis conventions

The analysis preserves the original RBP catalogue, rhythmicity criteria, enrichment backgrounds, phase permutations, event selection, and exploratory sample-level models. RBP expression rhythmicity requires nominal p < 0.05 and BH q < 0.05. Other filtering choices are exposed in the embedded functions and exported parameters. Gene symbols and version-stripped Ensembl gene/transcript identifiers are mapped explicitly; phases are in hours on a 24-hour cycle. Circular densities wrap at 0/24 h and the paired dashed guides show the second-moment axis.

CLIP is external binding support rather than binding measured in these cohorts. Phase differences and sample-level usage models are associations, not evidence of a causal RBP→event→mRNA sequence. The usage-model p-values preserve the original exploratory procedure; observations share samples and events. Site availability is an annotation-derived proxy, not a direct accessibility measurement. Reactome nominal top terms provide descriptive context when their FDR is not significant. CPTAC shares GTEx and reference annotations with the TCGA analysis.

## Publication outputs

Figures use 52.5-mm modules, global font roles with a 7-pt minimum, consistent colors, and minimal padding. Each of the 24 panels and three assemblies is exported as PNG and vector PDF under `data/figures/publication/fig05_rbp_processing/` and displayed inline. PNGs omit panel letters; PDFs include them. Analysis and plotted source tables go to `data/publication/fig05_rbp_processing/`. Supplementary numbering here is S14/S15; the original RBP notebook used S12/S13.


### Refresh relative to the reference screenshots

The historical renderer preferred stale aliases for some evidence, sample-model, regional-support, and site-availability tables. This notebook reads regenerated canonical tables. Figure 5K/L/M and the corresponding supplementary evidence/phase-module panels can therefore have different rows from the screenshots. The current processed APA input also changes 39 of 9,236 rows relative to the older stored phase snapshot; the top deep-priority ordering remains unchanged. See the companion README for the audit and interpretation.


## 1. Setup and publication configuration

In [ ]:
# Cell 1: Setup, paths, reproducibility, and global publication style
from pathlib import Path
import os
import sys
import json
import hashlib
import importlib.metadata
import platform
import time
import warnings
import numpy as np
import pandas as pd
import scipy
import scipy.stats
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

warnings.filterwarnings('ignore', category=FutureWarning)
USE_REVIEWED_CLIP_SNAPSHOT = True  # False rebuilds consensus links from raw POSTAR3.
PUBLICATION_SEED = 42
np.random.seed(PUBLICATION_SEED)
_started_at = time.perf_counter()
WORKSPACE_ROOT = Path(os.environ.get('WORKSPACE_ROOT', '/home/wdeng3/workspace/Codespace'))
candidates = [Path(os.environ['CHRONO_PROJECT_ROOT'])] if os.environ.get('CHRONO_PROJECT_ROOT') else [Path.cwd(), Path.cwd().parent, WORKSPACE_ROOT / 'PDAC_Rhythmicity']
PROJECT_ROOT = next((p.resolve() for p in candidates if (p / 'data').is_dir() and (p / 'notebook_publication').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Set CHRONO_PROJECT_ROOT to the repository containing data/ and notebook_publication/.')
INPUT_ROOT = PROJECT_ROOT / 'data'
ANALYSIS_SLUG = 'fig05_rbp_processing'
TABLE_DIR = PROJECT_ROOT / 'data' / 'publication' / ANALYSIS_SLUG
FIGURE_DIR = PROJECT_ROOT / 'data' / 'figures' / 'publication' / ANALYSIS_SLUG
LOG_DIR = PROJECT_ROOT / 'data' / 'logs' / 'publication' / ANALYSIS_SLUG
for directory in [TABLE_DIR, FIGURE_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# One global role-based font configuration shared by all panels.
PANEL_MM = 210 / 4
FONT = dict(base=8, title=9, axis_title=8, tick=7, legend=7, annotation=7, tag=10, subtitle=7)
assert min(FONT.values()) >= 7
mpl.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': FONT['base'],
    'axes.titlesize': FONT['title'], 'axes.labelsize': FONT['axis_title'],
    'xtick.labelsize': FONT['tick'], 'ytick.labelsize': FONT['tick'],
    'legend.fontsize': FONT['legend'], 'legend.title_fontsize': FONT['legend'],
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 110, 'savefig.dpi': 300, 'pdf.fonttype': 42, 'ps.fonttype': 42,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
})
print('Publication output:', TABLE_DIR.relative_to(PROJECT_ROOT))
print('Module:', PANEL_MM, 'mm; minimum text:', min(FONT.values()), 'pt')


## 2. Embedded analysis and processed inputs

This notebook regenerates the RBP--APA/SE source tables from processed cohort-level rhythmicity, APA, and SE inputs. It does not refit CHIRAL or rMATS. The analysis retains the reviewed thresholds: nominal p < 0.05 for the canonical SE event set, 500 lag permutations, and 2,000 UTR-rank permutations.

The setup cell defines `PROJECT_ROOT`, `INPUT_ROOT`, `TABLE_DIR`, `FIGURE_DIR`, and `LOG_DIR`. All tables generated here are written only to `TABLE_DIR`; Figure 5/S14/S15 plotting is performed in later cells. Primary binding evidence is consensus CLIP overlap. The default consumes the existing reviewed consensus-link snapshot to preserve current numerics, records its SHA-256 and QC, and labels it a processed snapshot rather than a freshness-verified raw-CLIP rebuild.


In [ ]:
# Cell 2: Imports and namespace facades
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Mapping, Sequence
import hashlib
import math
import os
import re
import time
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import fisher_exact
warnings.filterwarnings('ignore', category=FutureWarning)
np.random.seed(42)

class _Namespace:
    pass
base, source, deep, site = _Namespace(), _Namespace(), _Namespace(), _Namespace()
ANALYSIS_SLUG = 'publication/fig05_rbp_processing'
assert INPUT_ROOT == PROJECT_ROOT / 'data', 'Embedded source loaders use PROJECT_ROOT / data; set INPUT_ROOT accordingly.'
FIG05_REACTOME_FILE = Path(os.environ.get('FIG05_REACTOME_FILE', INPUT_ROOT / 'Ref' / 'Reactome_anno_HS.txt')).resolve()
assert TABLE_DIR == PROJECT_ROOT / 'data' / ANALYSIS_SLUG
assert FIGURE_DIR == PROJECT_ROOT / 'data' / 'figures' / ANALYSIS_SLUG
for _path in (TABLE_DIR, FIGURE_DIR, LOG_DIR):
    _path.mkdir(parents=True, exist_ok=True)


In [ ]:
# Cell 3: Embedded and namespace-isolated from scripts/rbp_rna_processing_links.py
"""RBP-centered links among APA, skipped exons, and rhythmic expression.

This module supports the PDAC_Rhythmicity RBP extension notebook. It keeps the
binding-evidence logic explicit: primary RBP-target claims should use
CLIP-supported gene links or peak overlaps, while motif-only evidence remains
outside this core analysis.
"""
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Mapping, Sequence
import numpy as np
import pandas as pd
try:
    from scipy.stats import fisher_exact
except Exception:
    fisher_exact = None
base_ANALYSIS_SLUG = 'rbp_rna_processing_links'
base_EXPRESSION_COHORTS = {'TCGA_Tumor': 'TCGA-PAAD_Tumor', 'CPTAC_Tumor': 'CPTAC-3_Tumor', 'GTEx_Normal': 'GTEx_Normal'}
base_COMPARISONS = {'TCGA_vs_GTEx': {'tumor_key': 'TCGA_Tumor', 'normal_key': 'GTEx_Normal', 'tumor_label': 'TCGA-PAAD Tumor', 'normal_label': 'GTEx Normal'}, 'CPTAC_vs_GTEx': {'tumor_key': 'CPTAC_Tumor', 'normal_key': 'GTEx_Normal', 'tumor_label': 'CPTAC-3 Tumor', 'normal_label': 'GTEx Normal'}}
base_RBP_COLUMNS = ['gene_symbol', 'ensembl_id', 'rbp_family', 'functional_class', 'clip_available']
base_CLIP_LINK_COLUMNS = ['rbp_symbol', 'target_gene_symbol', 'target_ensg', 'event_id', 'region', 'source', 'evidence_type', 'genome_build', 'chrom', 'peak_start', 'peak_end', 'method', 'sample', 'accession', 'score', 'clip_file']
base_CLIP_SUPPORT_COLUMNS = ['n_merged_peaks', 'n_accessions', 'n_methods', 'n_samples', 'max_score', 'median_score', 'peak_width_bp', 'consensus_filter']
base_CLIP_ALL_LINK_COLUMNS = base_CLIP_LINK_COLUMNS + base_CLIP_SUPPORT_COLUMNS
base_CONSENSUS_SCORE_THRESHOLD = 20.0
base_CONSENSUS_MIN_ACCESSIONS = 2
base_CONSENSUS_MIN_PEAKS = 2
base_CONSENSUS_MERGE_DISTANCE_BP = 20

@dataclass(frozen=True)
class base_RbpRecord:
    gene_symbol: str
    rbp_family: str
    functional_class: str

def base_strip_version(value: object) -> str:
    """Return an Ensembl-like identifier without version suffix."""
    if pd.isna(value):
        return ''
    text = str(value).strip().strip('"')
    if not text:
        return ''
    return text.split('.')[0]

def base_standardize_symbol(value: object) -> str:
    """Normalize a gene or RBP symbol for joins."""
    if pd.isna(value):
        return ''
    return str(value).strip().strip('"').upper()

def base_standardize_event_id(value: object) -> str:
    """Normalize event IDs that may be read as ints, floats, or strings."""
    if pd.isna(value):
        return ''
    text = str(value).strip().strip('"')
    if text.lower() in {'', 'nan', 'none', '<na>'}:
        return ''
    if text.endswith('.0'):
        prefix = text[:-2]
        if prefix.isdigit():
            return prefix
    return text

def base_ensure_output_dirs(project_root: Path, analysis_slug: str=base_ANALYSIS_SLUG) -> dict[str, Path]:
    """Create and return analysis output, figure, and log directories."""
    project_root = Path(project_root)
    dirs = {'output': project_root / 'data' / analysis_slug, 'figures': project_root / 'data' / 'figures' / analysis_slug, 'logs': project_root / 'data' / 'logs' / analysis_slug}
    for path in dirs.values():
        path.mkdir(parents=True, exist_ok=True)
    return dirs

def base_write_tsv(df: pd.DataFrame, path: Path) -> Path:
    """Write a TSV with parent directory creation and return its path."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, sep='\t', index=False)
    return path

def base__records_to_catalog(records: Sequence[base_RbpRecord]) -> pd.DataFrame:
    rows = [{'gene_symbol': base_standardize_symbol(r.gene_symbol), 'ensembl_id': '', 'rbp_family': r.rbp_family, 'functional_class': r.functional_class, 'clip_available': False} for r in records]
    catalog = pd.DataFrame(rows).drop_duplicates('gene_symbol')
    return catalog.sort_values(['functional_class', 'gene_symbol']).reset_index(drop=True)

def base_default_rbp_catalog() -> pd.DataFrame:
    """Return a curated RBP catalog covering splicing, APA, 3'UTR, and miRNA layers."""
    records = [base_RbpRecord('SRSF1', 'SR protein', 'splicing'), base_RbpRecord('SRSF2', 'SR protein', 'splicing'), base_RbpRecord('SRSF3', 'SR protein', 'splicing'), base_RbpRecord('SRSF4', 'SR protein', 'splicing'), base_RbpRecord('SRSF5', 'SR protein', 'splicing'), base_RbpRecord('SRSF6', 'SR protein', 'splicing'), base_RbpRecord('SRSF7', 'SR protein', 'splicing'), base_RbpRecord('SRSF8', 'SR protein', 'splicing'), base_RbpRecord('SRSF9', 'SR protein', 'splicing'), base_RbpRecord('SRSF10', 'SR protein', 'splicing'), base_RbpRecord('SRSF11', 'SR protein', 'splicing'), base_RbpRecord('SRSF12', 'SR protein', 'splicing'), base_RbpRecord('HNRNPA0', 'hnRNP', 'splicing'), base_RbpRecord('HNRNPA1', 'hnRNP', 'splicing'), base_RbpRecord('HNRNPA2B1', 'hnRNP', 'splicing'), base_RbpRecord('HNRNPA3', 'hnRNP', 'splicing'), base_RbpRecord('HNRNPAB', 'hnRNP', 'splicing'), base_RbpRecord('HNRNPC', 'hnRNP', 'splicing'), base_RbpRecord('HNRNPD', 'hnRNP', 'splicing'), base_RbpRecord('HNRNPDL', 'hnRNP', 'splicing'), base_RbpRecord('HNRNPF', 'hnRNP', 'splicing'), base_RbpRecord('HNRNPH1', 'hnRNP', 'splicing'), base_RbpRecord('HNRNPH2', 'hnRNP', 'splicing'), base_RbpRecord('HNRNPH3', 'hnRNP', 'splicing'), base_RbpRecord('HNRNPK', 'hnRNP', 'splicing'), base_RbpRecord('HNRNPL', 'hnRNP', 'splicing'), base_RbpRecord('HNRNPLL', 'hnRNP', 'splicing'), base_RbpRecord('HNRNPM', 'hnRNP', 'splicing'), base_RbpRecord('HNRNPR', 'hnRNP', 'splicing'), base_RbpRecord('HNRNPU', 'hnRNP', 'splicing'), base_RbpRecord('RBFOX1', 'RRM', 'splicing'), base_RbpRecord('RBFOX2', 'RRM', 'splicing'), base_RbpRecord('RBFOX3', 'RRM', 'splicing'), base_RbpRecord('PTBP1', 'RRM', 'splicing'), base_RbpRecord('PTBP2', 'RRM', 'splicing'), base_RbpRecord('PTBP3', 'RRM', 'splicing'), base_RbpRecord('QKI', 'STAR/KH', 'splicing'), base_RbpRecord('NOVA1', 'KH', 'splicing'), base_RbpRecord('NOVA2', 'KH', 'splicing'), base_RbpRecord('MBNL1', 'zinc finger', 'splicing'), base_RbpRecord('MBNL2', 'zinc finger', 'splicing'), base_RbpRecord('MBNL3', 'zinc finger', 'splicing'), base_RbpRecord('CELF1', 'CELF', 'splicing'), base_RbpRecord('CELF2', 'CELF', 'splicing'), base_RbpRecord('CELF3', 'CELF', 'splicing'), base_RbpRecord('CELF4', 'CELF', 'splicing'), base_RbpRecord('CELF5', 'CELF', 'splicing'), base_RbpRecord('CELF6', 'CELF', 'splicing'), base_RbpRecord('ESRP1', 'RRM', 'splicing'), base_RbpRecord('ESRP2', 'RRM', 'splicing'), base_RbpRecord('TRA2A', 'SR-related', 'splicing'), base_RbpRecord('TRA2B', 'SR-related', 'splicing'), base_RbpRecord('U2AF1', 'spliceosome', 'splicing'), base_RbpRecord('U2AF2', 'spliceosome', 'splicing'), base_RbpRecord('SF3B1', 'spliceosome', 'splicing'), base_RbpRecord('SF3B2', 'spliceosome', 'splicing'), base_RbpRecord('SF3B3', 'spliceosome', 'splicing'), base_RbpRecord('SF3B4', 'spliceosome', 'splicing'), base_RbpRecord('PRPF3', 'spliceosome', 'splicing'), base_RbpRecord('PRPF4', 'spliceosome', 'splicing'), base_RbpRecord('PRPF6', 'spliceosome', 'splicing'), base_RbpRecord('PRPF8', 'spliceosome', 'splicing'), base_RbpRecord('PRPF18', 'spliceosome', 'splicing'), base_RbpRecord('PRPF19', 'spliceosome', 'splicing'), base_RbpRecord('PRPF31', 'spliceosome', 'splicing'), base_RbpRecord('PRPF38A', 'spliceosome', 'splicing'), base_RbpRecord('PRPF38B', 'spliceosome', 'splicing'), base_RbpRecord('SNRPA', 'snRNP', 'splicing'), base_RbpRecord('SNRPB', 'snRNP', 'splicing'), base_RbpRecord('SNRPC', 'snRNP', 'splicing'), base_RbpRecord('SNRPE', 'snRNP', 'splicing'), base_RbpRecord('SNRPF', 'snRNP', 'splicing'), base_RbpRecord('SNRPG', 'snRNP', 'splicing'), base_RbpRecord('ELAVL1', 'ELAV/Hu', '3utr_stability_translation'), base_RbpRecord('ELAVL2', 'ELAV/Hu', '3utr_stability_translation'), base_RbpRecord('ELAVL3', 'ELAV/Hu', '3utr_stability_translation'), base_RbpRecord('ELAVL4', 'ELAV/Hu', '3utr_stability_translation'), base_RbpRecord('IGF2BP1', 'IGF2BP', '3utr_stability_translation'), base_RbpRecord('IGF2BP2', 'IGF2BP', '3utr_stability_translation'), base_RbpRecord('IGF2BP3', 'IGF2BP', '3utr_stability_translation'), base_RbpRecord('PUM1', 'Pumilio', '3utr_stability_translation'), base_RbpRecord('PUM2', 'Pumilio', '3utr_stability_translation'), base_RbpRecord('ZFP36', 'TTP', '3utr_stability_translation'), base_RbpRecord('ZFP36L1', 'TTP', '3utr_stability_translation'), base_RbpRecord('ZFP36L2', 'TTP', '3utr_stability_translation'), base_RbpRecord('TIA1', 'TIA/TIAR', '3utr_stability_translation'), base_RbpRecord('TIAL1', 'TIA/TIAR', '3utr_stability_translation'), base_RbpRecord('CPEB1', 'CPEB', '3utr_stability_translation'), base_RbpRecord('CPEB2', 'CPEB', '3utr_stability_translation'), base_RbpRecord('CPEB3', 'CPEB', '3utr_stability_translation'), base_RbpRecord('CPEB4', 'CPEB', '3utr_stability_translation'), base_RbpRecord('MSI1', 'Musashi', '3utr_stability_translation'), base_RbpRecord('MSI2', 'Musashi', '3utr_stability_translation'), base_RbpRecord('LIN28A', 'LIN28', '3utr_stability_translation'), base_RbpRecord('LIN28B', 'LIN28', '3utr_stability_translation'), base_RbpRecord('SYNCRIP', 'hnRNP', '3utr_stability_translation'), base_RbpRecord('PCBP1', 'PCBP', '3utr_stability_translation'), base_RbpRecord('PCBP2', 'PCBP', '3utr_stability_translation'), base_RbpRecord('PCBP3', 'PCBP', '3utr_stability_translation'), base_RbpRecord('PCBP4', 'PCBP', '3utr_stability_translation'), base_RbpRecord('AGO1', 'Argonaute', 'mirna_effector'), base_RbpRecord('AGO2', 'Argonaute', 'mirna_effector'), base_RbpRecord('AGO3', 'Argonaute', 'mirna_effector'), base_RbpRecord('AGO4', 'Argonaute', 'mirna_effector'), base_RbpRecord('TNRC6A', 'GW182', 'mirna_effector'), base_RbpRecord('TNRC6B', 'GW182', 'mirna_effector'), base_RbpRecord('TNRC6C', 'GW182', 'mirna_effector'), base_RbpRecord('DICER1', 'miRNA processing', 'mirna_effector'), base_RbpRecord('DROSHA', 'miRNA processing', 'mirna_effector'), base_RbpRecord('DGCR8', 'miRNA processing', 'mirna_effector'), base_RbpRecord('TARBP1', 'dsRBD', 'mirna_effector'), base_RbpRecord('TARBP2', 'dsRBD', 'mirna_effector'), base_RbpRecord('CPSF1', 'cleavage/polyadenylation', 'apa_machinery'), base_RbpRecord('CPSF2', 'cleavage/polyadenylation', 'apa_machinery'), base_RbpRecord('CPSF3', 'cleavage/polyadenylation', 'apa_machinery'), base_RbpRecord('CPSF4', 'cleavage/polyadenylation', 'apa_machinery'), base_RbpRecord('CPSF6', 'cleavage/polyadenylation', 'apa_machinery'), base_RbpRecord('CPSF7', 'cleavage/polyadenylation', 'apa_machinery'), base_RbpRecord('CSTF1', 'cleavage/polyadenylation', 'apa_machinery'), base_RbpRecord('CSTF2', 'cleavage/polyadenylation', 'apa_machinery'), base_RbpRecord('CSTF3', 'cleavage/polyadenylation', 'apa_machinery'), base_RbpRecord('NUDT21', 'CFIm', 'apa_machinery'), base_RbpRecord('FIP1L1', 'cleavage/polyadenylation', 'apa_machinery'), base_RbpRecord('PAPOLA', 'poly(A) polymerase', 'apa_machinery'), base_RbpRecord('PAPOLG', 'poly(A) polymerase', 'apa_machinery'), base_RbpRecord('PCF11', 'cleavage/polyadenylation', 'apa_machinery'), base_RbpRecord('CLP1', 'cleavage/polyadenylation', 'apa_machinery'), base_RbpRecord('SYMPK', 'cleavage/polyadenylation', 'apa_machinery'), base_RbpRecord('PABPN1', 'poly(A)-binding', 'apa_machinery'), base_RbpRecord('PABPC1', 'poly(A)-binding', '3utr_stability_translation'), base_RbpRecord('PABPC4', 'poly(A)-binding', '3utr_stability_translation'), base_RbpRecord('DDX1', 'DEAD-box helicase', 'rna_processing'), base_RbpRecord('DDX3X', 'DEAD-box helicase', 'rna_processing'), base_RbpRecord('DDX5', 'DEAD-box helicase', 'rna_processing'), base_RbpRecord('DDX6', 'DEAD-box helicase', 'rna_processing'), base_RbpRecord('DDX17', 'DEAD-box helicase', 'rna_processing'), base_RbpRecord('DDX21', 'DEAD-box helicase', 'rna_processing'), base_RbpRecord('DDX39A', 'DEAD-box helicase', 'rna_processing'), base_RbpRecord('DDX39B', 'DEAD-box helicase', 'rna_processing'), base_RbpRecord('DDX42', 'DEAD-box helicase', 'rna_processing'), base_RbpRecord('DDX46', 'DEAD-box helicase', 'rna_processing'), base_RbpRecord('DDX47', 'DEAD-box helicase', 'rna_processing'), base_RbpRecord('DDX50', 'DEAD-box helicase', 'rna_processing'), base_RbpRecord('DDX54', 'DEAD-box helicase', 'rna_processing'), base_RbpRecord('DDX56', 'DEAD-box helicase', 'rna_processing'), base_RbpRecord('DHX9', 'DEAH-box helicase', 'rna_processing'), base_RbpRecord('DHX15', 'DEAH-box helicase', 'rna_processing'), base_RbpRecord('DHX30', 'DEAH-box helicase', 'rna_processing'), base_RbpRecord('DHX36', 'DEAH-box helicase', 'rna_processing'), base_RbpRecord('LARP1', 'La-related', '3utr_stability_translation'), base_RbpRecord('LARP4', 'La-related', '3utr_stability_translation'), base_RbpRecord('LARP6', 'La-related', '3utr_stability_translation'), base_RbpRecord('LARP7', 'La-related', 'rna_processing'), base_RbpRecord('YBX1', 'cold-shock', 'rna_processing'), base_RbpRecord('RBM3', 'RBM', 'rna_processing'), base_RbpRecord('RBM4', 'RBM', 'splicing'), base_RbpRecord('RBM5', 'RBM', 'splicing'), base_RbpRecord('RBM6', 'RBM', 'splicing'), base_RbpRecord('RBM7', 'RBM', 'rna_processing'), base_RbpRecord('RBM8A', 'RBM', 'rna_processing'), base_RbpRecord('RBM10', 'RBM', 'splicing'), base_RbpRecord('RBM14', 'RBM', 'rna_processing'), base_RbpRecord('RBM15', 'RBM', 'rna_processing'), base_RbpRecord('RBM17', 'RBM', 'splicing'), base_RbpRecord('RBM20', 'RBM', 'splicing'), base_RbpRecord('RBM22', 'RBM', 'splicing'), base_RbpRecord('RBM24', 'RBM', 'splicing'), base_RbpRecord('RBM25', 'RBM', 'splicing'), base_RbpRecord('RBM26', 'RBM', 'splicing'), base_RbpRecord('RBM27', 'RBM', 'splicing'), base_RbpRecord('RBM28', 'RBM', 'rna_processing'), base_RbpRecord('RBM38', 'RBM', '3utr_stability_translation'), base_RbpRecord('RBM39', 'RBM', 'splicing'), base_RbpRecord('RBM41', 'RBM', 'rna_processing'), base_RbpRecord('RBM42', 'RBM', 'splicing'), base_RbpRecord('RBM45', 'RBM', 'rna_processing'), base_RbpRecord('RBM47', 'RBM', 'splicing'), base_RbpRecord('FUS', 'FET', 'rna_processing'), base_RbpRecord('TAF15', 'FET', 'rna_processing'), base_RbpRecord('EWSR1', 'FET', 'rna_processing'), base_RbpRecord('SFPQ', 'DBHS', 'rna_processing'), base_RbpRecord('NONO', 'DBHS', 'rna_processing'), base_RbpRecord('PSPC1', 'DBHS', 'rna_processing'), base_RbpRecord('MATR3', 'zinc finger', 'rna_processing'), base_RbpRecord('KHDRBS1', 'STAR/KH', 'splicing'), base_RbpRecord('KHDRBS2', 'STAR/KH', 'splicing'), base_RbpRecord('KHDRBS3', 'STAR/KH', 'splicing'), base_RbpRecord('FMR1', 'FMRP', '3utr_stability_translation'), base_RbpRecord('FXR1', 'FMRP', '3utr_stability_translation'), base_RbpRecord('FXR2', 'FMRP', '3utr_stability_translation'), base_RbpRecord('G3BP1', 'stress granule', '3utr_stability_translation'), base_RbpRecord('G3BP2', 'stress granule', '3utr_stability_translation'), base_RbpRecord('NCL', 'nucleolin', 'rna_processing'), base_RbpRecord('NPM1', 'nucleophosmin', 'rna_processing'), base_RbpRecord('CIRBP', 'cold-shock', '3utr_stability_translation')]
    return base__records_to_catalog(records)

def base_load_expression_tables(project_root: Path, cohorts: Mapping[str, str]=base_EXPRESSION_COHORTS) -> pd.DataFrame:
    """Load harmonic regression outputs for the requested cohorts."""
    rows = []
    base = Path(project_root) / 'data' / 'Merged' / 'regression_windowed'
    for cohort_key, cohort_token in cohorts.items():
        path = base / f'all_genes_regression_periodic_{cohort_token}.txt'
        if not path.exists():
            continue
        frame = pd.read_csv(path, sep='\\s+', engine='python')
        frame.columns = [str(c).strip().strip('"') for c in frame.columns]
        rename = {'Gene': 'gene_symbol', 'ENSG': 'ensg', 'Amplitude': 'amplitude', 'p': 'p_value', 'q': 'q_value', 'acrophase': 'acrophase_hours', 'Group': 'group'}
        frame = frame.rename(columns=rename)
        for col in ['gene_symbol', 'ensg', 'amplitude', 'p_value', 'q_value', 'acrophase_hours', 'group']:
            if col not in frame.columns:
                frame[col] = np.nan
        frame = frame[['group', 'gene_symbol', 'ensg', 'amplitude', 'p_value', 'q_value', 'acrophase_hours']].copy()
        frame['gene_symbol'] = frame['gene_symbol'].map(base_standardize_symbol)
        frame['ensg'] = frame['ensg'].map(base_strip_version)
        for col in ['amplitude', 'p_value', 'q_value', 'acrophase_hours']:
            frame[col] = pd.to_numeric(frame[col], errors='coerce')
        frame['cohort_key'] = cohort_key
        frame['cohort_label'] = cohort_token.replace('_', ' ')
        frame['source_path'] = str(path)
        rows.append(frame)
    if not rows:
        return pd.DataFrame(columns=['group', 'gene_symbol', 'ensg', 'amplitude', 'p_value', 'q_value', 'acrophase_hours', 'cohort_key', 'cohort_label', 'source_path'])
    return pd.concat(rows, ignore_index=True)

def base_map_catalog_to_expression(catalog: pd.DataFrame, expression: pd.DataFrame) -> pd.DataFrame:
    """Fill catalog Ensembl IDs from expression tables and keep CLIP availability boolean."""
    catalog = catalog.copy()
    catalog['gene_symbol'] = catalog['gene_symbol'].map(base_standardize_symbol)
    expr_map = expression.dropna(subset=['gene_symbol', 'ensg']).query("gene_symbol != '' and ensg != ''").drop_duplicates('gene_symbol').set_index('gene_symbol')['ensg'].to_dict()
    existing_ensg = catalog['ensembl_id'] if 'ensembl_id' in catalog.columns else pd.Series('', index=catalog.index)
    catalog['ensembl_id'] = existing_ensg.map(base_strip_version)
    catalog['ensembl_id'] = catalog.apply(lambda r: base_strip_version(r['ensembl_id']) or expr_map.get(r['gene_symbol'], ''), axis=1)
    if 'clip_available' not in catalog.columns:
        catalog['clip_available'] = False
    catalog['clip_available'] = catalog['clip_available'].fillna(False).astype(bool)
    return catalog[base_RBP_COLUMNS].sort_values('gene_symbol').reset_index(drop=True)

def base_annotate_rbp_rhythmicity(catalog: pd.DataFrame, expression: pd.DataFrame, p_threshold: float=0.05, q_threshold: float | None=0.05) -> pd.DataFrame:
    """Join RBP catalog to expression rhythmicity results by gene symbol."""
    catalog = base_map_catalog_to_expression(catalog, expression)
    rbp_symbols = set(catalog['gene_symbol'])
    frame = expression[expression['gene_symbol'].isin(rbp_symbols)].copy()
    frame = frame.merge(catalog, on='gene_symbol', how='left', suffixes=('', '_catalog'))
    if 'ensembl_id' in frame.columns:
        frame['ensembl_id'] = frame['ensembl_id'].where(frame['ensembl_id'].ne(''), frame['ensg'])
    else:
        frame['ensembl_id'] = frame['ensg']
    frame['p_rhythmic'] = frame['p_value'] < p_threshold
    if q_threshold is None:
        frame['q_rhythmic'] = pd.NA
        frame['rhythmic'] = frame['p_rhythmic']
        threshold_label = f'p<{p_threshold:g}'
    else:
        frame['q_rhythmic'] = frame['q_value'] < q_threshold
        frame['rhythmic'] = frame['p_rhythmic'] & frame['q_rhythmic']
        threshold_label = f'p<{p_threshold:g};q<{q_threshold:g}'
    frame['rhythmicity_threshold'] = threshold_label
    keep = ['cohort_key', 'cohort_label', 'gene_symbol', 'ensembl_id', 'rbp_family', 'functional_class', 'clip_available', 'amplitude', 'p_value', 'q_value', 'acrophase_hours', 'p_rhythmic', 'q_rhythmic', 'rhythmic', 'rhythmicity_threshold']
    return frame[keep].sort_values(['cohort_key', 'gene_symbol']).reset_index(drop=True)

def base_summarize_rbp_rhythmicity(rbp_expression: pd.DataFrame) -> pd.DataFrame:
    """Summarize RBP rhythmicity by cohort and by requested tumor-normal comparisons."""
    rows = []
    for cohort_key, grp in rbp_expression.groupby('cohort_key', dropna=False):
        rows.append({'summary_type': 'cohort', 'comparison': '', 'cohort_key': cohort_key, 'n_rbps_tested': int(grp['gene_symbol'].nunique()), 'n_rhythmic': int(grp.loc[grp['rhythmic'], 'gene_symbol'].nunique()), 'rhythmic_fraction': float(grp.loc[grp['rhythmic'], 'gene_symbol'].nunique() / max(grp['gene_symbol'].nunique(), 1)), 'rbp_set': ','.join(sorted(grp.loc[grp['rhythmic'], 'gene_symbol'].unique()))})
    for comparison, cfg in base_COMPARISONS.items():
        tumor = rbp_expression[rbp_expression['cohort_key'].eq(cfg['tumor_key'])]
        normal = rbp_expression[rbp_expression['cohort_key'].eq(cfg['normal_key'])]
        tumor_set = set(tumor.loc[tumor['rhythmic'], 'gene_symbol'])
        normal_set = set(normal.loc[normal['rhythmic'], 'gene_symbol'])
        rows.extend([{'summary_type': 'comparison_set', 'comparison': comparison, 'cohort_key': 'tumor_only', 'n_rbps_tested': int(len(set(tumor['gene_symbol']) | set(normal['gene_symbol']))), 'n_rhythmic': int(len(tumor_set - normal_set)), 'rhythmic_fraction': np.nan, 'rbp_set': ','.join(sorted(tumor_set - normal_set))}, {'summary_type': 'comparison_set', 'comparison': comparison, 'cohort_key': 'normal_only', 'n_rbps_tested': int(len(set(tumor['gene_symbol']) | set(normal['gene_symbol']))), 'n_rhythmic': int(len(normal_set - tumor_set)), 'rhythmic_fraction': np.nan, 'rbp_set': ','.join(sorted(normal_set - tumor_set))}, {'summary_type': 'comparison_set', 'comparison': comparison, 'cohort_key': 'shared', 'n_rbps_tested': int(len(set(tumor['gene_symbol']) | set(normal['gene_symbol']))), 'n_rhythmic': int(len(tumor_set & normal_set)), 'rhythmic_fraction': np.nan, 'rbp_set': ','.join(sorted(tumor_set & normal_set))}])
    primary = set(rbp_expression.loc[rbp_expression['cohort_key'].eq('TCGA_Tumor') & rbp_expression['rhythmic'], 'gene_symbol'])
    validation = set(rbp_expression.loc[rbp_expression['cohort_key'].eq('CPTAC_Tumor') & rbp_expression['rhythmic'], 'gene_symbol'])
    rows.append({'summary_type': 'validation', 'comparison': 'TCGA_Tumor_and_CPTAC_Tumor', 'cohort_key': 'validation_replicated', 'n_rbps_tested': int(rbp_expression.loc[rbp_expression['cohort_key'].isin(['TCGA_Tumor', 'CPTAC_Tumor']), 'gene_symbol'].nunique()), 'n_rhythmic': int(len(primary & validation)), 'rhythmic_fraction': np.nan, 'rbp_set': ','.join(sorted(primary & validation))})
    return pd.DataFrame(rows)

def base_phase_to_radians(hours: object) -> np.ndarray:
    """Convert circadian phase hours to radians on [0, 2*pi)."""
    return pd.to_numeric(hours, errors='coerce').astype(float).to_numpy() % 24.0 / 24.0 * 2.0 * np.pi

def base_circular_lag_hours(target_phase: object, source_phase: object) -> np.ndarray:
    """Return target-source circular lag in hours, wrapped to [-12, 12]."""
    target = pd.to_numeric(target_phase, errors='coerce').astype(float)
    source = pd.to_numeric(source_phase, errors='coerce').astype(float)
    lag = (target - source + 12.0) % 24.0 - 12.0
    return lag.to_numpy()

def base_phase_summary(phases: Iterable[object]) -> dict[str, float]:
    """Summarize circular concentration of 0-24h phases."""
    values = pd.to_numeric(pd.Series(list(phases)), errors='coerce').dropna()
    values = values[(values >= 0) & (values <= 24)]
    n = int(values.size)
    if n == 0:
        return {'n': 0, 'mean_phase_hours': np.nan, 'resultant_length': np.nan, 'axial_r2': np.nan, 'axis_hours': np.nan, 'rayleigh_p_approx': np.nan}
    theta = values.to_numpy() / 24.0 * 2.0 * np.pi
    c = np.cos(theta).mean()
    s = np.sin(theta).mean()
    mean_angle = np.arctan2(s, c) % (2.0 * np.pi)
    resultant = float(np.hypot(c, s))
    axial_theta = 2.0 * theta
    ac = np.cos(axial_theta).mean()
    ass = np.sin(axial_theta).mean()
    axial_r2 = float(ac * ac + ass * ass)
    axis_angle = np.arctan2(ass, ac) / 2.0 % np.pi
    rayleigh_p = float(np.exp(-n * resultant * resultant))
    return {'n': n, 'mean_phase_hours': float(mean_angle / (2.0 * np.pi) * 24.0), 'resultant_length': resultant, 'axial_r2': axial_r2, 'axis_hours': float(axis_angle / (2.0 * np.pi) * 24.0), 'rayleigh_p_approx': rayleigh_p}

def base_summarize_phase_by_group(df: pd.DataFrame, group_cols: Sequence[str], phase_col: str='acrophase_hours') -> pd.DataFrame:
    """Return circular summaries for phase distributions by group."""
    rows = []
    if df.empty:
        return pd.DataFrame(columns=list(group_cols) + list(base_phase_summary([]).keys()))
    for keys, grp in df.groupby(list(group_cols), dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = dict(zip(group_cols, keys))
        row.update(base_phase_summary(grp[phase_col]))
        rows.append(row)
    return pd.DataFrame(rows)

def base__read_tsv_if_exists(path: Path) -> pd.DataFrame:
    if not Path(path).exists():
        return pd.DataFrame()
    return pd.read_csv(path, sep='\t')

def base_load_apa_phase_tables(project_root: Path) -> pd.DataFrame:
    """Load APA-mRNA phase-lag tables for TCGA-vs-GTEx and CPTAC-vs-GTEx."""
    base = Path(project_root) / 'data' / 'polyAPA' / 'comparison'
    paths = {'TCGA_vs_GTEx': base / 'apa_expression_phase_lag_tcga_vs_gtex.tsv', 'CPTAC_vs_GTEx': base / 'apa_expression_phase_lag_cptac_vs_gtex.tsv'}
    frames = []
    for comparison, path in paths.items():
        frame = base__read_tsv_if_exists(path)
        if frame.empty:
            continue
        frame['comparison'] = comparison
        frame['cohort_key'] = base_COMPARISONS[comparison]['tumor_key']
        frame['normal_key'] = 'GTEx_Normal'
        frame['host_ensg'] = frame.get('host_ensg', '').map(base_strip_version)
        for col in ['host_gene_symbol', 'expr_gene_symbol', 'apa_gene_symbol']:
            if col in frame.columns:
                frame[col] = frame[col].map(base_standardize_symbol)
        for col in ['apa_acrophase_hours', 'expr_acrophase_hours', 'phase_lag_hours', 'amplitude', 'relative_amplitude', 'p_value', 'fdr', 'expr_amplitude', 'expr_p', 'expr_q']:
            if col in frame.columns:
                frame[col] = pd.to_numeric(frame[col], errors='coerce')
        frames.append(frame)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

def base_load_apa_membership(project_root: Path) -> pd.DataFrame:
    """Load APA host-gene set membership tables for matched gene backgrounds."""
    base = Path(project_root) / 'data' / 'polyAPA' / 'comparison'
    paths = {'TCGA_vs_GTEx': base / 'apa_host_gene_set_membership_tcga_vs_gtex.tsv', 'CPTAC_vs_GTEx': base / 'apa_host_gene_set_membership_cptac_vs_gtex.tsv'}
    frames = []
    for comparison, path in paths.items():
        frame = base__read_tsv_if_exists(path)
        if frame.empty:
            continue
        frame['comparison'] = comparison
        frame['cohort_key'] = base_COMPARISONS[comparison]['tumor_key']
        frame['normal_key'] = 'GTEx_Normal'
        frame['host_ensg'] = frame.get('host_ensg', '').map(base_strip_version)
        for col in frame.columns:
            if col.endswith('host_gene_symbol') or col in {'host_gene_symbol'}:
                frame[col] = frame[col].map(base_standardize_symbol)
        if 'host_gene_symbol' in frame.columns:
            frame['host_gene_symbol'] = frame['host_gene_symbol'].map(base_standardize_symbol)
        for col in frame.columns:
            if col.endswith('_apa_rhythmic'):
                frame[col] = frame[col].map(lambda x: str(x).strip().lower() in {'true', '1', 'yes'})
        frames.append(frame)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

def base__load_se_universe_for_cohort(project_root: Path, cohort_dir: str) -> pd.DataFrame:
    path = Path(project_root) / 'data' / 'AS' / cohort_dir / 'filtered_events' / f'{cohort_dir}_SE_variable_AS.csv'
    if not path.exists():
        return pd.DataFrame()
    frame = pd.read_csv(path)
    rename = {'ID': 'event_id', 'GeneID': 'gene_id', 'geneSymbol': 'gene_symbol', 'exonStart_0base': 'exon_start', 'exonEnd': 'exon_end', 'upstreamES': 'upstream_start', 'upstreamEE': 'upstream_end', 'downstreamES': 'downstream_start', 'downstreamEE': 'downstream_end'}
    frame = frame.rename(columns=rename)
    for col in ['event_id', 'gene_id', 'gene_symbol', 'chr', 'strand']:
        if col not in frame.columns:
            frame[col] = ''
    for col in ['exon_start', 'exon_end', 'upstream_start', 'upstream_end', 'downstream_start', 'downstream_end']:
        if col in frame.columns:
            frame[col] = pd.to_numeric(frame[col], errors='coerce')
    frame['event_id'] = frame['event_id'].astype(str)
    frame['gene_id'] = frame['gene_id'].map(base_strip_version)
    frame['gene_symbol'] = frame['gene_symbol'].map(base_standardize_symbol)
    frame['event_type'] = 'SE'
    frame['cohort_dir'] = cohort_dir
    return frame
base_SE_CANONICAL_COHORTS = [('TCGA', 'TCGA_Tumor', 'TCGA_vs_GTEx'), ('CPTAC', 'CPTAC_Tumor', 'CPTAC_vs_GTEx')]

def base__read_se_canonical_rhythmicity(project_root: Path, cohort_dir: str, cohort_key: str) -> pd.DataFrame:
    """Read the canonical (notebook 3.1.2) per-cohort SE harmonic-regression results.

    Source: data/AS/<cohort_dir>/rhythmic_results/<cohort_key>/SE/all_results.csv, the
    same SE rhythmicity profile used elsewhere in the manuscript. Holds every tested SE
    event with its rhythmicity p-value, BH-FDR, and amplitude. This replaces the older
    genome-wide AS table, whose pooled-FDR correction collapsed the CPTAC SE set to ~1.
    """
    path = Path(project_root) / 'data' / 'AS' / cohort_dir / 'rhythmic_results' / cohort_key / 'SE' / 'all_results.csv'
    if not path.exists():
        return pd.DataFrame()
    frame = pd.read_csv(path)
    if 'event_id' not in frame.columns:
        return pd.DataFrame()
    frame['event_id'] = frame['event_id'].astype(str).map(base_standardize_event_id)
    for col in ['acrophase_hours', 'p_value', 'fdr', 'qvalue', 'amplitude', 'relative_amplitude']:
        if col in frame.columns:
            frame[col] = pd.to_numeric(frame[col], errors='coerce')
    if 'fdr' not in frame.columns and 'qvalue' in frame.columns:
        frame['fdr'] = frame['qvalue']
    return frame

def base_load_se_universe(project_root: Path) -> pd.DataFrame:
    """Load the SE event background restricted to the canonical 3.1.2 tested set.

    Coordinates come from the filtered variable-SE table; the universe is limited to the
    events notebook 3.1.2 actually tested for rhythmicity, so the enrichment background
    and the rhythmic positive set share one SE event profile per cohort.
    """
    frames = []
    for cohort_dir, cohort_key, comparison in base_SE_CANONICAL_COHORTS:
        frame = base__load_se_universe_for_cohort(project_root, cohort_dir)
        if frame.empty:
            continue
        frame['event_id'] = frame['event_id'].astype(str).map(base_standardize_event_id)
        canonical = base__read_se_canonical_rhythmicity(project_root, cohort_dir, cohort_key)
        if not canonical.empty:
            tested = set(canonical['event_id'])
            frame = frame[frame['event_id'].isin(tested)].copy()
        frame['cohort_key'] = cohort_key
        frame['comparison'] = comparison
        frames.append(frame)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

def base_load_se_events(project_root: Path, p_threshold: float=0.05, fdr_threshold: float | None=None) -> pd.DataFrame:
    """Load rhythmic SE events from the canonical 3.1.2 per-cohort SE results.

    The rhythmic positive set matches notebook 3.1.2 (nominal p < ``p_threshold``) unless
    ``fdr_threshold`` is supplied, in which case BH-FDR within SE is used instead.
    Coordinates are merged from the filtered SE universe by event_id.
    """
    universe = base_load_se_universe(project_root)
    if universe.empty:
        return pd.DataFrame()
    coord_cols = ['event_id', 'gene_id', 'gene_symbol', 'chr', 'strand', 'exon_start', 'exon_end', 'upstream_start', 'upstream_end', 'downstream_start', 'downstream_end']
    frames = []
    for cohort_dir, cohort_key, comparison in base_SE_CANONICAL_COHORTS:
        canonical = base__read_se_canonical_rhythmicity(project_root, cohort_dir, cohort_key)
        if canonical.empty:
            continue
        if fdr_threshold is not None and 'fdr' in canonical.columns:
            rhythmic = canonical[canonical['fdr'].le(fdr_threshold)].copy()
        else:
            rhythmic = canonical[canonical['p_value'].lt(p_threshold)].copy()
        coords = universe.loc[universe['cohort_dir'].eq(cohort_dir), [c for c in coord_cols if c in universe.columns]].drop_duplicates('event_id')
        rhythmic = rhythmic.drop(columns=[c for c in ['gene_id', 'gene_symbol', 'chr', 'strand'] if c in rhythmic.columns])
        rhythmic = rhythmic.merge(coords, on='event_id', how='left')
        rhythmic['event_type'] = 'SE'
        rhythmic['cohort_key'] = cohort_key
        rhythmic['comparison'] = comparison
        frames.append(rhythmic)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)
base_CPTAC_SE_THRESHOLD_SOURCES = {'genome_wide': {'path': ('data', 'AS', 'CPTAC', 'rhythmic_results', 'genome_wide_rhythmic_AS_all.csv'), 'label': 'Genome-wide AS FDR', 'detail': 'Current main SE loader source; FDR adjusted across genome-wide AS calls.'}, 'se_specific': {'path': ('data', 'AS', 'CPTAC', 'rhythmic_results', 'SE', 'CPTAC_SE_rhythmic_AS_all.csv'), 'label': 'SE-specific FDR', 'detail': 'Event-type-specific SE rhythmicity table; FDR adjusted within SE calls.'}, 'cptac_tumor_se': {'path': ('data', 'AS', 'CPTAC', 'rhythmic_results', 'CPTAC_Tumor', 'SE', 'all_results.csv'), 'label': 'CPTAC tumor SE', 'detail': 'CPTAC_Tumor SE rhythmicity table; CPTAC normal is not read.'}}
base_CPTAC_SE_THRESHOLD_SPECS = [{'threshold_label': 'FDR<=0.10', 'threshold_family': 'fdr', 'fdr_le': 0.1}, {'threshold_label': 'FDR<=0.15', 'threshold_family': 'fdr', 'fdr_le': 0.15}, {'threshold_label': 'FDR<=0.20', 'threshold_family': 'fdr', 'fdr_le': 0.2}, {'threshold_label': 'FDR<=0.25', 'threshold_family': 'fdr', 'fdr_le': 0.25}, {'threshold_label': 'p<0.05', 'threshold_family': 'p_value_only', 'p_lt': 0.05}, {'threshold_label': 'p<0.10', 'threshold_family': 'p_value_only', 'p_lt': 0.1}, {'threshold_label': 'p<0.20', 'threshold_family': 'p_value_only', 'p_lt': 0.2}, {'threshold_label': 'p<0.10+rAMP>=0.15', 'threshold_family': 'p_value_amplitude', 'p_lt': 0.1, 'relative_amplitude_ge': 0.15}, {'threshold_label': 'p<0.10+amp>=0.05', 'threshold_family': 'p_value_amplitude', 'p_lt': 0.1, 'amplitude_ge': 0.05}, {'threshold_label': 'p<0.10+PSIrange>=0.20', 'threshold_family': 'p_value_psi', 'p_lt': 0.1, 'psi_range_ge': 0.2}, {'threshold_label': 'p<0.10+amp>=0.05+PSI', 'threshold_family': 'p_value_amplitude_psi', 'p_lt': 0.1, 'amplitude_ge': 0.05, 'psi_range_ge': 0.2, 'mean_psi_between': (0.05, 0.95)}, {'threshold_label': 'p<0.20+rAMP>=0.15+PSI', 'threshold_family': 'p_value_amplitude_psi', 'p_lt': 0.2, 'relative_amplitude_ge': 0.15, 'psi_range_ge': 0.2, 'mean_psi_between': (0.05, 0.95)}]

def base__read_cptac_se_threshold_source(project_root: Path, source_key: str, universe: pd.DataFrame) -> pd.DataFrame:
    """Read one CPTAC tumor SE rhythmicity source and align it to the SE universe."""
    source = base_CPTAC_SE_THRESHOLD_SOURCES[source_key]
    path = Path(project_root).joinpath(*source['path'])
    if not path.exists():
        return pd.DataFrame()
    frame = pd.read_csv(path)
    event_col = 'event_id' if 'event_id' in frame.columns else 'ID' if 'ID' in frame.columns else ''
    if not event_col:
        return pd.DataFrame()
    frame['event_id'] = frame[event_col].astype(str).map(base_standardize_event_id)
    for col in ['gene_id', 'gene_symbol']:
        if col not in frame.columns:
            frame[col] = ''
    frame['gene_id'] = frame['gene_id'].map(base_strip_version)
    frame['gene_symbol'] = frame['gene_symbol'].map(base_standardize_symbol)
    numeric_cols = ['acrophase_hours', 'p_value', 'fdr', 'qvalue', 'amplitude', 'relative_amplitude', 'mean_psi', 'sd_psi', 'min_psi', 'max_psi', 'psi_range', 'r_squared']
    for col in numeric_cols:
        if col in frame.columns:
            frame[col] = pd.to_numeric(frame[col], errors='coerce')
    if 'fdr' not in frame.columns and 'qvalue' in frame.columns:
        frame['fdr'] = frame['qvalue']
    if 'qvalue' not in frame.columns and 'fdr' in frame.columns:
        frame['qvalue'] = frame['fdr']
    coords = universe[universe['cohort_dir'].eq('CPTAC')][['event_id', 'gene_id', 'gene_symbol', 'chr', 'strand', 'exon_start', 'exon_end', 'upstream_start', 'upstream_end', 'downstream_start', 'downstream_end']].copy()
    coords['event_id'] = coords['event_id'].astype(str).map(base_standardize_event_id)
    coords = coords.drop_duplicates('event_id')
    se_ids = set(coords['event_id'])
    frame = frame[frame['event_id'].isin(se_ids)].copy()
    drop_cols = [c for c in ['gene_id', 'gene_symbol', 'chr', 'strand', 'exonStart', 'exonEnd'] if c in frame.columns]
    frame = frame.drop(columns=drop_cols).merge(coords, on='event_id', how='left')
    frame['event_type'] = 'SE'
    frame['cohort_dir'] = 'CPTAC'
    frame['cohort_key'] = 'CPTAC_Tumor'
    frame['comparison'] = 'CPTAC_vs_GTEx'
    frame['threshold_source'] = source_key
    frame['threshold_source_label'] = source['label']
    frame['threshold_source_detail'] = source['detail']
    frame['threshold_source_path'] = str(path)
    return frame

def base__threshold_mask(frame: pd.DataFrame, spec: Mapping[str, object]) -> pd.Series:
    """Return a boolean mask for one rhythmicity threshold specification."""
    mask = pd.Series(True, index=frame.index)
    if 'fdr_le' in spec:
        mask &= pd.to_numeric(frame.get('fdr', pd.Series(np.nan, index=frame.index)), errors='coerce').le(float(spec['fdr_le']))
    if 'p_lt' in spec:
        mask &= pd.to_numeric(frame.get('p_value', pd.Series(np.nan, index=frame.index)), errors='coerce').lt(float(spec['p_lt']))
    if 'relative_amplitude_ge' in spec:
        mask &= pd.to_numeric(frame.get('relative_amplitude', pd.Series(np.nan, index=frame.index)), errors='coerce').ge(float(spec['relative_amplitude_ge']))
    if 'amplitude_ge' in spec:
        mask &= pd.to_numeric(frame.get('amplitude', pd.Series(np.nan, index=frame.index)), errors='coerce').ge(float(spec['amplitude_ge']))
    if 'psi_range_ge' in spec:
        mask &= pd.to_numeric(frame.get('psi_range', pd.Series(np.nan, index=frame.index)), errors='coerce').ge(float(spec['psi_range_ge']))
    if 'mean_psi_between' in spec:
        lower, upper = spec['mean_psi_between']
        values = pd.to_numeric(frame.get('mean_psi', pd.Series(np.nan, index=frame.index)), errors='coerce')
        mask &= values.between(float(lower), float(upper))
    return mask.fillna(False)

def base_load_cptac_se_threshold_sensitivity(project_root: Path, sources: Sequence[str]=('genome_wide', 'se_specific', 'cptac_tumor_se'), threshold_specs: Sequence[Mapping[str, object]] | None=None) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load CPTAC tumor SE event sets across relaxed rhythmicity thresholds.

    The returned event table contains one row per selected event and threshold.
    The summary table records source-specific event counts and filter metadata.
    """
    universe = base_load_se_universe(project_root)
    universe = universe[universe['cohort_dir'].eq('CPTAC')].copy()
    if universe.empty:
        return (pd.DataFrame(), pd.DataFrame())
    threshold_specs = list(threshold_specs or base_CPTAC_SE_THRESHOLD_SPECS)
    event_frames = []
    summary_rows = []
    for source_key in sources:
        if source_key not in base_CPTAC_SE_THRESHOLD_SOURCES:
            continue
        frame = base__read_cptac_se_threshold_source(project_root, source_key, universe)
        if frame.empty:
            continue
        for rank, spec in enumerate(threshold_specs, start=1):
            label = str(spec['threshold_label'])
            mask = base__threshold_mask(frame, spec)
            selected = frame[mask].copy()
            selected['threshold_rank'] = rank
            selected['threshold_label'] = label
            selected['threshold_family'] = str(spec.get('threshold_family', 'custom'))
            selected['threshold_p_lt'] = spec.get('p_lt', np.nan)
            selected['threshold_fdr_le'] = spec.get('fdr_le', np.nan)
            selected['threshold_relative_amplitude_ge'] = spec.get('relative_amplitude_ge', np.nan)
            selected['threshold_amplitude_ge'] = spec.get('amplitude_ge', np.nan)
            selected['threshold_psi_range_ge'] = spec.get('psi_range_ge', np.nan)
            selected['threshold_mean_psi_between'] = f"{spec['mean_psi_between'][0]}-{spec['mean_psi_between'][1]}" if 'mean_psi_between' in spec else ''
            if not selected.empty:
                event_frames.append(selected)
            summary_rows.append({'threshold_source': source_key, 'threshold_source_label': base_CPTAC_SE_THRESHOLD_SOURCES[source_key]['label'], 'threshold_source_detail': base_CPTAC_SE_THRESHOLD_SOURCES[source_key]['detail'], 'threshold_rank': rank, 'threshold_label': label, 'threshold_family': str(spec.get('threshold_family', 'custom')), 'n_universe_events': int(frame['event_id'].nunique()), 'n_events': int(selected['event_id'].nunique()), 'n_genes': int(selected['gene_symbol'].nunique()) if 'gene_symbol' in selected else 0, 'median_p_value': float(selected['p_value'].median()) if 'p_value' in selected and selected['p_value'].notna().any() else np.nan, 'median_fdr': float(selected['fdr'].median()) if 'fdr' in selected and selected['fdr'].notna().any() else np.nan, 'median_amplitude': float(selected['amplitude'].median()) if 'amplitude' in selected and selected['amplitude'].notna().any() else np.nan, 'median_relative_amplitude': float(selected['relative_amplitude'].median()) if 'relative_amplitude' in selected and selected['relative_amplitude'].notna().any() else np.nan, 'median_psi_range': float(selected['psi_range'].median()) if 'psi_range' in selected and selected['psi_range'].notna().any() else np.nan, 'source_path': str(Path(project_root).joinpath(*base_CPTAC_SE_THRESHOLD_SOURCES[source_key]['path']))})
    events = pd.concat(event_frames, ignore_index=True, sort=False) if event_frames else pd.DataFrame()
    summary = pd.DataFrame(summary_rows)
    if not summary.empty:
        summary = summary.sort_values(['threshold_source', 'threshold_rank']).reset_index(drop=True)
    return (events, summary)

def base_discover_clip_files(project_root: Path, clip_dir: Path | None=None) -> pd.DataFrame:
    """Return a manifest for CLIP/eCLIP/POSTAR files under data/Ref/clip_rbp."""
    clip_dir = Path(clip_dir) if clip_dir is not None else Path(project_root) / 'data' / 'Ref' / 'clip_rbp'
    rows = []
    if not clip_dir.exists():
        return pd.DataFrame([{'clip_dir': str(clip_dir), 'clip_file': '', 'file_type': 'missing_directory', 'source_guess': '', 'genome_build_guess': '', 'n_rows': 0, 'status': 'CLIP directory not found; CLIP-supported analyses are gated off.'}])
    suffixes = {'.bed', '.gz', '.tsv', '.csv', '.txt', '.narrowpeak', '.broadpeak'}
    for path in sorted(clip_dir.rglob('*')):
        if not path.is_file():
            continue
        if path.suffix.lower() not in suffixes and (not any((str(path).lower().endswith(x) for x in ['.bed.gz', '.tsv.gz', '.txt.gz']))):
            continue
        lower = path.name.lower()
        source_guess = 'ENCODE_eCLIP' if 'encode' in lower or 'eclip' in lower else 'POSTAR3' if 'postar' in lower else 'CLIP'
        build_guess = 'hg38' if 'hg38' in lower or 'grch38' in lower else 'hg19' if 'hg19' in lower or 'grch37' in lower else ''
        rows.append({'clip_dir': str(clip_dir), 'clip_file': str(path), 'file_type': base__guess_clip_file_type(path), 'source_guess': source_guess, 'genome_build_guess': build_guess, 'n_rows': np.nan, 'status': 'discovered'})
    if not rows:
        rows.append({'clip_dir': str(clip_dir), 'clip_file': '', 'file_type': 'no_supported_files', 'source_guess': '', 'genome_build_guess': '', 'n_rows': 0, 'status': 'No supported CLIP files found; add POSTAR3/ENCODE files here.'})
    return pd.DataFrame(rows)

def base__guess_clip_file_type(path: Path) -> str:
    lower = str(path).lower()
    if 'postar3' in lower and lower.endswith(('.txt', '.txt.gz')):
        return 'peak_bed'
    if lower.endswith(('.bed', '.bed.gz', '.narrowpeak', '.broadpeak')):
        return 'peak_bed'
    return 'interaction_table'

def base__read_any_table(path: Path, max_rows: int | None=None) -> pd.DataFrame:
    lower = str(path).lower()
    compression = 'gzip' if lower.endswith('.gz') else None
    if lower.endswith('.csv') or lower.endswith('.csv.gz'):
        return pd.read_csv(path, nrows=max_rows, compression=compression)
    sep = '\t' if lower.endswith(('.tsv', '.tsv.gz', '.bed', '.bed.gz', '.txt', '.txt.gz', '.narrowpeak', '.broadpeak')) else None
    return pd.read_csv(path, sep=sep, nrows=max_rows, compression=compression, comment='#', engine='python')

def base__infer_rbp_from_filename(path: Path, catalog_symbols: set[str]) -> str:
    stem = Path(str(path).replace('.gz', '')).stem.upper()
    parts = stem.replace('-', '_').replace('.', '_').split('_')
    for part in parts:
        if part in catalog_symbols:
            return part
    for symbol in sorted(catalog_symbols, key=len, reverse=True):
        if symbol in stem:
            return symbol
    return ''

def base_load_clip_interactions(project_root: Path, catalog: pd.DataFrame, expression: pd.DataFrame, clip_dir: Path | None=None) -> pd.DataFrame:
    """Load CLIP-derived RBP-target interaction tables if present."""
    clip_dir = Path(clip_dir) if clip_dir is not None else Path(project_root) / 'data' / 'Ref' / 'clip_rbp'
    if not clip_dir.exists():
        return pd.DataFrame(columns=base_CLIP_LINK_COLUMNS)
    catalog_symbols = set(catalog['gene_symbol'].map(base_standardize_symbol))
    symbol_to_ensg = expression.dropna(subset=['gene_symbol', 'ensg']).assign(gene_symbol=lambda x: x['gene_symbol'].map(base_standardize_symbol), ensg=lambda x: x['ensg'].map(base_strip_version)).drop_duplicates('gene_symbol').set_index('gene_symbol')['ensg'].to_dict()
    rows = []
    for path in sorted(clip_dir.rglob('*')):
        if not path.is_file() or base__guess_clip_file_type(path) == 'peak_bed':
            continue
        try:
            table = base__read_any_table(path)
        except Exception:
            continue
        table.columns = [str(c).strip() for c in table.columns]
        lower_cols = {c.lower(): c for c in table.columns}
        rbp_col = base__pick_column(lower_cols, ['rbp', 'rbp_symbol', 'protein', 'clip_rbp', 'rbp_gene'])
        target_col = base__pick_column(lower_cols, ['target', 'target_gene', 'target_symbol', 'gene', 'gene_symbol', 'gene_name', 'genename', 'host_gene_symbol'])
        target_ensg_col = base__pick_column(lower_cols, ['target_ensg', 'target_ensembl', 'ensembl', 'gene_id', 'ensg'])
        region_col = base__pick_column(lower_cols, ['region', 'annotation', 'feature', 'binding_region'])
        build_col = base__pick_column(lower_cols, ['genome_build', 'genome', 'assembly'])
        if target_col is None and target_ensg_col is None:
            continue
        rbp_from_file = base__infer_rbp_from_filename(path, catalog_symbols)
        source_guess = 'POSTAR3' if 'postar' in path.name.lower() else 'ENCODE_eCLIP' if 'eclip' in path.name.lower() else 'CLIP'
        for _, row in table.iterrows():
            rbp = base_standardize_symbol(row[rbp_col]) if rbp_col else rbp_from_file
            if rbp not in catalog_symbols:
                continue
            target_symbol = base_standardize_symbol(row[target_col]) if target_col else ''
            target_ensg = base_strip_version(row[target_ensg_col]) if target_ensg_col else symbol_to_ensg.get(target_symbol, '')
            if not target_symbol and target_ensg:
                target_symbol = base__lookup_symbol_by_ensg(expression, target_ensg)
            if not target_symbol and (not target_ensg):
                continue
            region = str(row[region_col]).strip() if region_col else 'gene_level_clip'
            genome_build = str(row[build_col]).strip() if build_col else base__guess_genome_build_from_name(path.name)
            rows.append({'rbp_symbol': rbp, 'target_gene_symbol': target_symbol, 'target_ensg': target_ensg, 'event_id': '', 'region': region, 'source': source_guess, 'evidence_type': 'clip_interaction_table', 'genome_build': genome_build, 'clip_file': str(path)})
    if not rows:
        return pd.DataFrame(columns=base_CLIP_LINK_COLUMNS)
    return pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)

def base__pick_column(lower_cols: Mapping[str, str], aliases: Sequence[str]) -> str | None:
    for alias in aliases:
        if alias.lower() in lower_cols:
            return lower_cols[alias.lower()]
    return None

def base__lookup_symbol_by_ensg(expression: pd.DataFrame, ensg: str) -> str:
    hit = expression[expression['ensg'].map(base_strip_version).eq(base_strip_version(ensg))]
    if hit.empty:
        return ''
    return base_standardize_symbol(hit.iloc[0]['gene_symbol'])

def base__guess_genome_build_from_name(name: str) -> str:
    lower = name.lower()
    if 'hg38' in lower or 'grch38' in lower:
        return 'hg38'
    if 'hg19' in lower or 'grch37' in lower:
        return 'hg19'
    return ''

def base__iter_bed_like_clip_peak_chunks(path: Path, catalog_symbols: set[str], chunksize: int=1000000):
    """Yield normalized BED-like CLIP peak chunks."""
    lower = str(path).lower()
    compression = 'gzip' if lower.endswith('.gz') else None
    is_postar3 = 'postar3' in lower
    source = 'POSTAR3' if is_postar3 else 'ENCODE_eCLIP' if 'eclip' in lower or 'encode' in lower else 'CLIP_peak'
    genome_build = base__guess_genome_build_from_name(path.name)
    if is_postar3 and (not genome_build):
        genome_build = 'hg38'
    rbp_from_file = base__infer_rbp_from_filename(path, catalog_symbols)
    reader = pd.read_csv(path, sep='\t', header=None, comment='#', compression=compression, chunksize=chunksize, low_memory=False)
    for chunk in reader:
        if chunk.shape[1] < 3:
            continue
        peak = pd.DataFrame({'chrom': chunk.iloc[:, 0].astype(str), 'start': pd.to_numeric(chunk.iloc[:, 1], errors='coerce'), 'end': pd.to_numeric(chunk.iloc[:, 2], errors='coerce')})
        peak['strand'] = '.'
        peak['rbp_symbol'] = rbp_from_file
        peak['method'] = ''
        peak['sample'] = ''
        peak['accession'] = ''
        peak['score'] = np.nan
        if is_postar3 and chunk.shape[1] >= 10:
            peak['strand'] = chunk.iloc[:, 4].astype(str).where(chunk.iloc[:, 4].astype(str).isin(['+', '-']), '.')
            peak['rbp_symbol'] = chunk.iloc[:, 5].map(base_standardize_symbol)
            peak['method'] = chunk.iloc[:, 6].astype(str)
            peak['sample'] = chunk.iloc[:, 7].astype(str)
            peak['accession'] = chunk.iloc[:, 8].astype(str)
            peak['score'] = pd.to_numeric(chunk.iloc[:, 9], errors='coerce')
        else:
            if chunk.shape[1] >= 4:
                maybe_rbp = chunk.iloc[:, 3].map(base_standardize_symbol)
                peak['rbp_symbol'] = maybe_rbp.where(maybe_rbp.isin(catalog_symbols), peak['rbp_symbol'])
            if chunk.shape[1] >= 6:
                peak['strand'] = chunk.iloc[:, 5].astype(str).where(chunk.iloc[:, 5].astype(str).isin(['+', '-']), '.')
        peak['source'] = source
        peak['genome_build'] = genome_build
        peak['clip_file'] = str(path)
        peak = peak.dropna(subset=['start', 'end'])
        peak = peak[peak['rbp_symbol'].isin(catalog_symbols)]
        if not peak.empty:
            yield peak

def base__read_bed_like_clip_peaks(path: Path, catalog_symbols: set[str], chunksize: int=1000000) -> pd.DataFrame:
    """Read BED-like CLIP peaks, including POSTAR3's 10-column peak table."""
    frames = list(base__iter_bed_like_clip_peak_chunks(path, catalog_symbols, chunksize=chunksize))
    if not frames:
        return pd.DataFrame(columns=['chrom', 'start', 'end', 'rbp_symbol', 'strand', 'method', 'sample', 'accession', 'score', 'source', 'genome_build', 'clip_file'])
    return pd.concat(frames, ignore_index=True).drop_duplicates().reset_index(drop=True)

def base_load_clip_peaks(project_root: Path, catalog: pd.DataFrame, clip_dir: Path | None=None) -> pd.DataFrame:
    """Load BED-like CLIP peaks with normalized RBP symbols."""
    clip_dir = Path(clip_dir) if clip_dir is not None else Path(project_root) / 'data' / 'Ref' / 'clip_rbp'
    columns = ['chrom', 'start', 'end', 'rbp_symbol', 'strand', 'method', 'sample', 'accession', 'score', 'source', 'genome_build', 'clip_file']
    if not clip_dir.exists():
        return pd.DataFrame(columns=columns)
    catalog_symbols = set(catalog['gene_symbol'].map(base_standardize_symbol))
    frames = []
    for path in sorted(clip_dir.rglob('*')):
        if not path.is_file() or base__guess_clip_file_type(path) != 'peak_bed':
            continue
        try:
            peak = base__read_bed_like_clip_peaks(path, catalog_symbols)
        except Exception:
            continue
        if not peak.empty:
            frames.append(peak)
    if not frames:
        return pd.DataFrame(columns=columns)
    return pd.concat(frames, ignore_index=True).drop_duplicates().reset_index(drop=True)

def base_load_utr_bed(project_root: Path) -> pd.DataFrame:
    """Load extracted hg38 3'UTR intervals used for CLIP peak-to-gene links."""
    path = Path(project_root) / 'data' / 'Ref' / 'hg38_extracted_3UTR.bed'
    if not path.exists():
        return pd.DataFrame(columns=['chrom', 'start', 'end', 'target_gene_symbol', 'strand', 'region', 'target_ensg'])
    bed = pd.read_csv(path, sep='\t', header=None, names=['chrom', 'start', 'end', 'name', 'score', 'strand'])
    parts = bed['name'].astype(str).str.split('|', expand=True)
    bed['transcript_id'] = parts[0].fillna('')
    bed['target_gene_symbol'] = parts[1].fillna('').map(base_standardize_symbol)
    bed['target_gene_symbol'] = bed['target_gene_symbol'].replace('NA', '')
    bed['target_ensg'] = ''
    bed['region'] = '3UTR'
    return bed[['chrom', 'start', 'end', 'target_gene_symbol', 'target_ensg', 'strand', 'region', 'transcript_id']]

def base_make_se_overlap_intervals(se_events: pd.DataFrame, intron_window_bp: int=250) -> pd.DataFrame:
    """Create skipped-exon body and flanking intronic windows for CLIP overlap."""
    if se_events.empty:
        return pd.DataFrame(columns=['chrom', 'start', 'end', 'target_gene_symbol', 'target_ensg', 'event_id', 'strand', 'region'])
    rows = []
    for _, row in se_events.iterrows():
        chrom = row.get('chr', '')
        start = pd.to_numeric(row.get('exon_start'), errors='coerce')
        end = pd.to_numeric(row.get('exon_end'), errors='coerce')
        if pd.isna(start) or pd.isna(end) or end <= start:
            continue
        event_id = str(row.get('event_id', ''))
        target_symbol = base_standardize_symbol(row.get('gene_symbol', ''))
        target_ensg = base_strip_version(row.get('gene_id', ''))
        strand = str(row.get('strand', '.'))
        starts = [max(0, int(start) - intron_window_bp), int(start), int(end)]
        ends = [int(start), int(end), int(end) + intron_window_bp]
        labels = ['SE_upstream_flank', 'SE_exon_body', 'SE_downstream_flank']
        for interval_start, interval_end, label in zip(starts, ends, labels):
            if interval_end <= interval_start:
                continue
            rows.append({'chrom': chrom, 'start': interval_start, 'end': interval_end, 'target_gene_symbol': target_symbol, 'target_ensg': target_ensg, 'event_id': event_id, 'strand': strand, 'region': label, 'comparison': row.get('comparison', ''), 'cohort_key': row.get('cohort_key', '')})
    return pd.DataFrame(rows)

def base_overlap_clip_peaks_to_intervals(peaks: pd.DataFrame, intervals: pd.DataFrame) -> pd.DataFrame:
    """Overlap BED-like CLIP peaks with genomic intervals without requiring pybedtools."""
    if peaks.empty or intervals.empty:
        return pd.DataFrame(columns=base_CLIP_ALL_LINK_COLUMNS + ['cohort_key', 'comparison'])
    rows = []
    peak = peaks.copy()
    ints = intervals.copy()
    for frame in [peak, ints]:
        frame['chrom'] = frame['chrom'].astype(str)
        frame['start'] = pd.to_numeric(frame['start'], errors='coerce')
        frame['end'] = pd.to_numeric(frame['end'], errors='coerce')
    peak = peak.dropna(subset=['start', 'end'])
    ints = ints.dropna(subset=['start', 'end'])
    for chrom, chrom_peaks in peak.groupby('chrom'):
        chrom_intervals = ints[ints['chrom'].eq(chrom)]
        if chrom_intervals.empty:
            continue
        chrom_intervals = chrom_intervals.sort_values('start').reset_index(drop=True)
        interval_starts = chrom_intervals['start'].to_numpy()
        for _, p in chrom_peaks.iterrows():
            candidates = chrom_intervals.iloc[:np.searchsorted(interval_starts, p['end'], side='left')]
            candidates = candidates[candidates['end'].gt(p['start'])]
            if candidates.empty:
                continue
            for _, interval in candidates.iterrows():
                rows.append({'rbp_symbol': p['rbp_symbol'], 'target_gene_symbol': interval.get('target_gene_symbol', ''), 'target_ensg': base_strip_version(interval.get('target_ensg', '')), 'event_id': str(interval.get('event_id', '')), 'region': interval.get('region', ''), 'source': p.get('source', 'CLIP_peak'), 'evidence_type': 'clip_peak_overlap', 'genome_build': p.get('genome_build', ''), 'chrom': p.get('chrom', ''), 'peak_start': p.get('start', np.nan), 'peak_end': p.get('end', np.nan), 'method': p.get('method', ''), 'sample': p.get('sample', ''), 'accession': p.get('accession', ''), 'score': p.get('score', np.nan), 'clip_file': p.get('clip_file', ''), 'cohort_key': interval.get('cohort_key', ''), 'comparison': interval.get('comparison', '')})
    if not rows:
        return pd.DataFrame(columns=base_CLIP_ALL_LINK_COLUMNS + ['cohort_key', 'comparison'])
    return pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)

def base__expand_intervals_to_bins(intervals: pd.DataFrame, bin_size: int=50000) -> pd.DataFrame:
    """Expand target intervals into genomic bins for fast chunked peak overlap."""
    if intervals.empty:
        return pd.DataFrame()
    frame = intervals.copy()
    frame['chrom'] = frame['chrom'].astype(str)
    frame['start'] = pd.to_numeric(frame['start'], errors='coerce')
    frame['end'] = pd.to_numeric(frame['end'], errors='coerce')
    frame = frame.dropna(subset=['start', 'end'])
    frame = frame[frame['end'].gt(frame['start'])].copy()
    rows = []
    for idx, row in frame.reset_index(drop=True).iterrows():
        start_bin = int(row['start']) // bin_size
        end_bin = max(start_bin, (int(row['end']) - 1) // bin_size)
        for bin_id in range(start_bin, end_bin + 1):
            item = row.to_dict()
            item['interval_id'] = idx
            item['bin'] = bin_id
            rows.append(item)
    return pd.DataFrame(rows)

def base__expand_peaks_to_bins(peaks: pd.DataFrame, bin_size: int=50000) -> pd.DataFrame:
    """Expand peak chunks into bins; most CLIP peaks stay in one bin."""
    if peaks.empty:
        return pd.DataFrame()
    frame = peaks.copy()
    frame['start'] = pd.to_numeric(frame['start'], errors='coerce')
    frame['end'] = pd.to_numeric(frame['end'], errors='coerce')
    frame = frame.dropna(subset=['start', 'end'])
    frame = frame[frame['end'].gt(frame['start'])].copy()
    start_bins = (frame['start'].astype(int) // bin_size).to_numpy()
    end_bins = ((frame['end'].astype(int) - 1) // bin_size).to_numpy()
    if np.array_equal(start_bins, end_bins):
        frame['bin'] = start_bins
        return frame
    frame['start_bin'] = start_bins
    frame['end_bin'] = end_bins
    single = frame[frame['start_bin'].eq(frame['end_bin'])].copy()
    single['bin'] = single['start_bin']
    multi = frame[frame['start_bin'].ne(frame['end_bin'])].copy()
    if multi.empty:
        return single.drop(columns=['start_bin', 'end_bin'])
    lengths = (multi['end_bin'] - multi['start_bin'] + 1).astype(int).to_numpy()
    repeated = multi.iloc[np.repeat(np.arange(len(multi)), lengths)].copy()
    repeated['bin'] = np.concatenate([np.arange(start, end + 1, dtype=int) for start, end in zip(multi['start_bin'].astype(int), multi['end_bin'].astype(int))])
    return pd.concat([single, repeated], ignore_index=True, sort=False).drop(columns=['start_bin', 'end_bin'])

def base__deduplicate_clip_links(links: pd.DataFrame) -> pd.DataFrame:
    """Deduplicate peak-derived links while retaining the maximum available score."""
    if links.empty:
        return links
    for col in base_CLIP_ALL_LINK_COLUMNS + ['cohort_key', 'comparison']:
        if col not in links.columns:
            links[col] = '' if col != 'score' else np.nan
    numeric_cols = ['score', 'peak_start', 'peak_end', 'n_merged_peaks', 'n_accessions', 'n_methods', 'n_samples', 'max_score', 'median_score', 'peak_width_bp']
    for col in numeric_cols:
        if col in links.columns:
            links[col] = pd.to_numeric(links[col], errors='coerce')
    group_cols = [c for c in base_CLIP_ALL_LINK_COLUMNS + ['cohort_key', 'comparison'] if c != 'score']
    out = links.groupby(group_cols, dropna=False, as_index=False)['score'].max().sort_values(['rbp_symbol', 'region', 'target_gene_symbol', 'event_id']).reset_index(drop=True)
    return out

def base__join_unique(values: Iterable[object], max_items: int=12) -> str:
    """Join unique, non-missing values for compact evidence summaries."""
    cleaned = []
    for value in values:
        if pd.isna(value):
            continue
        text = str(value).strip()
        if not text or text in {'nan', 'None', '<NA>'}:
            continue
        cleaned.append(text)
    unique = sorted(set(cleaned))
    if len(unique) > max_items:
        return ','.join(unique[:max_items]) + f',...(+{len(unique) - max_items})'
    return ','.join(unique)

def base__finalize_consensus_peak(base: Mapping[str, object], segment: pd.DataFrame, score_threshold: float, min_accessions: int, min_peak_support: int) -> dict | None:
    """Return one merged peak row if it passes the consensus evidence rule."""
    scores = pd.to_numeric(segment['score'], errors='coerce')
    accessions = [str(x).strip() for x in segment.get('accession', pd.Series(dtype=str)) if not pd.isna(x) and str(x).strip() and (str(x).strip() not in {'nan', 'None', '<NA>'})]
    methods = [str(x).strip() for x in segment.get('method', pd.Series(dtype=str)) if not pd.isna(x) and str(x).strip() and (str(x).strip() not in {'nan', 'None', '<NA>'})]
    samples = [str(x).strip() for x in segment.get('sample', pd.Series(dtype=str)) if not pd.isna(x) and str(x).strip() and (str(x).strip() not in {'nan', 'None', '<NA>'})]
    n_accessions = len(set(accessions))
    n_methods = len(set(methods))
    n_samples = len(set(samples))
    max_score = float(scores.max()) if scores.notna().any() else np.nan
    median_score = float(scores.median()) if scores.notna().any() else np.nan
    n_peaks = int(len(segment))
    filters = []
    if n_accessions >= min_accessions:
        filters.append(f'accessions_ge_{min_accessions}')
    if np.isfinite(max_score) and max_score >= score_threshold:
        filters.append(f'score_ge_{score_threshold:g}')
    if n_accessions == 0 and n_peaks >= min_peak_support:
        filters.append(f'peaks_ge_{min_peak_support}')
    if not filters:
        return None
    start = int(pd.to_numeric(segment['peak_start'], errors='coerce').min())
    end = int(pd.to_numeric(segment['peak_end'], errors='coerce').max())
    row = dict(base)
    row.update({'evidence_type': 'clip_peak_consensus_overlap', 'peak_start': start, 'peak_end': end, 'method': base__join_unique(methods), 'sample': base__join_unique(samples), 'accession': base__join_unique(accessions), 'score': max_score, 'clip_file': base__join_unique(segment.get('clip_file', pd.Series(dtype=str)), max_items=4), 'n_merged_peaks': n_peaks, 'n_accessions': n_accessions, 'n_methods': n_methods, 'n_samples': n_samples, 'max_score': max_score, 'median_score': median_score, 'peak_width_bp': end - start, 'consensus_filter': ';'.join(filters)})
    return row

def base_build_consensus_clip_peak_links(raw_peak_links: pd.DataFrame, merge_distance_bp: int=base_CONSENSUS_MERGE_DISTANCE_BP, score_threshold: float=base_CONSENSUS_SCORE_THRESHOLD, min_accessions: int=base_CONSENSUS_MIN_ACCESSIONS, min_peak_support: int=base_CONSENSUS_MIN_PEAKS) -> pd.DataFrame:
    """Merge raw CLIP peak overlaps into high-confidence target-level binding segments."""
    if raw_peak_links.empty:
        return pd.DataFrame(columns=base_CLIP_ALL_LINK_COLUMNS + ['cohort_key', 'comparison'])
    links = raw_peak_links.copy()
    for col in base_CLIP_ALL_LINK_COLUMNS + ['cohort_key', 'comparison']:
        if col not in links.columns:
            links[col] = '' if col not in {'score', 'peak_start', 'peak_end'} else np.nan
    links['peak_start'] = pd.to_numeric(links['peak_start'], errors='coerce')
    links['peak_end'] = pd.to_numeric(links['peak_end'], errors='coerce')
    links['score'] = pd.to_numeric(links['score'], errors='coerce')
    links = links.dropna(subset=['peak_start', 'peak_end'])
    links = links[links['peak_end'].gt(links['peak_start'])].copy()
    if links.empty:
        return pd.DataFrame(columns=base_CLIP_ALL_LINK_COLUMNS + ['cohort_key', 'comparison'])
    text_cols = ['rbp_symbol', 'target_gene_symbol', 'target_ensg', 'event_id', 'region', 'source', 'genome_build', 'chrom', 'cohort_key', 'comparison']
    for col in text_cols:
        links[col] = links[col].fillna('').astype(str)
    links['rbp_symbol'] = links['rbp_symbol'].map(base_standardize_symbol)
    links['target_gene_symbol'] = links['target_gene_symbol'].map(base_standardize_symbol)
    links['target_ensg'] = links['target_ensg'].map(base_strip_version)
    links['event_id'] = links['event_id'].map(base_standardize_event_id)
    group_cols = ['rbp_symbol', 'target_gene_symbol', 'target_ensg', 'event_id', 'region', 'source', 'genome_build', 'chrom', 'cohort_key', 'comparison']
    links = links.sort_values(group_cols + ['peak_start', 'peak_end'], kind='mergesort').reset_index(drop=True)
    group_change = links[group_cols].ne(links[group_cols].shift()).any(axis=1)
    group_id = group_change.cumsum()
    previous_group_max_end = links.groupby(group_id, sort=False)['peak_end'].cummax().shift()
    previous_group_max_end[group_change] = np.nan
    segment_change = group_change | links['peak_start'].gt(previous_group_max_end + merge_distance_bp)
    links['_segment_id'] = segment_change.cumsum()
    grouped = links.groupby(group_cols + ['_segment_id'], sort=False, dropna=False)
    out = grouped.agg(peak_start=('peak_start', 'min'), peak_end=('peak_end', 'max'), n_merged_peaks=('peak_start', 'size'), max_score=('score', 'max'), median_score=('score', 'median'), n_accessions=('accession', lambda x: len(set((v for v in x.astype(str) if v and v not in {'nan', 'None', '<NA>'})))), n_methods=('method', lambda x: len(set((v for v in x.astype(str) if v and v not in {'nan', 'None', '<NA>'})))), n_samples=('sample', lambda x: len(set((v for v in x.astype(str) if v and v not in {'nan', 'None', '<NA>'})))), method=('method', base__join_unique), sample=('sample', base__join_unique), accession=('accession', base__join_unique), clip_file=('clip_file', lambda x: base__join_unique(x, max_items=4))).reset_index()
    if out.empty:
        return pd.DataFrame(columns=base_CLIP_ALL_LINK_COLUMNS + ['cohort_key', 'comparison'])
    out['max_score'] = pd.to_numeric(out['max_score'], errors='coerce')
    out['median_score'] = pd.to_numeric(out['median_score'], errors='coerce')
    out['n_accessions'] = pd.to_numeric(out['n_accessions'], errors='coerce').fillna(0).astype(int)
    out['n_merged_peaks'] = pd.to_numeric(out['n_merged_peaks'], errors='coerce').fillna(0).astype(int)
    score_pass = out['max_score'].ge(score_threshold).fillna(False)
    accession_pass = out['n_accessions'].ge(min_accessions)
    peak_support_pass = out['n_accessions'].eq(0) & out['n_merged_peaks'].ge(min_peak_support)
    out = out[score_pass | accession_pass | peak_support_pass].copy()
    if out.empty:
        return pd.DataFrame(columns=base_CLIP_ALL_LINK_COLUMNS + ['cohort_key', 'comparison'])
    filters = []
    for _, row in out.iterrows():
        row_filters = []
        if row['n_accessions'] >= min_accessions:
            row_filters.append(f'accessions_ge_{min_accessions}')
        if pd.notna(row['max_score']) and row['max_score'] >= score_threshold:
            row_filters.append(f'score_ge_{score_threshold:g}')
        if row['n_accessions'] == 0 and row['n_merged_peaks'] >= min_peak_support:
            row_filters.append(f'peaks_ge_{min_peak_support}')
        filters.append(';'.join(row_filters))
    out['consensus_filter'] = filters
    out['evidence_type'] = 'clip_peak_consensus_overlap'
    out['score'] = out['max_score']
    out['peak_width_bp'] = pd.to_numeric(out['peak_end'], errors='coerce') - pd.to_numeric(out['peak_start'], errors='coerce')
    out = out.drop(columns=['_segment_id'], errors='ignore')
    for col in base_CLIP_ALL_LINK_COLUMNS + ['cohort_key', 'comparison']:
        if col not in out.columns:
            out[col] = '' if col not in {'score', 'peak_start', 'peak_end'} else np.nan
    return base__deduplicate_clip_links(out[base_CLIP_ALL_LINK_COLUMNS + ['cohort_key', 'comparison']])

def base_summarize_clip_filtering_qc(links: pd.DataFrame, raw_peak_links: pd.DataFrame | None=None, score_threshold: float=base_CONSENSUS_SCORE_THRESHOLD, min_accessions: int=base_CONSENSUS_MIN_ACCESSIONS, min_peak_support: int=base_CONSENSUS_MIN_PEAKS, merge_distance_bp: int=base_CONSENSUS_MERGE_DISTANCE_BP, cache_status: str='') -> pd.DataFrame:
    """Build a compact manifest for the CLIP consensus filtering strategy."""
    raw_peak_links = raw_peak_links if raw_peak_links is not None else pd.DataFrame()
    rows = [{'metric': 'binding_map_mode', 'value': 'target_interval_consensus_peak_overlap', 'detail': "Raw CLIP peaks are overlapped to 3'UTR/SE intervals, merged, and filtered before target claims."}, {'metric': 'merge_distance_bp', 'value': merge_distance_bp, 'detail': 'Nearby peaks within this distance are merged per RBP-target/event-region.'}, {'metric': 'score_threshold', 'value': score_threshold, 'detail': 'Consensus peaks pass when max POSTAR3/confidence score is at least this value.'}, {'metric': 'min_accessions', 'value': min_accessions, 'detail': 'Consensus peaks pass when this many distinct accessions support the merged segment.'}, {'metric': 'min_peak_support_no_accession', 'value': min_peak_support, 'detail': 'Fallback support rule for peak files without accession IDs.'}, {'metric': 'cache_status', 'value': cache_status, 'detail': 'Whether consensus links were loaded from cache or rebuilt.'}, {'metric': 'raw_peak_overlap_links', 'value': int(len(raw_peak_links)), 'detail': 'Raw peak-overlap rows before consensus filtering.'}, {'metric': 'final_clip_links', 'value': int(len(links)), 'detail': 'CLIP links used by downstream APA/SE/mRNA analyses.'}]
    if not links.empty:
        evidence_counts = links['evidence_type'].fillna('').astype(str).value_counts()
        for evidence, count in evidence_counts.items():
            rows.append({'metric': f'evidence_type:{evidence}', 'value': int(count), 'detail': 'Final link count by evidence type.'})
        if 'consensus_filter' in links.columns:
            exploded = links['consensus_filter'].fillna('').astype(str).str.split(';').explode().replace('', np.nan).dropna().value_counts()
            for filt, count in exploded.items():
                rows.append({'metric': f'consensus_filter:{filt}', 'value': int(count), 'detail': 'Final consensus link count by passing rule.'})
        for source, grp in links.groupby('source'):
            rows.append({'metric': f'source:{source}:links', 'value': int(len(grp)), 'detail': 'Final link count by CLIP source.'})
            rows.append({'metric': f'source:{source}:rbps', 'value': int(grp['rbp_symbol'].nunique()), 'detail': 'RBP count by CLIP source.'})
    return pd.DataFrame(rows)

def base_filter_clip_links_by_support(clip_links: pd.DataFrame, support_mode: str='consensus_all') -> pd.DataFrame:
    """Filter consensus CLIP links for support-threshold sensitivity analyses."""
    if clip_links.empty:
        return clip_links.copy()
    support_mode = support_mode.lower()
    links = clip_links.copy()
    for col in ['n_accessions', 'n_merged_peaks', 'max_score']:
        if col not in links.columns:
            links[col] = np.nan
        links[col] = pd.to_numeric(links[col], errors='coerce')
    if support_mode in {'consensus_all', 'all'}:
        mask = pd.Series(True, index=links.index)
    elif support_mode in {'multi_accession', 'accessions_ge_2'}:
        mask = links['n_accessions'].ge(base_CONSENSUS_MIN_ACCESSIONS)
    elif support_mode in {'high_score', 'score_ge_20'}:
        mask = links['max_score'].ge(base_CONSENSUS_SCORE_THRESHOLD)
    elif support_mode in {'strict', 'strict_multi_accession_and_score'}:
        mask = links['n_accessions'].ge(base_CONSENSUS_MIN_ACCESSIONS) & links['max_score'].ge(base_CONSENSUS_SCORE_THRESHOLD)
    elif support_mode in {'multi_peak', 'peaks_ge_2'}:
        mask = links['n_merged_peaks'].ge(base_CONSENSUS_MIN_PEAKS)
    else:
        raise ValueError('support_mode must be one of consensus_all, multi_accession, high_score, strict_multi_accession_and_score, or multi_peak.')
    return links[mask.fillna(False)].copy().reset_index(drop=True)

def base_overlap_clip_peak_chunks_to_intervals(project_root: Path, catalog: pd.DataFrame, intervals: pd.DataFrame, clip_dir: Path | None=None, bin_size: int=50000, chunksize: int=250000) -> pd.DataFrame:
    """Stream CLIP peaks and return deduplicated overlaps to target intervals."""
    clip_dir = Path(clip_dir) if clip_dir is not None else Path(project_root) / 'data' / 'Ref' / 'clip_rbp'
    if not clip_dir.exists() or intervals.empty:
        return pd.DataFrame(columns=base_CLIP_ALL_LINK_COLUMNS + ['cohort_key', 'comparison'])
    interval_bins = base__expand_intervals_to_bins(intervals, bin_size=bin_size)
    if interval_bins.empty:
        return pd.DataFrame(columns=base_CLIP_ALL_LINK_COLUMNS + ['cohort_key', 'comparison'])
    interval_bounds = intervals.assign(chrom=lambda x: x['chrom'].astype(str), start=lambda x: pd.to_numeric(x['start'], errors='coerce'), end=lambda x: pd.to_numeric(x['end'], errors='coerce')).dropna(subset=['start', 'end']).groupby('chrom').agg(min_start=('start', 'min'), max_end=('end', 'max'))
    interval_chroms = set(interval_bounds.index)
    catalog_symbols = set(catalog['gene_symbol'].map(base_standardize_symbol))
    link_frames = []
    for path in sorted(clip_dir.rglob('*')):
        if not path.is_file() or base__guess_clip_file_type(path) != 'peak_bed':
            continue
        try:
            peak_iter = base__iter_bed_like_clip_peak_chunks(path, catalog_symbols, chunksize=chunksize)
            for peak_chunk in peak_iter:
                peak_chunk = peak_chunk[peak_chunk['chrom'].astype(str).isin(interval_chroms)].copy()
                if peak_chunk.empty:
                    continue
                min_start = peak_chunk['chrom'].astype(str).map(interval_bounds['min_start'])
                max_end = peak_chunk['chrom'].astype(str).map(interval_bounds['max_end'])
                peak_chunk = peak_chunk[peak_chunk['end'].gt(min_start) & peak_chunk['start'].lt(max_end)].copy()
                if peak_chunk.empty:
                    continue
                peak_bins = base__expand_peaks_to_bins(peak_chunk, bin_size=bin_size)
                if peak_bins.empty:
                    continue
                merged = peak_bins.merge(interval_bins, on=['chrom', 'bin'], how='inner', suffixes=('_peak', '_interval'))
                if merged.empty:
                    continue
                overlap = merged[merged['start_peak'].lt(merged['end_interval']) & merged['end_peak'].gt(merged['start_interval'])].copy()
                if overlap.empty:
                    continue
                if 'strand_interval' in overlap.columns and 'strand_peak' in overlap.columns:
                    interval_strand = overlap['strand_interval'].astype(str)
                    peak_strand = overlap['strand_peak'].astype(str)
                    overlap = overlap[interval_strand.isin(['+', '-']).eq(False) | peak_strand.isin(['+', '-']).eq(False) | interval_strand.eq(peak_strand)].copy()
                if overlap.empty:
                    continue
                links = pd.DataFrame({'rbp_symbol': overlap['rbp_symbol'], 'target_gene_symbol': overlap.get('target_gene_symbol', ''), 'target_ensg': overlap.get('target_ensg', '').map(base_strip_version) if 'target_ensg' in overlap.columns else '', 'event_id': overlap.get('event_id', '').astype(str) if 'event_id' in overlap.columns else '', 'region': overlap.get('region', ''), 'source': overlap.get('source', ''), 'evidence_type': 'clip_peak_overlap', 'genome_build': overlap.get('genome_build', ''), 'chrom': overlap.get('chrom', ''), 'peak_start': pd.to_numeric(overlap.get('start_peak', np.nan), errors='coerce'), 'peak_end': pd.to_numeric(overlap.get('end_peak', np.nan), errors='coerce'), 'method': overlap.get('method', ''), 'sample': overlap.get('sample', ''), 'accession': overlap.get('accession', ''), 'score': pd.to_numeric(overlap.get('score', np.nan), errors='coerce'), 'clip_file': overlap.get('clip_file', ''), 'cohort_key': overlap.get('cohort_key', ''), 'comparison': overlap.get('comparison', '')})
                link_frames.append(base__deduplicate_clip_links(links))
        except Exception:
            continue
    if not link_frames:
        return pd.DataFrame(columns=base_CLIP_ALL_LINK_COLUMNS + ['cohort_key', 'comparison'])
    return base__deduplicate_clip_links(pd.concat(link_frames, ignore_index=True, sort=False))

def base_build_clip_links(project_root: Path, catalog: pd.DataFrame, expression: pd.DataFrame, se_events: pd.DataFrame | None=None, apa_membership: pd.DataFrame | None=None, clip_dir: Path | None=None, use_cache: bool=True, peak_evidence_mode: str='consensus', consensus_score_threshold: float=base_CONSENSUS_SCORE_THRESHOLD, consensus_min_accessions: int=base_CONSENSUS_MIN_ACCESSIONS, consensus_min_peak_support: int=base_CONSENSUS_MIN_PEAKS, consensus_merge_distance_bp: int=base_CONSENSUS_MERGE_DISTANCE_BP, return_qc: bool=False) -> tuple[pd.DataFrame, pd.DataFrame] | tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Build all currently available CLIP-supported RBP-target links."""
    project_root = Path(project_root)
    peak_evidence_mode = peak_evidence_mode.lower()
    if peak_evidence_mode not in {'consensus', 'raw'}:
        raise ValueError("peak_evidence_mode must be 'consensus' or 'raw'.")
    cache_suffix = 'consensus' if peak_evidence_mode == 'consensus' else 'raw'
    cache_path = project_root / 'data' / base_ANALYSIS_SLUG / f'clip_supported_rbp_target_links_{cache_suffix}.tsv'
    if use_cache and cache_path.exists() and (cache_path.stat().st_size > 0):
        try:
            cached = pd.read_csv(cache_path, sep='\t', low_memory=False)
            if not cached.empty:
                for col in base_CLIP_ALL_LINK_COLUMNS + ['cohort_key', 'comparison']:
                    if col not in cached.columns:
                        cached[col] = '' if col != 'score' else np.nan
                numeric_cols = {'score', 'peak_start', 'peak_end', 'n_merged_peaks', 'n_accessions', 'n_methods', 'n_samples', 'max_score', 'median_score', 'peak_width_bp'}
                for col in [c for c in base_CLIP_ALL_LINK_COLUMNS + ['cohort_key', 'comparison'] if c not in numeric_cols]:
                    cached[col] = cached[col].fillna('').astype(str).replace({'nan': '', 'None': '', '<NA>': ''})
                cached['event_id'] = cached['event_id'].map(base_standardize_event_id)
                for col in numeric_cols:
                    if col in cached.columns:
                        cached[col] = pd.to_numeric(cached[col], errors='coerce')
                catalog_for_coverage = base_map_catalog_to_expression(catalog, expression)
                coverage = base_summarize_clip_coverage(cached, catalog_for_coverage)
                qc = base_summarize_clip_filtering_qc(cached, raw_peak_links=pd.DataFrame(), score_threshold=consensus_score_threshold, min_accessions=consensus_min_accessions, min_peak_support=consensus_min_peak_support, merge_distance_bp=consensus_merge_distance_bp, cache_status=f'loaded:{cache_path.name}')
                if return_qc:
                    return (cached, coverage, qc)
                return (cached, coverage)
        except Exception:
            pass
    catalog = base_map_catalog_to_expression(catalog, expression)
    interactions = base_load_clip_interactions(project_root, catalog, expression, clip_dir=clip_dir)
    utr_intervals = base_load_utr_bed(project_root)
    symbol_to_ensg = expression.dropna(subset=['gene_symbol', 'ensg']).assign(gene_symbol=lambda x: x['gene_symbol'].map(base_standardize_symbol), ensg=lambda x: x['ensg'].map(base_strip_version)).drop_duplicates('gene_symbol').set_index('gene_symbol')['ensg'].to_dict()
    if not utr_intervals.empty:
        utr_intervals['target_ensg'] = utr_intervals['target_gene_symbol'].map(symbol_to_ensg).fillna('')
        if apa_membership is not None and (not apa_membership.empty):
            apa_symbols = set()
            for col in apa_membership.columns:
                if col.endswith('host_gene_symbol') or col == 'host_gene_symbol':
                    apa_symbols.update(apa_membership[col].dropna().map(base_standardize_symbol))
            apa_symbols.discard('')
            if apa_symbols:
                utr_intervals = utr_intervals[utr_intervals['target_gene_symbol'].isin(apa_symbols)].copy()
    peak_intervals = []
    if not utr_intervals.empty:
        peak_intervals.append(utr_intervals)
    se_intervals = base_make_se_overlap_intervals(se_events if se_events is not None else pd.DataFrame())
    if not se_intervals.empty:
        peak_intervals.append(se_intervals)
    if peak_intervals:
        raw_cache_path = project_root / 'data' / base_ANALYSIS_SLUG / 'clip_supported_rbp_target_links_raw_peak_overlap.tsv'
        if use_cache and raw_cache_path.exists() and (raw_cache_path.stat().st_size > 0):
            try:
                raw_peak_links = pd.read_csv(raw_cache_path, sep='\t', low_memory=False)
            except Exception:
                raw_peak_links = pd.DataFrame()
        else:
            raw_peak_links = pd.DataFrame()
        if raw_peak_links.empty:
            raw_peak_links = base_overlap_clip_peak_chunks_to_intervals(project_root, catalog, pd.concat(peak_intervals, ignore_index=True, sort=False), clip_dir=clip_dir)
            try:
                base_write_tsv(raw_peak_links, raw_cache_path)
            except Exception:
                pass
        if peak_evidence_mode == 'consensus':
            peak_links = base_build_consensus_clip_peak_links(raw_peak_links, merge_distance_bp=consensus_merge_distance_bp, score_threshold=consensus_score_threshold, min_accessions=consensus_min_accessions, min_peak_support=consensus_min_peak_support)
        else:
            peak_links = raw_peak_links
    else:
        raw_peak_links = pd.DataFrame(columns=base_CLIP_ALL_LINK_COLUMNS + ['cohort_key', 'comparison'])
        peak_links = pd.DataFrame(columns=base_CLIP_ALL_LINK_COLUMNS + ['cohort_key', 'comparison'])
    links = pd.concat([interactions, peak_links], ignore_index=True, sort=False)
    if links.empty:
        links = pd.DataFrame(columns=base_CLIP_ALL_LINK_COLUMNS + ['cohort_key', 'comparison'])
    for col in base_CLIP_ALL_LINK_COLUMNS + ['cohort_key', 'comparison']:
        if col not in links.columns:
            links[col] = '' if col != 'score' else np.nan
    numeric_cols = {'score', 'peak_start', 'peak_end', 'n_merged_peaks', 'n_accessions', 'n_methods', 'n_samples', 'max_score', 'median_score', 'peak_width_bp'}
    for col in [c for c in base_CLIP_ALL_LINK_COLUMNS + ['cohort_key', 'comparison'] if c not in numeric_cols]:
        links[col] = links[col].fillna('').astype(str).replace({'nan': '', 'None': '', '<NA>': ''})
    for col in numeric_cols:
        if col in links.columns:
            links[col] = pd.to_numeric(links[col], errors='coerce')
    links['event_id'] = links['event_id'].map(base_standardize_event_id)
    links['rbp_symbol'] = links['rbp_symbol'].map(base_standardize_symbol)
    links['target_gene_symbol'] = links['target_gene_symbol'].map(base_standardize_symbol)
    links['target_ensg'] = links['target_ensg'].map(base_strip_version)
    links = links.drop_duplicates().reset_index(drop=True)
    coverage = base_summarize_clip_coverage(links, catalog)
    qc = base_summarize_clip_filtering_qc(links, raw_peak_links=raw_peak_links, score_threshold=consensus_score_threshold, min_accessions=consensus_min_accessions, min_peak_support=consensus_min_peak_support, merge_distance_bp=consensus_merge_distance_bp, cache_status=f'rebuilt:{cache_path.name}')
    try:
        base_write_tsv(links, cache_path)
    except Exception:
        pass
    if return_qc:
        return (links, coverage, qc)
    return (links, coverage)

def base_summarize_clip_coverage(links: pd.DataFrame, catalog: pd.DataFrame) -> pd.DataFrame:
    """Summarize CLIP target counts by RBP and evidence region."""
    if links.empty:
        return pd.DataFrame(columns=['rbp_symbol', 'functional_class', 'rbp_family', 'n_clip_links', 'n_target_genes', 'n_target_events', 'regions', 'sources'])
    catalog_small = catalog[['gene_symbol', 'functional_class', 'rbp_family']].rename(columns={'gene_symbol': 'rbp_symbol'})
    rows = []
    for rbp, grp in links.groupby('rbp_symbol'):
        rows.append({'rbp_symbol': rbp, 'n_clip_links': int(len(grp)), 'n_target_genes': int(grp.loc[grp['target_gene_symbol'].ne('') | grp['target_ensg'].ne(''), ['target_gene_symbol', 'target_ensg']].drop_duplicates().shape[0]), 'n_target_events': int(grp.loc[grp.get('event_id', '').astype(str).ne(''), 'event_id'].nunique()) if 'event_id' in grp else 0, 'regions': ','.join(sorted(grp['region'].dropna().astype(str).unique())), 'sources': ','.join(sorted(grp['source'].dropna().astype(str).unique()))})
    out = pd.DataFrame(rows)
    return out.merge(catalog_small, on='rbp_symbol', how='left').sort_values(['n_clip_links', 'rbp_symbol'], ascending=[False, True])

def base_bh_fdr(p_values: Sequence[object]) -> np.ndarray:
    """Benjamini-Hochberg FDR adjustment."""
    p = pd.to_numeric(pd.Series(p_values), errors='coerce').to_numpy(dtype=float)
    q = np.full_like(p, np.nan, dtype=float)
    valid = np.isfinite(p)
    if valid.sum() == 0:
        return q
    idx = np.where(valid)[0]
    order = idx[np.argsort(p[valid])]
    ranked = p[order]
    n = len(ranked)
    adjusted = ranked * n / np.arange(1, n + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    q[order] = np.clip(adjusted, 0, 1)
    return q

def base__gene_key_series(df: pd.DataFrame, ensg_col: str, symbol_col: str) -> pd.Series:
    ensg = df.get(ensg_col, pd.Series('', index=df.index)).map(base_strip_version)
    symbol = df.get(symbol_col, pd.Series('', index=df.index)).map(base_standardize_symbol)
    return ensg.where(ensg.ne(''), symbol)

def base_compute_gene_target_enrichment(clip_links: pd.DataFrame, rbp_expression: pd.DataFrame, universe_genes: Iterable[str], positive_genes: Iterable[str], comparison: str, cohort_key: str, region_keywords: Sequence[str]=('3UTR', 'gene_level_clip'), min_clip_targets: int=3) -> pd.DataFrame:
    """Test whether positive genes are enriched among CLIP targets of rhythmic RBPs."""
    universe = {base_strip_version(g) if str(g).startswith('ENS') else base_standardize_symbol(g) for g in universe_genes if str(g)}
    positive = {base_strip_version(g) if str(g).startswith('ENS') else base_standardize_symbol(g) for g in positive_genes if str(g)}
    positive &= universe
    if not universe or clip_links.empty:
        return pd.DataFrame()
    links = clip_links.copy()
    links['target_key'] = base__gene_key_series(links, 'target_ensg', 'target_gene_symbol')
    links = links[links['target_key'].isin(universe)]
    if region_keywords:
        pattern = '|'.join([str(x) for x in region_keywords])
        links = links[links['region'].astype(str).str.contains(pattern, case=False, na=False)]
    rhythmic_rbps = set(rbp_expression.loc[rbp_expression['cohort_key'].eq(cohort_key) & rbp_expression['rhythmic'], 'gene_symbol'].map(base_standardize_symbol))
    rows = []
    for rbp, grp in links.groupby('rbp_symbol'):
        if rbp not in rhythmic_rbps:
            continue
        targets = set(grp['target_key']) & universe
        if len(targets) < min_clip_targets:
            continue
        a = len(targets & positive)
        b = len(targets - positive)
        c = len(positive - targets)
        d = len(universe - positive - targets)
        if fisher_exact is None:
            odds_ratio, p_value = (np.nan, np.nan)
        else:
            odds_ratio, p_value = fisher_exact([[a, b], [c, d]], alternative='greater')
        rows.append({'comparison': comparison, 'cohort_key': cohort_key, 'rbp_symbol': rbp, 'n_universe_genes': len(universe), 'n_positive_genes': len(positive), 'n_clip_targets_in_universe': len(targets), 'n_positive_clip_targets': a, 'odds_ratio': odds_ratio, 'p_value': p_value, 'regions_used': ','.join(region_keywords)})
    out = pd.DataFrame(rows)
    if not out.empty:
        out['fdr'] = base_bh_fdr(out['p_value'])
        out = out.sort_values(['fdr', 'p_value', 'n_positive_clip_targets'], ascending=[True, True, False])
    return out

def base_compute_event_target_enrichment(clip_links: pd.DataFrame, rbp_expression: pd.DataFrame, universe_events: Iterable[str], positive_events: Iterable[str], comparison: str, cohort_key: str, region_keywords: Sequence[str]=('SE_',), min_clip_events: int=3) -> pd.DataFrame:
    """Test whether rhythmic SE events are enriched among CLIP-overlapped events."""
    universe = {str(x) for x in universe_events if str(x)}
    positive = {str(x) for x in positive_events if str(x)} & universe
    if not universe or clip_links.empty:
        return pd.DataFrame()
    links = clip_links.copy()
    links['event_id'] = links.get('event_id', '').astype(str)
    links = links[links['event_id'].isin(universe)]
    if region_keywords:
        pattern = '|'.join([str(x) for x in region_keywords])
        links = links[links['region'].astype(str).str.contains(pattern, case=False, na=False)]
    rhythmic_rbps = set(rbp_expression.loc[rbp_expression['cohort_key'].eq(cohort_key) & rbp_expression['rhythmic'], 'gene_symbol'].map(base_standardize_symbol))
    rows = []
    for rbp, grp in links.groupby('rbp_symbol'):
        if rbp not in rhythmic_rbps:
            continue
        targets = set(grp['event_id']) & universe
        if len(targets) < min_clip_events:
            continue
        a = len(targets & positive)
        b = len(targets - positive)
        c = len(positive - targets)
        d = len(universe - positive - targets)
        if fisher_exact is None:
            odds_ratio, p_value = (np.nan, np.nan)
        else:
            odds_ratio, p_value = fisher_exact([[a, b], [c, d]], alternative='greater')
        rows.append({'comparison': comparison, 'cohort_key': cohort_key, 'rbp_symbol': rbp, 'n_universe_events': len(universe), 'n_positive_events': len(positive), 'n_clip_events_in_universe': len(targets), 'n_positive_clip_events': a, 'odds_ratio': odds_ratio, 'p_value': p_value, 'regions_used': ','.join(region_keywords)})
    out = pd.DataFrame(rows)
    if not out.empty:
        out['fdr'] = base_bh_fdr(out['p_value'])
        out = out.sort_values(['fdr', 'p_value', 'n_positive_clip_events'], ascending=[True, True, False])
    return out

def base__normalize_target_ids(values: Iterable[object], id_type: str) -> pd.Series:
    """Normalize gene or event identifiers for set operations."""
    series = pd.Series(list(values), dtype='object')
    if id_type == 'gene':
        return series.map(lambda x: base_strip_version(x) if str(x).startswith('ENS') else base_standardize_symbol(x))
    if id_type == 'event':
        return series.map(base_standardize_event_id)
    raise ValueError("id_type must be 'gene' or 'event'.")

def base__clip_target_key(links: pd.DataFrame, id_type: str) -> pd.Series:
    """Return the CLIP target key used for gene-level APA or event-level SE tests."""
    if id_type == 'gene':
        return base__gene_key_series(links, 'target_ensg', 'target_gene_symbol')
    if id_type == 'event':
        return links.get('event_id', pd.Series('', index=links.index)).map(base_standardize_event_id)
    raise ValueError("id_type must be 'gene' or 'event'.")

def base__phase_resultant_length(hours: Iterable[object]) -> float:
    """Return circular mean resultant length for phase-like hours."""
    return float(base_phase_summary(hours).get('resultant_length', np.nan))

def base_compute_target_background_contrast(clip_links: pd.DataFrame, rbp_expression: pd.DataFrame, universe_ids: Iterable[object], positive_ids: Iterable[object], phase_table: pd.DataFrame, phase_id_col: str, phase_col: str, comparison: str, cohort_key: str, event_kind: str, id_type: str, region_keywords: Sequence[str], min_clip_targets: int=3) -> pd.DataFrame:
    """Compare rhythmic event rates and phase concentration in CLIP targets vs non-target background."""
    universe = set(base__normalize_target_ids(universe_ids, id_type))
    universe.discard('')
    positive = set(base__normalize_target_ids(positive_ids, id_type)) & universe
    if not universe or clip_links.empty:
        return pd.DataFrame()
    links = clip_links.copy()
    links['target_key'] = base__clip_target_key(links, id_type)
    links = links[links['target_key'].isin(universe)].copy()
    if region_keywords:
        pattern = '|'.join([str(x) for x in region_keywords])
        links = links[links['region'].astype(str).str.contains(pattern, case=False, na=False)].copy()
    if links.empty:
        return pd.DataFrame()
    rhythmic_rbps = set(rbp_expression.loc[rbp_expression['cohort_key'].eq(cohort_key) & rbp_expression['rhythmic'], 'gene_symbol'].map(base_standardize_symbol))
    phase = phase_table.copy()
    if phase.empty or phase_id_col not in phase.columns or phase_col not in phase.columns:
        phase = pd.DataFrame(columns=['target_key', 'phase_hours'])
    else:
        phase['target_key'] = base__normalize_target_ids(phase[phase_id_col], id_type)
        phase['phase_hours'] = pd.to_numeric(phase[phase_col], errors='coerce')
        phase = phase[phase['target_key'].isin(positive)].dropna(subset=['phase_hours']).copy()
    rows = []
    for rbp, grp in links.groupby('rbp_symbol'):
        rbp = base_standardize_symbol(rbp)
        if rbp not in rhythmic_rbps:
            continue
        targets = set(grp['target_key']) & universe
        if len(targets) < min_clip_targets:
            continue
        background = universe - targets
        target_positive = targets & positive
        background_positive = background & positive
        a = len(target_positive)
        b = len(targets - positive)
        c = len(background_positive)
        d = len(background - positive)
        if fisher_exact is None:
            odds_ratio, p_value = (np.nan, np.nan)
        else:
            odds_ratio, p_value = fisher_exact([[a, b], [c, d]], alternative='greater')
        target_phases = phase.loc[phase['target_key'].isin(target_positive), 'phase_hours']
        background_phases = phase.loc[phase['target_key'].isin(background_positive), 'phase_hours']
        target_phase_summary = base_phase_summary(target_phases)
        background_phase_summary = base_phase_summary(background_phases)
        rows.append({'comparison': comparison, 'cohort_key': cohort_key, 'event_kind': event_kind, 'rbp_symbol': rbp, 'n_universe': len(universe), 'n_positive': len(positive), 'n_clip_targets': len(targets), 'n_background': len(background), 'n_positive_clip_targets': a, 'n_positive_background': c, 'target_rhythmic_rate': a / len(targets) if targets else np.nan, 'background_rhythmic_rate': c / len(background) if background else np.nan, 'rate_difference': (a / len(targets) if targets else np.nan) - (c / len(background) if background else np.nan), 'odds_ratio': odds_ratio, 'p_value': p_value, 'target_phase_n': target_phase_summary['n'], 'target_phase_resultant_length': target_phase_summary['resultant_length'], 'target_phase_mean_hours': target_phase_summary['mean_phase_hours'], 'background_phase_n': background_phase_summary['n'], 'background_phase_resultant_length': background_phase_summary['resultant_length'], 'phase_resultant_delta': target_phase_summary['resultant_length'] - background_phase_summary['resultant_length'] if pd.notna(target_phase_summary['resultant_length']) and pd.notna(background_phase_summary['resultant_length']) else np.nan, 'regions_used': ','.join(region_keywords)})
    out = pd.DataFrame(rows)
    if not out.empty:
        out['fdr'] = base_bh_fdr(out['p_value'])
        out = out.sort_values(['fdr', 'p_value', 'rate_difference'], ascending=[True, True, False])
    return out

def base_summarize_phase_lag_coherence(lag_df: pd.DataFrame, lag_col: str='rbp_to_event_lag_hours', min_lags: int=5, n_permutations: int=500, random_seed: int=42) -> pd.DataFrame:
    """Summarize RBP-event lag concentration and compare it with a pooled phase-lag null."""
    if lag_df.empty or lag_col not in lag_df.columns:
        return pd.DataFrame()
    rng = np.random.default_rng(random_seed)
    frame = lag_df.copy()
    frame[lag_col] = pd.to_numeric(frame[lag_col], errors='coerce')
    frame = frame.dropna(subset=[lag_col])
    if frame.empty:
        return pd.DataFrame()
    rows = []
    for (comparison, event_kind), base in frame.groupby(['comparison', 'event_kind'], dropna=False):
        pool = base[lag_col].dropna().to_numpy(dtype=float)
        if pool.size < min_lags:
            continue
        for rbp, grp in base.groupby('rbp_symbol', dropna=False):
            values = grp[lag_col].dropna().to_numpy(dtype=float)
            if values.size < min_lags:
                continue
            phase_values = values % 24.0
            observed = base_phase_summary(phase_values)
            null_r = []
            for _ in range(n_permutations):
                sampled = rng.choice(pool, size=values.size, replace=True) % 24.0
                null_r.append(base__phase_resultant_length(sampled))
            null_r = np.asarray(null_r, dtype=float)
            p_value = (float((null_r >= observed['resultant_length']).sum()) + 1.0) / (len(null_r) + 1.0)
            mean_lag = (observed['mean_phase_hours'] + 12.0) % 24.0 - 12.0 if pd.notna(observed['mean_phase_hours']) else np.nan
            rows.append({'comparison': comparison, 'event_kind': event_kind, 'rbp_symbol': base_standardize_symbol(rbp), 'lag_col': lag_col, 'n_lags': int(values.size), 'mean_lag_hours': mean_lag, 'median_lag_hours': float(np.median(values)), 'lag_resultant_length': observed['resultant_length'], 'null_mean_resultant_length': float(np.nanmean(null_r)) if null_r.size else np.nan, 'empirical_p_value': p_value})
    out = pd.DataFrame(rows)
    if not out.empty:
        out['empirical_fdr'] = base_bh_fdr(out['empirical_p_value'])
        out = out.sort_values(['empirical_fdr', 'empirical_p_value', 'lag_resultant_length'], ascending=[True, True, False])
    return out

def base_compute_apa_phase_lags(clip_links: pd.DataFrame, rbp_expression: pd.DataFrame, apa_phase: pd.DataFrame, comparison: str, cohort_key: str) -> pd.DataFrame:
    """Build rhythmic RBP -> CLIP-supported APA target -> rhythmic mRNA lag rows."""
    if clip_links.empty or apa_phase.empty:
        return pd.DataFrame()
    rbp_phase = rbp_expression[rbp_expression['cohort_key'].eq(cohort_key) & rbp_expression['rhythmic']][['gene_symbol', 'acrophase_hours', 'amplitude', 'p_value', 'q_value']].copy()
    rbp_phase = rbp_phase.rename(columns={'gene_symbol': 'rbp_symbol', 'acrophase_hours': 'rbp_acrophase_hours', 'amplitude': 'rbp_amplitude', 'p_value': 'rbp_p_value', 'q_value': 'rbp_q_value'})
    events = apa_phase[apa_phase['comparison'].eq(comparison)].copy()
    events['target_key'] = base__gene_key_series(events, 'host_ensg', 'host_gene_symbol')
    links = clip_links.copy()
    links['target_key'] = base__gene_key_series(links, 'target_ensg', 'target_gene_symbol')
    links = links[links['region'].astype(str).str.contains('3UTR|gene_level_clip', case=False, na=False)].copy()
    for col in ['n_accessions', 'n_merged_peaks', 'max_score', 'median_score']:
        if col not in links.columns:
            links[col] = np.nan
        links[col] = pd.to_numeric(links[col], errors='coerce')
    links = links.sort_values(['n_accessions', 'n_merged_peaks', 'max_score'], ascending=[False, False, False])
    links = links.drop_duplicates(['rbp_symbol', 'target_key', 'region', 'source', 'evidence_type', 'genome_build'])
    joined = links.merge(events, on='target_key', how='inner', suffixes=('', '_apa'))
    joined = joined.merge(rbp_phase, on='rbp_symbol', how='inner')
    if joined.empty:
        return pd.DataFrame()
    joined['event_kind'] = 'APA'
    joined['comparison'] = comparison
    joined['cohort_key'] = cohort_key
    joined['event_phase_hours'] = joined['apa_acrophase_hours']
    joined['target_mrna_phase_hours'] = joined['expr_acrophase_hours']
    joined['rbp_to_event_lag_hours'] = base_circular_lag_hours(joined['event_phase_hours'], joined['rbp_acrophase_hours'])
    joined['event_to_mrna_lag_hours'] = base_circular_lag_hours(joined['target_mrna_phase_hours'], joined['event_phase_hours'])
    keep = ['comparison', 'cohort_key', 'event_kind', 'rbp_symbol', 'target_key', 'host_ensg', 'host_gene_symbol', 'apa_event_id', 'region', 'source', 'evidence_type', 'genome_build', 'rbp_acrophase_hours', 'event_phase_hours', 'target_mrna_phase_hours', 'rbp_to_event_lag_hours', 'event_to_mrna_lag_hours', 'rbp_amplitude', 'amplitude', 'expr_amplitude', 'fdr', 'expr_q', 'clip_file', 'n_merged_peaks', 'n_accessions', 'n_methods', 'n_samples', 'max_score', 'median_score', 'peak_width_bp', 'consensus_filter']
    for col in keep:
        if col not in joined.columns:
            joined[col] = ''
    return joined[keep].drop_duplicates().reset_index(drop=True)

def base_compute_se_phase_lags(clip_links: pd.DataFrame, rbp_expression: pd.DataFrame, se_events: pd.DataFrame, expression: pd.DataFrame, comparison: str, cohort_key: str) -> pd.DataFrame:
    """Build rhythmic RBP -> CLIP-supported SE target -> rhythmic mRNA lag rows."""
    if clip_links.empty or se_events.empty:
        return pd.DataFrame()
    rbp_phase = rbp_expression[rbp_expression['cohort_key'].eq(cohort_key) & rbp_expression['rhythmic']][['gene_symbol', 'acrophase_hours', 'amplitude', 'p_value', 'q_value']].copy()
    rbp_phase = rbp_phase.rename(columns={'gene_symbol': 'rbp_symbol', 'acrophase_hours': 'rbp_acrophase_hours', 'amplitude': 'rbp_amplitude', 'p_value': 'rbp_p_value', 'q_value': 'rbp_q_value'})
    expr_phase = expression[expression['cohort_key'].eq(cohort_key) & expression['p_value'].lt(0.05)][['gene_symbol', 'ensg', 'acrophase_hours', 'amplitude', 'q_value']].copy()
    expr_phase = expr_phase.rename(columns={'gene_symbol': 'expr_gene_symbol', 'ensg': 'gene_id', 'acrophase_hours': 'target_mrna_phase_hours', 'amplitude': 'expr_amplitude', 'q_value': 'expr_q'})
    expr_phase['gene_id'] = expr_phase['gene_id'].map(base_strip_version)
    events = se_events[se_events['comparison'].eq(comparison)].copy()
    events['event_id'] = events['event_id'].astype(str)
    links = clip_links.copy()
    links['event_id'] = links.get('event_id', '').astype(str)
    links = links[links['region'].astype(str).str.contains('SE_', case=False, na=False)].copy()
    for col in ['n_accessions', 'n_merged_peaks', 'max_score', 'median_score']:
        if col not in links.columns:
            links[col] = np.nan
        links[col] = pd.to_numeric(links[col], errors='coerce')
    links = links.sort_values(['n_accessions', 'n_merged_peaks', 'max_score'], ascending=[False, False, False])
    links = links.drop_duplicates(['rbp_symbol', 'event_id', 'region', 'source', 'evidence_type', 'genome_build'])
    joined = links.merge(events, on='event_id', how='inner', suffixes=('', '_se'))
    joined = joined.merge(rbp_phase, on='rbp_symbol', how='inner')
    joined = joined.merge(expr_phase, on='gene_id', how='left')
    if joined.empty:
        return pd.DataFrame()
    joined['event_kind'] = 'SE'
    joined['comparison'] = comparison
    joined['cohort_key'] = cohort_key
    joined['event_phase_hours'] = joined['acrophase_hours']
    joined['rbp_to_event_lag_hours'] = base_circular_lag_hours(joined['event_phase_hours'], joined['rbp_acrophase_hours'])
    joined['event_to_mrna_lag_hours'] = base_circular_lag_hours(joined['target_mrna_phase_hours'], joined['event_phase_hours'])
    keep = ['comparison', 'cohort_key', 'event_kind', 'rbp_symbol', 'event_id', 'gene_id', 'gene_symbol', 'region', 'source', 'evidence_type', 'genome_build', 'rbp_acrophase_hours', 'event_phase_hours', 'target_mrna_phase_hours', 'rbp_to_event_lag_hours', 'event_to_mrna_lag_hours', 'rbp_amplitude', 'amplitude', 'expr_amplitude', 'fdr', 'expr_q', 'clip_file', 'n_merged_peaks', 'n_accessions', 'n_methods', 'n_samples', 'max_score', 'median_score', 'peak_width_bp', 'consensus_filter']
    for col in keep:
        if col not in joined.columns:
            joined[col] = ''
    return joined[keep].drop_duplicates().reset_index(drop=True)

def base_build_triplets(apa_lags: pd.DataFrame, se_lags: pd.DataFrame) -> pd.DataFrame:
    """Combine APA and SE lag rows and annotate validation-replicated triplets."""
    frames = []
    if apa_lags is not None and (not apa_lags.empty):
        apa = apa_lags.copy()
        apa['target_gene_id'] = apa['host_ensg'].map(base_strip_version)
        apa['target_gene_symbol'] = apa['host_gene_symbol'].map(base_standardize_symbol)
        apa['event_id'] = apa['apa_event_id'].astype(str)
        frames.append(apa)
    if se_lags is not None and (not se_lags.empty):
        se = se_lags.copy()
        se['target_gene_id'] = se['gene_id'].map(base_strip_version)
        se['target_gene_symbol'] = se['gene_symbol'].map(base_standardize_symbol)
        frames.append(se)
    if not frames:
        return pd.DataFrame()
    triplets = pd.concat(frames, ignore_index=True, sort=False)
    triplets['triplet_key'] = triplets['rbp_symbol'].map(base_standardize_symbol) + '|' + triplets['event_kind'].astype(str) + '|' + triplets['target_gene_id'].fillna('').astype(str)
    replicated = triplets.groupby('triplet_key')['comparison'].nunique().rename('n_comparisons').reset_index()
    triplets = triplets.merge(replicated, on='triplet_key', how='left')
    triplets['validation_replicated'] = triplets['n_comparisons'].ge(2)
    lag_abs = triplets['rbp_to_event_lag_hours'].abs().fillna(12) + triplets['event_to_mrna_lag_hours'].abs().fillna(12)
    triplets['phase_lag_priority_score'] = triplets['validation_replicated'].astype(int) * 100 - lag_abs
    return triplets.sort_values(['validation_replicated', 'phase_lag_priority_score'], ascending=[False, False]).reset_index(drop=True)

def base_empty_table(columns: Sequence[str]) -> pd.DataFrame:
    """Return an empty table with a stable schema."""
    return pd.DataFrame(columns=list(columns))


In [ ]:
# Cell 4: Activate the visible base-analysis namespace.
for _name, _value in list(globals().items()):
    if _name.startswith('base_'):
        setattr(base, _name.removeprefix('base_'), _value)
# The embedded source uses this global for its CLIP cache and therefore must
# point at the publication namespace before any source-table function runs.
base_ANALYSIS_SLUG = ANALYSIS_SLUG

# SciPy is part of the base layer public API used by enrichment helpers.
base.fisher_exact = fisher_exact


In [ ]:
# Cell 5: Embedded and namespace-isolated from scripts/generate_rbp_processing_source_tables.py
"""Generate source tables for the current 3.4.3 RBP-processing figure section."""
from __future__ import annotations
from pathlib import Path
import argparse
import numpy as np
import pandas as pd
source_ANALYSIS = 'rbp_rna_processing_links'
source_CURRENT_PANEL_SOURCE_TABLES = ['rbp_rhythmicity_summary.tsv', 'rbp_expression_rhythmicity.tsv', 'rbp_phase_circular_density.tsv', 'rbp_phase_polarization_summary.tsv', 'rbp_tumor_normal_rhythmic_overlap.tsv', 'rbp_overlap_reactome_enrichment_top.tsv', 'rbp_apa_clip_enrichment.tsv', 'rbp_se_clip_enrichment.tsv', 'rbp_target_background_contrast.tsv', 'rbp_phase_lag_coherence.tsv', 'rbp_event_gene_triplets.tsv', 'clip_supported_rbp_target_links_consensus.tsv', 'rbp_main_figure_candidate_summary.tsv', 'rbp_apa_phase_lags.tsv', 'rbp_se_phase_lags.tsv', 'apa_expression_phase_lag_inputs.tsv', 'apa_host_gene_membership_inputs.tsv', 'se_variable_event_universe.tsv', 'se_rhythmic_events_p0.05.tsv']

def source__cohort_prefix(comparison: str) -> str:
    return 'TCGA' if comparison == 'TCGA_vs_GTEx' else 'CPTAC'

def source__rhythmic_mask(frame: pd.DataFrame, rhythmic_col: str) -> pd.Series:
    if rhythmic_col not in frame.columns:
        return pd.Series(False, index=frame.index)
    series = frame[rhythmic_col]
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors='coerce').fillna(0).ne(0)
    return series.astype(str).str.strip().str.lower().isin({'true', 't', '1', 'yes', 'y'})

def source__save_table(tables: dict[str, pd.DataFrame], manifest_rows: list[dict[str, object]], output_dir: Path, stage: str, filename: str, df: pd.DataFrame) -> pd.DataFrame:
    df = pd.DataFrame(df)
    path = base.write_tsv(df, output_dir / filename)
    tables[filename] = df
    manifest_rows.append({'stage': stage, 'table': filename, 'path': str(path), 'rows': int(df.shape[0]), 'columns': int(df.shape[1]), 'bytes': int(path.stat().st_size) if path.exists() else 0})
    return df

def source_circular_density(phases_hours: pd.Series, n_grid: int=360, bw: float=0.3) -> pd.DataFrame:
    """Von-Mises-like circular kernel density on a 0-24h phase scale."""
    values = pd.to_numeric(pd.Series(phases_hours), errors='coerce').dropna()
    values = values[(values >= 0) & (values <= 24)]
    if len(values) < 3:
        return pd.DataFrame(columns=['theta', 'phase_hours', 'density'])
    theta = values.to_numpy(dtype=float) % 24.0 / 24.0 * 2.0 * np.pi
    grid = np.linspace(0, 2.0 * np.pi, n_grid + 1)[:-1]
    kappa = 1.0 / (float(bw) * float(bw))
    density = np.array([np.exp(kappa * np.cos(theta - g)).mean() for g in grid])
    density = density / (density.sum() * (2.0 * np.pi / n_grid))
    out = pd.DataFrame({'theta': grid, 'phase_hours': grid / (2.0 * np.pi) * 24.0, 'density': density})
    return pd.concat([out, pd.DataFrame({'theta': [2.0 * np.pi], 'phase_hours': [24.0], 'density': [out['density'].iloc[0]]})], ignore_index=True)

def source_axial_phase_summary(phases_hours: pd.Series) -> dict[str, float]:
    """Summarize antipodal phase orientation with the second trigonometric moment."""
    values = pd.to_numeric(pd.Series(phases_hours), errors='coerce').dropna()
    values = values[(values >= 0) & (values <= 24)]
    n = int(len(values))
    if n == 0:
        return {'n': 0, 'axial_r2': np.nan, 'axis_hours': np.nan, 'opposite_axis_hours': np.nan}
    theta = values.to_numpy(dtype=float) % 24.0 / 24.0 * 2.0 * np.pi
    z2 = np.mean(np.exp(1j * 2.0 * theta))
    axis_hours = np.angle(z2) / 2.0 % np.pi * 24.0 / (2.0 * np.pi)
    return {'n': n, 'axial_r2': float(np.abs(z2)), 'axis_hours': float(axis_hours), 'opposite_axis_hours': float((axis_hours + 12.0) % 24.0)}

def source_build_catalog_expression_tables(project_root: Path, output_dir: Path, tables: dict[str, pd.DataFrame], manifest_rows: list[dict[str, object]]) -> dict[str, pd.DataFrame]:
    catalog = base.default_rbp_catalog()
    expression = base.load_expression_tables(project_root)
    catalog = base.map_catalog_to_expression(catalog, expression)
    rbp_expression = base.annotate_rbp_rhythmicity(catalog, expression, p_threshold=0.05, q_threshold=0.05)
    rbp_summary = base.summarize_rbp_rhythmicity(rbp_expression)
    rbp_phase_summary = base.summarize_phase_by_group(rbp_expression[rbp_expression['rhythmic']], ['cohort_key', 'functional_class'], phase_col='acrophase_hours')
    forbidden = sorted([x for x in expression['cohort_key'].dropna().unique() if 'CPTAC' in x and 'Normal' in x])
    if forbidden:
        raise ValueError(f'CPTAC normal should not be present in this analysis: {forbidden}')
    source__save_table(tables, manifest_rows, output_dir, 'catalog_expression', 'rbp_catalog.tsv', catalog)
    source__save_table(tables, manifest_rows, output_dir, 'catalog_expression', 'expression_rhythmicity_inputs.tsv', expression)
    source__save_table(tables, manifest_rows, output_dir, 'catalog_expression', 'rbp_expression_rhythmicity.tsv', rbp_expression)
    source__save_table(tables, manifest_rows, output_dir, 'catalog_expression', 'rbp_rhythmicity_summary.tsv', rbp_summary)
    source__save_table(tables, manifest_rows, output_dir, 'catalog_expression', 'rbp_phase_summary.tsv', rbp_phase_summary)
    return {'catalog': catalog, 'expression': expression, 'rbp_expression': rbp_expression, 'rbp_summary': rbp_summary, 'rbp_phase_summary': rbp_phase_summary}

def source_build_event_input_tables(project_root: Path, output_dir: Path, tables: dict[str, pd.DataFrame], manifest_rows: list[dict[str, object]]) -> dict[str, pd.DataFrame]:
    apa_phase = base.load_apa_phase_tables(project_root)
    apa_membership = base.load_apa_membership(project_root)
    se_universe = base.load_se_universe(project_root)
    se_events = base.load_se_events(project_root)
    input_summary = pd.DataFrame([{'input': 'APA phase lags', 'n_rows': len(apa_phase), 'n_genes_or_events': apa_phase.get('host_ensg', pd.Series(dtype=str)).nunique()}, {'input': 'APA membership', 'n_rows': len(apa_membership), 'n_genes_or_events': apa_membership.get('host_ensg', pd.Series(dtype=str)).nunique()}, {'input': 'SE universe', 'n_rows': len(se_universe), 'n_genes_or_events': se_universe.get('event_id', pd.Series(dtype=str)).nunique()}, {'input': 'Rhythmic SE p0.05 (3.1.2 canonical)', 'n_rows': len(se_events), 'n_genes_or_events': se_events.get('event_id', pd.Series(dtype=str)).nunique()}])
    source__save_table(tables, manifest_rows, output_dir, 'event_inputs', 'apa_expression_phase_lag_inputs.tsv', apa_phase)
    source__save_table(tables, manifest_rows, output_dir, 'event_inputs', 'apa_host_gene_membership_inputs.tsv', apa_membership)
    source__save_table(tables, manifest_rows, output_dir, 'event_inputs', 'se_variable_event_universe.tsv', se_universe)
    source__save_table(tables, manifest_rows, output_dir, 'event_inputs', 'se_rhythmic_events_p0.05.tsv', se_events)
    source__save_table(tables, manifest_rows, output_dir, 'event_inputs', 'analysis_input_summary.tsv', input_summary)
    return {'apa_phase': apa_phase, 'apa_membership': apa_membership, 'se_universe': se_universe, 'se_events': se_events, 'input_summary': input_summary}

def source_build_clip_source_tables(project_root: Path, output_dir: Path, tables: dict[str, pd.DataFrame], manifest_rows: list[dict[str, object]], catalog: pd.DataFrame, expression: pd.DataFrame, se_universe: pd.DataFrame, apa_membership: pd.DataFrame, use_clip_cache: bool=True) -> dict[str, pd.DataFrame]:
    clip_manifest = base.discover_clip_files(project_root)
    clip_links, clip_coverage, clip_filtering_qc = base.build_clip_links(project_root, catalog=catalog, expression=expression, se_events=se_universe, apa_membership=apa_membership, use_cache=use_clip_cache, peak_evidence_mode='consensus', consensus_score_threshold=base.CONSENSUS_SCORE_THRESHOLD, consensus_min_accessions=base.CONSENSUS_MIN_ACCESSIONS, consensus_min_peak_support=base.CONSENSUS_MIN_PEAKS, consensus_merge_distance_bp=base.CONSENSUS_MERGE_DISTANCE_BP, return_qc=True)
    catalog = catalog.copy()
    catalog['clip_available'] = catalog['gene_symbol'].isin(set(clip_coverage.get('rbp_symbol', [])))
    source__save_table(tables, manifest_rows, output_dir, 'clip_links', 'rbp_catalog.tsv', catalog)
    source__save_table(tables, manifest_rows, output_dir, 'clip_links', 'clip_source_manifest.tsv', clip_manifest)
    source__save_table(tables, manifest_rows, output_dir, 'clip_links', 'clip_supported_rbp_target_links.tsv', clip_links)
    source__save_table(tables, manifest_rows, output_dir, 'clip_links', 'clip_supported_rbp_target_links_consensus.tsv', clip_links)
    source__save_table(tables, manifest_rows, output_dir, 'clip_links', 'clip_coverage_summary.tsv', clip_coverage)
    source__save_table(tables, manifest_rows, output_dir, 'clip_links', 'clip_consensus_filtering_manifest.tsv', clip_filtering_qc)
    return {'catalog': catalog, 'clip_manifest': clip_manifest, 'clip_links': clip_links, 'clip_coverage': clip_coverage, 'clip_filtering_qc': clip_filtering_qc}

def source_build_rbp_phase_tables(output_dir: Path, tables: dict[str, pd.DataFrame], manifest_rows: list[dict[str, object]], rbp_expression: pd.DataFrame, bandwidth: float=0.3) -> dict[str, pd.DataFrame]:
    cohort_order = ['TCGA_Tumor', 'GTEx_Normal', 'CPTAC_Tumor']
    phase_df = rbp_expression[rbp_expression['rhythmic']].dropna(subset=['acrophase_hours']).copy()
    phase_density_frames = []
    phase_polarization_rows = []
    for cohort in cohort_order:
        values = phase_df.loc[phase_df['cohort_key'].eq(cohort), 'acrophase_hours']
        density = source_circular_density(values, bw=bandwidth)
        if not density.empty:
            density['cohort_key'] = cohort
            phase_density_frames.append(density)
        phase_polarization_rows.append({'cohort_key': cohort, **source_axial_phase_summary(values)})
    phase_density = pd.concat(phase_density_frames, ignore_index=True, sort=False) if phase_density_frames else base.empty_table(['theta', 'phase_hours', 'density', 'cohort_key'])
    phase_polarization = pd.DataFrame(phase_polarization_rows)
    source__save_table(tables, manifest_rows, output_dir, 'rbp_phase', 'rbp_phase_circular_density.tsv', phase_density)
    source__save_table(tables, manifest_rows, output_dir, 'rbp_phase', 'rbp_phase_polarization_summary.tsv', phase_polarization)
    return {'rbp_phase_density': phase_density, 'rbp_phase_polarization': phase_polarization}

def source_build_overlap_tables(output_dir: Path, tables: dict[str, pd.DataFrame], manifest_rows: list[dict[str, object]], rbp_expression: pd.DataFrame) -> dict[str, pd.DataFrame]:
    overlap_rows = []
    overlap_gene_rows = []
    for comparison, cfg in base.COMPARISONS.items():
        tumor_key = cfg['tumor_key']
        normal_key = cfg['normal_key']
        tested = set(rbp_expression.loc[rbp_expression['cohort_key'].isin([tumor_key, normal_key]), 'gene_symbol'].map(base.standardize_symbol))
        tumor_set = set(rbp_expression.loc[rbp_expression['cohort_key'].eq(tumor_key) & rbp_expression['rhythmic'], 'gene_symbol'].map(base.standardize_symbol))
        normal_set = set(rbp_expression.loc[rbp_expression['cohort_key'].eq(normal_key) & rbp_expression['rhythmic'], 'gene_symbol'].map(base.standardize_symbol))
        categories = {'shared': tumor_set & normal_set, 'tumor_only': tumor_set - normal_set, 'normal_only': normal_set - tumor_set, 'not_rhythmic': tested - tumor_set - normal_set}
        for category, genes in categories.items():
            overlap_rows.append({'comparison': comparison, 'tumor_key': tumor_key, 'normal_key': normal_key, 'category': category, 'n_rbps': len(genes), 'rbp_symbols': ','.join(sorted(genes))})
            for gene in sorted(genes):
                overlap_gene_rows.append({'comparison': comparison, 'tumor_key': tumor_key, 'normal_key': normal_key, 'category': category, 'gene_symbol': gene})
    overlap_summary = pd.DataFrame(overlap_rows)
    overlap_genes = pd.DataFrame(overlap_gene_rows)
    source__save_table(tables, manifest_rows, output_dir, 'overlap', 'rbp_tumor_normal_rhythmic_overlap.tsv', overlap_summary)
    source__save_table(tables, manifest_rows, output_dir, 'overlap', 'rbp_tumor_normal_rhythmic_overlap_genes.tsv', overlap_genes)
    return {'rbp_overlap_summary': overlap_summary, 'rbp_overlap_genes': overlap_genes}

def source_build_reactome_overlap_tables(workspace_root: Path, output_dir: Path, tables: dict[str, pd.DataFrame], manifest_rows: list[dict[str, object]], catalog: pd.DataFrame, rbp_overlap_genes: pd.DataFrame) -> dict[str, pd.DataFrame]:
    reactome_path = FIG05_REACTOME_FILE
    reactome_cols = ['ensembl_id', 'pathway_id', 'url', 'pathway_name', 'evidence', 'species']
    category_focus = ['tumor_only', 'shared', 'normal_only']
    category_label_map = {'tumor_only': 'Tumor only', 'shared': 'Shared', 'normal_only': 'GTEx only'}
    if reactome_path.exists():
        reactome = pd.read_csv(reactome_path, sep='\t', names=reactome_cols, dtype=str)
        reactome['ensembl_id'] = reactome['ensembl_id'].map(base.strip_version)
        reactome = reactome[reactome['species'].eq('Homo sapiens') & reactome['ensembl_id'].ne('') & reactome['pathway_id'].notna() & reactome['pathway_name'].notna()].drop_duplicates(['ensembl_id', 'pathway_id'])
    else:
        reactome = base.empty_table(reactome_cols)
    rbp_ensembl_map = catalog[['gene_symbol', 'ensembl_id']].copy()
    rbp_ensembl_map['gene_symbol'] = rbp_ensembl_map['gene_symbol'].map(base.standardize_symbol)
    rbp_ensembl_map['ensembl_id'] = rbp_ensembl_map['ensembl_id'].map(base.strip_version)
    rbp_ensembl_map = rbp_ensembl_map[rbp_ensembl_map['gene_symbol'].ne('') & rbp_ensembl_map['ensembl_id'].ne('')].drop_duplicates()
    ensembl_to_symbols = rbp_ensembl_map.groupby('ensembl_id')['gene_symbol'].apply(lambda x: sorted(set(x))).to_dict()
    reactome_background = set(rbp_ensembl_map['ensembl_id']) & set(reactome.get('ensembl_id', pd.Series(dtype=str)))
    pathway_records = []
    if len(reactome) and reactome_background:
        for (pathway_id, pathway_name, url), grp in reactome.groupby(['pathway_id', 'pathway_name', 'url'], dropna=False):
            pathway_genes = set(grp['ensembl_id']) & reactome_background
            if len(pathway_genes) < 2:
                continue
            pathway_records.append({'pathway_id': pathway_id, 'pathway_name': pathway_name, 'url': url, 'pathway_background_genes': pathway_genes, 'pathway_background_size': len(pathway_genes)})

    def symbols_for_ensembl(ids: set[str]) -> list[str]:
        symbols = []
        for ensg in sorted(set(ids)):
            symbols.extend(ensembl_to_symbols.get(ensg, []))
        return sorted(set(symbols))
    query_rows = []
    enrichment_frames = []
    for comparison in ['TCGA_vs_GTEx', 'CPTAC_vs_GTEx']:
        for category in category_focus:
            query_symbols = sorted(set(rbp_overlap_genes.loc[rbp_overlap_genes['comparison'].eq(comparison) & rbp_overlap_genes['category'].eq(category), 'gene_symbol'].map(base.standardize_symbol)))
            query_map = pd.DataFrame({'gene_symbol': query_symbols}).merge(rbp_ensembl_map, on='gene_symbol', how='left')
            query_ensembl = set(query_map['ensembl_id'].dropna().map(base.strip_version))
            query_ensembl = {x for x in query_ensembl if x} & reactome_background
            query_rows.append({'comparison': comparison, 'category': category, 'category_label': category_label_map[category], 'n_query_symbols': len(query_symbols), 'n_query_symbols_with_ensembl': int(query_map['ensembl_id'].dropna().map(base.strip_version).ne('').sum()), 'n_query_rbps_in_reactome_background': len(query_ensembl), 'query_symbols': ','.join(query_symbols), 'query_symbols_in_reactome': ','.join(symbols_for_ensembl(query_ensembl))})
            rows = []
            universe = set(reactome_background)
            for pathway in pathway_records:
                pathway_genes = set(pathway['pathway_background_genes'])
                overlap_genes = query_ensembl & pathway_genes
                a = len(overlap_genes)
                b = len(query_ensembl - pathway_genes)
                c = len(pathway_genes - query_ensembl)
                d = len(universe - (query_ensembl | pathway_genes))
                if base.fisher_exact is None or len(query_ensembl) == 0:
                    odds_ratio = np.nan
                    p_value = np.nan
                else:
                    odds_ratio, p_value = base.fisher_exact([[a, b], [c, d]], alternative='greater')
                rows.append({'comparison': comparison, 'category': category, 'category_label': category_label_map[category], 'pathway_id': pathway['pathway_id'], 'pathway_name': pathway['pathway_name'], 'url': pathway['url'], 'n_background_rbps': len(universe), 'n_query_rbps': len(query_ensembl), 'pathway_background_size': pathway['pathway_background_size'], 'overlap_count': a, 'odds_ratio': odds_ratio, 'p_value': p_value, 'overlap_ensembl': ','.join(sorted(overlap_genes)), 'overlap_symbols': ','.join(symbols_for_ensembl(overlap_genes))})
            group_df = pd.DataFrame(rows)
            if len(group_df):
                group_df['fdr'] = base.bh_fdr(group_df['p_value'])
                enrichment_frames.append(group_df)
    query_sets = pd.DataFrame(query_rows)
    enrichment = pd.concat(enrichment_frames, ignore_index=True, sort=False) if enrichment_frames else base.empty_table(['comparison', 'category', 'category_label', 'pathway_id', 'pathway_name', 'url', 'n_background_rbps', 'n_query_rbps', 'pathway_background_size', 'overlap_count', 'odds_ratio', 'p_value', 'fdr', 'overlap_ensembl', 'overlap_symbols'])
    if len(enrichment):
        enrichment = enrichment.sort_values(['comparison', 'category', 'p_value', 'fdr', 'overlap_count'], ascending=[True, True, True, True, False])
        top = enrichment[enrichment['overlap_count'].gt(0)].sort_values(['comparison', 'category', 'p_value', 'fdr', 'overlap_count'], ascending=[True, True, True, True, False]).groupby(['comparison', 'category'], dropna=False).head(8).reset_index(drop=True)
    else:
        top = base.empty_table(enrichment.columns)
    manifest = pd.DataFrame([{'source': 'Reactome_anno_HS', 'source_path': str(reactome_path), 'source_exists': bool(reactome_path.exists()), 'background': 'RBP catalog genes mapped to Ensembl IDs and present in Reactome', 'n_reactome_rows': int(len(reactome)), 'n_reactome_pathways_tested': int(len(pathway_records)), 'n_background_rbps': int(len(reactome_background)), 'min_pathway_background_rbps': 2, 'test': 'one-sided Fisher exact test with BH FDR per comparison/category query set'}])
    source__save_table(tables, manifest_rows, output_dir, 'reactome_overlap', 'rbp_overlap_reactome_query_sets.tsv', query_sets)
    source__save_table(tables, manifest_rows, output_dir, 'reactome_overlap', 'rbp_overlap_reactome_enrichment.tsv', enrichment)
    source__save_table(tables, manifest_rows, output_dir, 'reactome_overlap', 'rbp_overlap_reactome_enrichment_top.tsv', top)
    source__save_table(tables, manifest_rows, output_dir, 'reactome_overlap', 'rbp_overlap_reactome_manifest.tsv', manifest)
    return {'rbp_overlap_pathway_query_sets': query_sets, 'rbp_overlap_pathway_enrichment': enrichment, 'rbp_overlap_pathway_top': top, 'rbp_overlap_pathway_manifest': manifest}

def source_build_apa_link_tables(output_dir: Path, tables: dict[str, pd.DataFrame], manifest_rows: list[dict[str, object]], clip_links: pd.DataFrame, rbp_expression: pd.DataFrame, apa_membership: pd.DataFrame, apa_phase: pd.DataFrame) -> dict[str, pd.DataFrame]:
    enrichment_frames = []
    lag_frames = []
    for comparison, cfg in base.COMPARISONS.items():
        cohort_key = cfg['tumor_key']
        membership = apa_membership[apa_membership['comparison'].eq(comparison)].copy()
        if membership.empty:
            continue
        rhythmic_col = f'{source__cohort_prefix(comparison)}_apa_rhythmic'
        rhythmic = source__rhythmic_mask(membership, rhythmic_col)
        universe = membership['host_ensg'].map(base.strip_version)
        positive = membership.loc[rhythmic, 'host_ensg'].map(base.strip_version)
        enr = base.compute_gene_target_enrichment(clip_links=clip_links, rbp_expression=rbp_expression, universe_genes=universe, positive_genes=positive, comparison=comparison, cohort_key=cohort_key, region_keywords=('3UTR', 'gene_level_clip'), min_clip_targets=3)
        if not enr.empty:
            enrichment_frames.append(enr)
        lags = base.compute_apa_phase_lags(clip_links=clip_links, rbp_expression=rbp_expression, apa_phase=apa_phase, comparison=comparison, cohort_key=cohort_key)
        if not lags.empty:
            lag_frames.append(lags)
    enrichment = pd.concat(enrichment_frames, ignore_index=True) if enrichment_frames else base.empty_table(['comparison', 'cohort_key', 'rbp_symbol', 'n_universe_genes', 'n_positive_genes', 'n_clip_targets_in_universe', 'n_positive_clip_targets', 'odds_ratio', 'p_value', 'fdr', 'regions_used'])
    lags = pd.concat(lag_frames, ignore_index=True) if lag_frames else base.empty_table(['comparison', 'cohort_key', 'event_kind', 'rbp_symbol', 'target_key', 'host_ensg', 'host_gene_symbol', 'apa_event_id', 'region', 'source', 'evidence_type', 'genome_build', 'rbp_acrophase_hours', 'event_phase_hours', 'target_mrna_phase_hours', 'rbp_to_event_lag_hours', 'event_to_mrna_lag_hours', 'rbp_amplitude', 'amplitude', 'expr_amplitude', 'fdr', 'expr_q', 'clip_file'])
    source__save_table(tables, manifest_rows, output_dir, 'apa_links', 'rbp_apa_clip_enrichment.tsv', enrichment)
    source__save_table(tables, manifest_rows, output_dir, 'apa_links', 'rbp_apa_phase_lags.tsv', lags)
    return {'apa_enrichment': enrichment, 'apa_lags': lags}

def source_build_se_link_tables(output_dir: Path, tables: dict[str, pd.DataFrame], manifest_rows: list[dict[str, object]], clip_links: pd.DataFrame, rbp_expression: pd.DataFrame, expression: pd.DataFrame, se_universe: pd.DataFrame, se_events: pd.DataFrame) -> dict[str, pd.DataFrame]:
    enrichment_frames = []
    lag_frames = []
    for comparison, cfg in base.COMPARISONS.items():
        cohort_key = cfg['tumor_key']
        universe = se_universe.loc[se_universe['comparison'].eq(comparison), 'event_id'].astype(str)
        positive = se_events.loc[se_events['comparison'].eq(comparison), 'event_id'].astype(str)
        enr = base.compute_event_target_enrichment(clip_links=clip_links, rbp_expression=rbp_expression, universe_events=universe, positive_events=positive, comparison=comparison, cohort_key=cohort_key, region_keywords=('SE_',), min_clip_events=3)
        if not enr.empty:
            enrichment_frames.append(enr)
        lags = base.compute_se_phase_lags(clip_links=clip_links, rbp_expression=rbp_expression, se_events=se_events, expression=expression, comparison=comparison, cohort_key=cohort_key)
        if not lags.empty:
            lag_frames.append(lags)
    enrichment = pd.concat(enrichment_frames, ignore_index=True) if enrichment_frames else base.empty_table(['comparison', 'cohort_key', 'rbp_symbol', 'n_universe_events', 'n_positive_events', 'n_clip_events_in_universe', 'n_positive_clip_events', 'odds_ratio', 'p_value', 'fdr', 'regions_used'])
    lags = pd.concat(lag_frames, ignore_index=True) if lag_frames else base.empty_table(['comparison', 'cohort_key', 'event_kind', 'rbp_symbol', 'event_id', 'gene_id', 'gene_symbol', 'region', 'source', 'evidence_type', 'genome_build', 'rbp_acrophase_hours', 'event_phase_hours', 'target_mrna_phase_hours', 'rbp_to_event_lag_hours', 'event_to_mrna_lag_hours', 'rbp_amplitude', 'amplitude', 'expr_amplitude', 'fdr', 'expr_q', 'clip_file'])
    source__save_table(tables, manifest_rows, output_dir, 'se_links', 'rbp_se_clip_enrichment.tsv', enrichment)
    source__save_table(tables, manifest_rows, output_dir, 'se_links', 'rbp_se_phase_lags.tsv', lags)
    return {'se_enrichment': enrichment, 'se_lags': lags}

def source_build_target_contrast_tables(output_dir: Path, tables: dict[str, pd.DataFrame], manifest_rows: list[dict[str, object]], clip_links: pd.DataFrame, rbp_expression: pd.DataFrame, apa_membership: pd.DataFrame, apa_phase: pd.DataFrame, se_universe: pd.DataFrame, se_events: pd.DataFrame, apa_lags: pd.DataFrame, se_lags: pd.DataFrame, n_lag_permutations: int=500) -> dict[str, pd.DataFrame]:
    apa_contrast_frames = []
    se_contrast_frames = []
    for comparison, cfg in base.COMPARISONS.items():
        cohort_key = cfg['tumor_key']
        membership = apa_membership[apa_membership['comparison'].eq(comparison)].copy()
        if not membership.empty:
            rhythmic_col = f'{source__cohort_prefix(comparison)}_apa_rhythmic'
            rhythmic = source__rhythmic_mask(membership, rhythmic_col)
            apa_contrast = base.compute_target_background_contrast(clip_links=clip_links, rbp_expression=rbp_expression, universe_ids=membership['host_ensg'].map(base.strip_version), positive_ids=membership.loc[rhythmic, 'host_ensg'].map(base.strip_version), phase_table=apa_phase[apa_phase['comparison'].eq(comparison)], phase_id_col='host_ensg', phase_col='apa_acrophase_hours', comparison=comparison, cohort_key=cohort_key, event_kind='APA', id_type='gene', region_keywords=('3UTR', 'gene_level_clip'), min_clip_targets=3)
            if not apa_contrast.empty:
                apa_contrast_frames.append(apa_contrast)
        se_contrast = base.compute_target_background_contrast(clip_links=clip_links, rbp_expression=rbp_expression, universe_ids=se_universe.loc[se_universe['comparison'].eq(comparison), 'event_id'].astype(str), positive_ids=se_events.loc[se_events['comparison'].eq(comparison), 'event_id'].astype(str), phase_table=se_events[se_events['comparison'].eq(comparison)], phase_id_col='event_id', phase_col='acrophase_hours', comparison=comparison, cohort_key=cohort_key, event_kind='SE', id_type='event', region_keywords=('SE_',), min_clip_targets=3)
        if not se_contrast.empty:
            se_contrast_frames.append(se_contrast)
    apa_target_contrast = pd.concat(apa_contrast_frames, ignore_index=True) if apa_contrast_frames else base.empty_table([])
    se_target_contrast = pd.concat(se_contrast_frames, ignore_index=True) if se_contrast_frames else base.empty_table([])
    target_contrast = pd.concat([apa_target_contrast, se_target_contrast], ignore_index=True, sort=False) if len(apa_target_contrast) or len(se_target_contrast) else base.empty_table([])
    lag_input = pd.concat([apa_lags, se_lags], ignore_index=True, sort=False)
    lag_coherence = base.summarize_phase_lag_coherence(lag_input, lag_col='rbp_to_event_lag_hours', min_lags=5, n_permutations=n_lag_permutations, random_seed=42)
    source__save_table(tables, manifest_rows, output_dir, 'target_contrast', 'rbp_apa_target_background_contrast.tsv', apa_target_contrast)
    source__save_table(tables, manifest_rows, output_dir, 'target_contrast', 'rbp_se_target_background_contrast.tsv', se_target_contrast)
    source__save_table(tables, manifest_rows, output_dir, 'target_contrast', 'rbp_target_background_contrast.tsv', target_contrast)
    source__save_table(tables, manifest_rows, output_dir, 'target_contrast', 'rbp_phase_lag_coherence.tsv', lag_coherence)
    return {'apa_target_contrast': apa_target_contrast, 'se_target_contrast': se_target_contrast, 'target_contrast': target_contrast, 'lag_coherence': lag_coherence}

def source_build_triplet_candidate_tables(output_dir: Path, tables: dict[str, pd.DataFrame], manifest_rows: list[dict[str, object]], apa_enrichment: pd.DataFrame, se_enrichment: pd.DataFrame, apa_lags: pd.DataFrame, se_lags: pd.DataFrame, target_contrast: pd.DataFrame, lag_coherence: pd.DataFrame) -> dict[str, pd.DataFrame]:
    triplets = base.build_triplets(apa_lags, se_lags)
    enrichment_for_candidates = pd.concat([apa_enrichment.assign(event_kind='APA'), se_enrichment.assign(event_kind='SE')], ignore_index=True, sort=False)
    if enrichment_for_candidates.empty:
        candidates = base.empty_table([])
    else:
        enrichment_for_candidates['sig_fdr05'] = pd.to_numeric(enrichment_for_candidates['fdr'], errors='coerce').lt(0.05)
        candidate_rows = []
        for (event_kind, rbp_symbol), grp in enrichment_for_candidates.groupby(['event_kind', 'rbp_symbol'], dropna=False):
            sig = grp[grp['sig_fdr05']].copy()
            finite_or = pd.to_numeric(grp['odds_ratio'], errors='coerce').replace(np.inf, np.nan)
            candidate_rows.append({'event_kind': event_kind, 'rbp_symbol': rbp_symbol, 'n_sig_enrichment_comparisons': int(sig['comparison'].nunique()), 'sig_enrichment_comparisons': ','.join(sorted(sig['comparison'].dropna().astype(str).unique())), 'best_enrichment_fdr': float(pd.to_numeric(grp['fdr'], errors='coerce').min()) if pd.to_numeric(grp['fdr'], errors='coerce').notna().any() else np.nan, 'max_enrichment_odds_ratio': float(finite_or.max()) if finite_or.notna().any() else np.nan})
        candidates = pd.DataFrame(candidate_rows)
        if not target_contrast.empty:
            contrast_summary = target_contrast.groupby(['event_kind', 'rbp_symbol'], dropna=False).agg(max_rate_difference=('rate_difference', 'max'), best_target_background_fdr=('fdr', 'min'), max_phase_resultant_delta=('phase_resultant_delta', 'max')).reset_index()
            candidates = candidates.merge(contrast_summary, on=['event_kind', 'rbp_symbol'], how='left')
        if not lag_coherence.empty:
            coherence_summary = lag_coherence.groupby(['event_kind', 'rbp_symbol'], dropna=False).agg(best_lag_coherence_fdr=('empirical_fdr', 'min'), max_lag_resultant_length=('lag_resultant_length', 'max'), max_lag_count=('n_lags', 'max')).reset_index()
            candidates = candidates.merge(coherence_summary, on=['event_kind', 'rbp_symbol'], how='left')
        if not triplets.empty:
            triplet_summary = triplets.groupby(['event_kind', 'rbp_symbol'], dropna=False).agg(n_triplet_rows=('triplet_key', 'size'), n_unique_triplets=('triplet_key', 'nunique'), n_replicated_rows=('validation_replicated', 'sum'), n_unique_targets=('target_gene_symbol', 'nunique')).reset_index()
            candidates = candidates.merge(triplet_summary, on=['event_kind', 'rbp_symbol'], how='left')
        for col in ['n_triplet_rows', 'n_unique_triplets', 'n_replicated_rows', 'n_unique_targets']:
            if col not in candidates.columns:
                candidates[col] = 0
            candidates[col] = candidates[col].fillna(0).astype(int)
        candidates['evidence_tier'] = 'exploratory'
        candidates.loc[candidates['event_kind'].eq('APA') & candidates['n_sig_enrichment_comparisons'].ge(2) & candidates['n_replicated_rows'].gt(0), 'evidence_tier'] = 'main_candidate'
        candidates.loc[candidates['event_kind'].eq('SE') & candidates['n_sig_enrichment_comparisons'].ge(1), 'evidence_tier'] = 'exploratory_se'
        tier_order = {'main_candidate': 0, 'exploratory_se': 1, 'exploratory': 2}
        candidates['evidence_tier_order'] = candidates['evidence_tier'].map(tier_order).fillna(9).astype(int)
        candidates = candidates.sort_values(['evidence_tier_order', 'n_sig_enrichment_comparisons', 'n_replicated_rows', 'best_enrichment_fdr'], ascending=[True, False, False, True]).drop(columns=['evidence_tier_order'])
    source__save_table(tables, manifest_rows, output_dir, 'triplets_candidates', 'rbp_event_gene_triplets.tsv', triplets)
    source__save_table(tables, manifest_rows, output_dir, 'triplets_candidates', 'rbp_main_figure_candidate_summary.tsv', candidates)
    return {'triplets': triplets, 'main_figure_candidates': candidates}

def source_generate_current_source_tables(project_root: str | Path, analysis: str=source_ANALYSIS, n_lag_permutations: int=500, use_clip_cache: bool=True) -> dict[str, object]:
    """Generate current upstream source tables used by the RBP-processing panels."""
    project_root = Path(project_root)
    workspace_root = project_root.parent
    dirs = base.ensure_output_dirs(project_root, analysis)
    output_dir = dirs['output']
    tables: dict[str, pd.DataFrame] = {}
    manifest_rows: list[dict[str, object]] = []
    core = source_build_catalog_expression_tables(project_root, output_dir, tables, manifest_rows)
    events = source_build_event_input_tables(project_root, output_dir, tables, manifest_rows)
    clip = source_build_clip_source_tables(project_root, output_dir, tables, manifest_rows, catalog=core['catalog'], expression=core['expression'], se_universe=events['se_universe'], apa_membership=events['apa_membership'], use_clip_cache=use_clip_cache)
    source_build_rbp_phase_tables(output_dir, tables, manifest_rows, core['rbp_expression'])
    overlap = source_build_overlap_tables(output_dir, tables, manifest_rows, core['rbp_expression'])
    reactome = source_build_reactome_overlap_tables(workspace_root, output_dir, tables, manifest_rows, clip['catalog'], overlap['rbp_overlap_genes'])
    apa = source_build_apa_link_tables(output_dir, tables, manifest_rows, clip['clip_links'], core['rbp_expression'], events['apa_membership'], events['apa_phase'])
    se = source_build_se_link_tables(output_dir, tables, manifest_rows, clip['clip_links'], core['rbp_expression'], core['expression'], events['se_universe'], events['se_events'])
    contrast = source_build_target_contrast_tables(output_dir, tables, manifest_rows, clip['clip_links'], core['rbp_expression'], events['apa_membership'], events['apa_phase'], events['se_universe'], events['se_events'], apa['apa_lags'], se['se_lags'], n_lag_permutations=n_lag_permutations)
    triplets = source_build_triplet_candidate_tables(output_dir, tables, manifest_rows, apa['apa_enrichment'], se['se_enrichment'], apa['apa_lags'], se['se_lags'], contrast['target_contrast'], contrast['lag_coherence'])
    manifest = pd.DataFrame(manifest_rows)
    manifest_path = base.write_tsv(manifest, output_dir / 'rbp_processing_source_table_generation_manifest.tsv')
    missing_current = []
    for filename in source_CURRENT_PANEL_SOURCE_TABLES:
        path = output_dir / filename
        if not path.exists():
            missing_current.append(filename)
    if missing_current:
        raise FileNotFoundError(f'Missing current panel source tables after generation: {missing_current}')
    return {'manifest': manifest, 'manifest_path': manifest_path, 'tables': tables, 'core': core, 'events': events, 'clip': clip, 'overlap': overlap, 'reactome': reactome, 'apa': apa, 'se': se, 'contrast': contrast, 'triplets': triplets}

def source_main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--project-root', type=Path, default=Path(__file__).resolve().parents[1])
    parser.add_argument('--analysis', default=source_ANALYSIS)
    parser.add_argument('--n-lag-permutations', type=int, default=500)
    parser.add_argument('--no-clip-cache', action='store_true')
    args = parser.parse_args()
    results = source_generate_current_source_tables(project_root=args.project_root, analysis=args.analysis, n_lag_permutations=args.n_lag_permutations, use_clip_cache=not args.no_clip_cache)
    manifest = results['manifest']
    print(f"Wrote {len(manifest)} source-table records to {results['manifest_path']}")
    print(manifest[['stage', 'table', 'rows', 'columns']].to_string(index=False))


In [ ]:
# Cell 6: Activate the visible source-table namespace.
for _name, _value in list(globals().items()):
    if _name.startswith('source_'):
        setattr(source, _name.removeprefix('source_'), _value)


In [ ]:
# Cell 7: Embedded and namespace-isolated from scripts/rbp_rna_processing_deep_dive.py
"""Deeper RBP-centered APA and skipped-exon mechanism summaries.

This module builds on outputs from ``rbp_rna_processing_links.py``.  It avoids
recomputing CLIP overlaps and instead adds event-level annotations, phase-module
summaries, positional SE binding summaries, and candidate prioritization.
"""
from __future__ import annotations
from dataclasses import dataclass
from math import log10
import os
from pathlib import Path
from typing import Iterable
import numpy as np
import pandas as pd
deep_ANALYSIS = 'rbp_rna_processing_links'
deep_FIGURE_FORMATS = ('pdf', 'png')
deep_PALETTE = {'blue': '#0072B2', 'orange': '#D55E00', 'green': '#009E73', 'purple': '#CC79A7', 'gold': '#E69F00', 'sky': '#56B4E9', 'gray': '#7A7A7A', 'light_gray': '#E6E6E6', 'dark': '#222222'}

@dataclass(frozen=True)
class deep_DeepDiveOutputs:
    annotated_apa: Path
    apa_utr_bias: Path
    phase_modules: Path
    se_region_distribution: Path
    se_region_bias: Path
    candidate_priority: Path
    manifest: Path

def deep_resolve_project_root(project_root: str | os.PathLike[str] | None=None) -> Path:
    if project_root is not None:
        return Path(project_root).expanduser().resolve()
    workspace_root = Path(os.environ.get('WORKSPACE_ROOT', '/home/wdeng3/workspace/Codespace'))
    candidates = [workspace_root / 'PDAC_Rhythmicity', Path.cwd() / 'PDAC_Rhythmicity', Path.cwd(), Path(__file__).resolve().parents[1]]
    for candidate in candidates:
        if candidate.name == 'PDAC_Rhythmicity' and (candidate / 'data').exists():
            return candidate.resolve()
    return Path(__file__).resolve().parents[1]

def deep__read_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    sep = '\t' if path.suffix.lower() in {'.tsv', '.txt', '.bed'} else ','
    return pd.read_csv(path, sep=sep, low_memory=False)

def deep__write_tsv(df: pd.DataFrame, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, sep='\t', index=False)
    return path

def deep__relative_path(path: Path, root: Path) -> str:
    try:
        return str(path.relative_to(root))
    except ValueError:
        return str(path)

def deep__minus_log10(value: object, cap: float=20.0) -> float:
    numeric = pd.to_numeric(pd.Series([value]), errors='coerce').iloc[0]
    if pd.isna(numeric) or numeric <= 0:
        return 0.0
    return float(min(-log10(float(numeric)), cap))

def deep__safe_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors='coerce')

def deep__event_key(frame: pd.DataFrame) -> pd.Series:
    if 'apa_event_id' in frame.columns and frame['apa_event_id'].notna().any():
        apa = frame['apa_event_id'].fillna('').astype(str)
    else:
        apa = pd.Series('', index=frame.index)
    if 'event_id' in frame.columns:
        se = frame['event_id'].fillna('').astype(str)
    else:
        se = pd.Series('', index=frame.index)
    out = apa.where(apa.ne(''), se)
    return out.map(base.standardize_event_id)

def deep__bh_fdr(p_values: Iterable[object]) -> np.ndarray:
    values = pd.to_numeric(pd.Series(list(p_values)), errors='coerce').to_numpy(dtype=float)
    out = np.full(values.shape, np.nan, dtype=float)
    mask = np.isfinite(values)
    if not mask.any():
        return out
    finite = values[mask]
    order = np.argsort(finite)
    ranked = finite[order]
    n = len(ranked)
    adjusted = ranked * n / np.arange(1, n + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    q = np.empty_like(finite)
    q[order] = np.clip(adjusted, 0.0, 1.0)
    out[mask] = q
    return out

def deep__circular_mean_lag(lags: Iterable[object]) -> dict[str, float]:
    values = pd.to_numeric(pd.Series(list(lags)), errors='coerce').dropna()
    values = values[values.between(-12.0, 12.0)]
    if values.empty:
        return {'n': 0, 'mean_lag_hours': np.nan, 'median_lag_hours': np.nan, 'resultant_length': np.nan}
    theta = values.to_numpy(dtype=float) % 24.0 / 24.0 * 2.0 * np.pi
    z = np.mean(np.exp(1j * theta))
    mean = np.angle(z) / (2.0 * np.pi) * 24.0 % 24.0
    if mean > 12.0:
        mean -= 24.0
    return {'n': int(len(values)), 'mean_lag_hours': float(mean), 'median_lag_hours': float(values.median()), 'resultant_length': float(abs(z))}

def deep__lag_timing_class(mean_lag: float) -> str:
    if pd.isna(mean_lag):
        return 'insufficient'
    abs_lag = abs(float(mean_lag))
    if abs_lag <= 2.0:
        return 'near_synchronous'
    if abs_lag <= 6.0:
        return 'short_lag'
    if abs_lag <= 10.0:
        return 'long_lag'
    return 'anti_phase'

def deep__lag_order(mean_lag: float) -> str:
    if pd.isna(mean_lag):
        return 'insufficient'
    if abs(float(mean_lag)) <= 2.0:
        return 'synchronous'
    return 'event_after_rbp' if float(mean_lag) > 0 else 'event_before_rbp'

def deep_load_utr_annotation(project_root: Path) -> pd.DataFrame:
    path = project_root / 'data' / 'Ref' / 'hg38_extracted_3UTR.bed'
    columns = ['transcript_id', 'transcript_core', 'target_gene_symbol', 'chrom', 'strand', 'utr_length_bp', 'utr_span_bp', 'n_utr_intervals', 'n_gene_utr_isoforms', 'utr_length_rank', 'utr_length_quantile', 'utr_length_label']
    if not path.exists():
        return pd.DataFrame(columns=columns)
    bed = pd.read_csv(path, sep='\t', header=None, names=['chrom', 'start', 'end', 'name', 'score', 'strand'])
    parts = bed['name'].astype(str).str.split('|', expand=True)
    bed['transcript_id'] = parts[0].fillna('')
    bed['transcript_core'] = bed['transcript_id'].str.replace('\\.\\d+$', '', regex=True)
    bed['target_gene_symbol'] = parts[1].fillna('').map(base.standardize_symbol).replace('NA', '')
    bed['length'] = deep__safe_numeric(bed['end']) - deep__safe_numeric(bed['start'])
    bed = bed[bed['transcript_id'].ne('') & bed['target_gene_symbol'].ne('') & bed['length'].gt(0)].copy()
    if bed.empty:
        return pd.DataFrame(columns=columns)
    agg = bed.groupby(['transcript_id', 'transcript_core', 'target_gene_symbol', 'chrom', 'strand'], dropna=False).agg(utr_length_bp=('length', 'sum'), utr_start=('start', 'min'), utr_end=('end', 'max'), n_utr_intervals=('length', 'size')).reset_index()
    agg['utr_span_bp'] = deep__safe_numeric(agg['utr_end']) - deep__safe_numeric(agg['utr_start'])
    agg = agg.drop(columns=['utr_start', 'utr_end'])
    agg['n_gene_utr_isoforms'] = agg.groupby('target_gene_symbol')['transcript_id'].transform('nunique')
    agg['utr_length_rank'] = agg.groupby('target_gene_symbol')['utr_length_bp'].rank(method='average', pct=False)
    denom = (agg['n_gene_utr_isoforms'] - 1).replace(0, np.nan)
    agg['utr_length_quantile'] = (agg['utr_length_rank'] - 1) / denom
    agg.loc[agg['n_gene_utr_isoforms'].eq(1), 'utr_length_quantile'] = 0.5
    agg['utr_length_label'] = 'middle_utr'
    agg.loc[agg['n_gene_utr_isoforms'].eq(1), 'utr_length_label'] = 'single_annotated_utr'
    agg.loc[agg['n_gene_utr_isoforms'].gt(1) & agg['utr_length_quantile'].le(0.25), 'utr_length_label'] = 'short_utr_isoform'
    agg.loc[agg['n_gene_utr_isoforms'].gt(1) & agg['utr_length_quantile'].ge(0.75), 'utr_length_label'] = 'long_utr_isoform'
    return agg[columns].copy()

def deep_annotate_apa_utr_rank(apa_phase: pd.DataFrame, utr: pd.DataFrame) -> pd.DataFrame:
    out = apa_phase.copy()
    if out.empty:
        return out
    out['transcript_id_clean'] = out.get('transcript_id', out.get('apa_event_id', '')).fillna('').astype(str).str.split('|').str[0]
    out['transcript_core'] = out['transcript_id_clean'].str.replace('\\.\\d+$', '', regex=True)
    if utr.empty:
        for col in ['utr_length_bp', 'utr_span_bp', 'n_utr_intervals', 'n_gene_utr_isoforms', 'utr_length_rank', 'utr_length_quantile', 'utr_length_label']:
            out[col] = np.nan if col != 'utr_length_label' else 'missing_utr_annotation'
        return out
    full_map = utr.drop_duplicates('transcript_id').set_index('transcript_id')
    core_map = utr.sort_values('utr_length_bp').drop_duplicates('transcript_core').set_index('transcript_core')
    merge_cols = ['target_gene_symbol', 'chrom', 'strand', 'utr_length_bp', 'utr_span_bp', 'n_utr_intervals', 'n_gene_utr_isoforms', 'utr_length_rank', 'utr_length_quantile', 'utr_length_label']
    full = out['transcript_id_clean'].map(full_map[merge_cols].to_dict('index'))
    core = out['transcript_core'].map(core_map[merge_cols].to_dict('index'))
    rows = []
    for full_item, core_item in zip(full, core):
        rows.append(full_item if isinstance(full_item, dict) else core_item if isinstance(core_item, dict) else {})
    annot = pd.DataFrame(rows, index=out.index)
    for col in merge_cols:
        out[col] = annot[col] if col in annot else np.nan
    out['utr_annotation_available'] = out['utr_length_bp'].notna()
    out['utr_length_label'] = out['utr_length_label'].fillna('missing_utr_annotation')
    return out

def deep_compute_apa_utr_bias(annotated_apa: pd.DataFrame, apa_lags: pd.DataFrame, n_permutations: int=2000, seed: int=42) -> pd.DataFrame:
    if annotated_apa.empty or apa_lags.empty:
        return pd.DataFrame()
    rng = np.random.default_rng(seed)
    apa = annotated_apa[annotated_apa['utr_annotation_available'].astype(bool)].copy()
    apa['event_key'] = apa['apa_event_id'].map(base.standardize_event_id)
    apa = apa.drop_duplicates(['comparison', 'event_key'])
    lag = apa_lags.copy()
    lag['event_key'] = lag['apa_event_id'].map(base.standardize_event_id)
    lag = lag.drop_duplicates(['comparison', 'rbp_symbol', 'event_key'])
    rows = []
    for (comparison, rbp_symbol), grp in lag.groupby(['comparison', 'rbp_symbol'], dropna=False):
        background = apa[apa['comparison'].eq(comparison)].copy()
        if background.empty:
            continue
        bound_keys = set(grp['event_key'])
        bound = background[background['event_key'].isin(bound_keys)].copy()
        if len(bound) < 8 or len(background) < 20:
            continue
        bound_q = deep__safe_numeric(bound['utr_length_quantile']).dropna().to_numpy(dtype=float)
        bg_q = deep__safe_numeric(background['utr_length_quantile']).dropna().to_numpy(dtype=float)
        bound_len = deep__safe_numeric(bound['utr_length_bp']).dropna().to_numpy(dtype=float)
        bg_len = deep__safe_numeric(background['utr_length_bp']).dropna().to_numpy(dtype=float)
        if len(bound_q) < 8 or len(bg_q) < 20:
            continue
        observed_delta = float(np.nanmedian(bound_q) - np.nanmedian(bg_q))
        if len(bg_q) >= len(bound_q):
            perm_idx = rng.integers(0, len(bg_q), size=(int(n_permutations), len(bound_q)))
            perm_delta = np.nanmedian(bg_q[perm_idx], axis=1) - np.nanmedian(bg_q)
            empirical_p = float((np.sum(np.abs(perm_delta) >= abs(observed_delta)) + 1) / (len(perm_delta) + 1))
        else:
            empirical_p = np.nan
        long_bound = float(np.mean(bound_q >= 0.75))
        long_bg = float(np.mean(bg_q >= 0.75))
        short_bound = float(np.mean(bound_q <= 0.25))
        short_bg = float(np.mean(bg_q <= 0.25))
        examples = bound.sort_values(['utr_length_quantile', 'fdr'], ascending=[False, True]).get('host_gene_symbol', pd.Series(dtype=str)).dropna().astype(str).head(8).tolist()
        rows.append({'comparison': comparison, 'rbp_symbol': rbp_symbol, 'n_bound_apa_events': int(len(bound)), 'n_background_apa_events': int(len(background)), 'median_bound_utr_length_bp': float(np.nanmedian(bound_len)) if len(bound_len) else np.nan, 'median_background_utr_length_bp': float(np.nanmedian(bg_len)) if len(bg_len) else np.nan, 'median_bound_utr_quantile': float(np.nanmedian(bound_q)), 'median_background_utr_quantile': float(np.nanmedian(bg_q)), 'delta_median_utr_quantile': observed_delta, 'long_utr_fraction_bound': long_bound, 'long_utr_fraction_background': long_bg, 'delta_long_utr_fraction': long_bound - long_bg, 'short_utr_fraction_bound': short_bound, 'short_utr_fraction_background': short_bg, 'delta_short_utr_fraction': short_bound - short_bg, 'empirical_p_value': empirical_p, 'top_long_utr_examples': ','.join(dict.fromkeys(examples))})
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out['empirical_fdr'] = deep__bh_fdr(out['empirical_p_value'])
    out['utr_bias_direction'] = 'mixed_utr_rank'
    out.loc[out['delta_median_utr_quantile'].ge(0.1), 'utr_bias_direction'] = 'longer_utr_rank'
    out.loc[out['delta_median_utr_quantile'].le(-0.1), 'utr_bias_direction'] = 'shorter_utr_rank'
    return out.sort_values(['empirical_fdr', 'n_bound_apa_events', 'delta_median_utr_quantile'], ascending=[True, False, False])

def deep_summarize_phase_modules(triplets: pd.DataFrame) -> pd.DataFrame:
    if triplets.empty:
        return pd.DataFrame()
    df = triplets.copy()
    df['event_key'] = deep__event_key(df)
    dedup_cols = ['comparison', 'event_kind', 'rbp_symbol', 'event_key', 'target_gene_id']
    df = df.sort_values(['max_score', 'n_accessions'], ascending=[False, False]).drop_duplicates(dedup_cols)
    rows = []
    for keys, grp in df.groupby(['comparison', 'event_kind', 'rbp_symbol'], dropna=False):
        comparison, event_kind, rbp_symbol = keys
        rbp_event = deep__circular_mean_lag(grp['rbp_to_event_lag_hours'])
        event_mrna = deep__circular_mean_lag(grp['event_to_mrna_lag_hours'])
        rbp_event_lag = deep__safe_numeric(grp['rbp_to_event_lag_hours'])
        event_mrna_lag = deep__safe_numeric(grp['event_to_mrna_lag_hours'])
        rows.append({'comparison': comparison, 'event_kind': event_kind, 'rbp_symbol': rbp_symbol, 'n_triplet_rows_deduplicated': int(len(grp)), 'n_events': int(grp['event_key'].nunique()), 'n_targets': int(grp['target_gene_id'].fillna('').replace('', np.nan).nunique()), 'n_validation_replicated_events': int(grp.loc[grp.get('validation_replicated', False).astype(bool), 'event_key'].nunique()), 'rbp_to_event_mean_lag_hours': rbp_event['mean_lag_hours'], 'rbp_to_event_median_lag_hours': rbp_event['median_lag_hours'], 'rbp_to_event_resultant_length': rbp_event['resultant_length'], 'event_to_mrna_mean_lag_hours': event_mrna['mean_lag_hours'], 'event_to_mrna_median_lag_hours': event_mrna['median_lag_hours'], 'event_to_mrna_resultant_length': event_mrna['resultant_length'], 'rbp_event_same_phase_fraction': float(rbp_event_lag.abs().le(2.0).mean()), 'rbp_event_antiphase_fraction': float((rbp_event_lag.abs() - 12.0).abs().le(2.0).mean()), 'event_mrna_same_phase_fraction': float(event_mrna_lag.abs().le(2.0).mean()), 'event_mrna_antiphase_fraction': float((event_mrna_lag.abs() - 12.0).abs().le(2.0).mean()), 'median_clip_max_score': float(deep__safe_numeric(grp['max_score']).median()) if 'max_score' in grp else np.nan, 'median_clip_n_accessions': float(deep__safe_numeric(grp['n_accessions']).median()) if 'n_accessions' in grp else np.nan})
    out = pd.DataFrame(rows)
    out['rbp_event_timing_class'] = out['rbp_to_event_mean_lag_hours'].map(deep__lag_timing_class)
    out['rbp_event_order'] = out['rbp_to_event_mean_lag_hours'].map(deep__lag_order)
    out['event_mrna_timing_class'] = out['event_to_mrna_mean_lag_hours'].map(deep__lag_timing_class)
    out['event_mrna_order'] = out['event_to_mrna_mean_lag_hours'].map(lambda x: 'mrna_after_event' if pd.notna(x) and x > 2 else 'mrna_before_event' if pd.notna(x) and x < -2 else 'synchronous')
    return out.sort_values(['event_kind', 'n_events', 'rbp_to_event_resultant_length'], ascending=[True, False, False])

def deep_summarize_se_region_bias(se_lags: pd.DataFrame, clip_links: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    if se_lags.empty:
        return (pd.DataFrame(), pd.DataFrame())
    se = se_lags.copy()
    se['event_key'] = se['event_id'].astype(str).map(base.standardize_event_id)
    se['region_group'] = se['region'].astype(str).str.replace('SE_', '', regex=False)
    se['region_group'] = se['region_group'].replace({'upstream_flank': 'upstream_flank', 'downstream_flank': 'downstream_flank', 'exon_body': 'exon_body'})
    se = se.drop_duplicates(['comparison', 'rbp_symbol', 'event_key', 'region_group'])
    rows = []
    for keys, grp in se.groupby(['comparison', 'rbp_symbol'], dropna=False):
        comparison, rbp_symbol = keys
        total = len(grp)
        for region in ['upstream_flank', 'exon_body', 'downstream_flank']:
            count = int(grp['region_group'].eq(region).sum())
            rows.append({'comparison': comparison, 'rbp_symbol': rbp_symbol, 'region_group': region, 'n_region_links': count, 'n_total_region_links': total, 'region_fraction': float(count / total) if total else np.nan})
    dist = pd.DataFrame(rows)
    if dist.empty:
        return (dist, pd.DataFrame())
    wide = dist.pivot_table(index=['comparison', 'rbp_symbol'], columns='region_group', values='region_fraction', fill_value=0).reset_index()
    for col in ['upstream_flank', 'exon_body', 'downstream_flank']:
        if col not in wide.columns:
            wide[col] = 0.0
    count_wide = dist.pivot_table(index=['comparison', 'rbp_symbol'], columns='region_group', values='n_region_links', fill_value=0, aggfunc='sum').reset_index()
    count_wide['n_total_region_links'] = count_wide[[c for c in count_wide.columns if c not in {'comparison', 'rbp_symbol'}]].sum(axis=1)
    summary = wide.merge(count_wide[['comparison', 'rbp_symbol', 'n_total_region_links']], on=['comparison', 'rbp_symbol'], how='left')
    summary['flank_fraction'] = summary['upstream_flank'] + summary['downstream_flank']
    summary['dominant_region'] = summary[['upstream_flank', 'exon_body', 'downstream_flank']].idxmax(axis=1)
    summary['dominant_region_fraction'] = summary[['upstream_flank', 'exon_body', 'downstream_flank']].max(axis=1)
    summary = summary.sort_values(['n_total_region_links', 'dominant_region_fraction'], ascending=[False, False])
    return (dist, summary)

def deep_build_candidate_priority(candidate_summary: pd.DataFrame, phase_modules: pd.DataFrame, apa_utr_bias: pd.DataFrame, se_region_bias: pd.DataFrame, target_contrast: pd.DataFrame) -> pd.DataFrame:
    if phase_modules.empty:
        return pd.DataFrame()
    out = phase_modules.copy()
    if not candidate_summary.empty:
        out = out.merge(candidate_summary, on=['event_kind', 'rbp_symbol'], how='left', suffixes=('', '_global'))
    if not target_contrast.empty:
        contrast = target_contrast.groupby(['comparison', 'event_kind', 'rbp_symbol'], dropna=False).agg(best_contrast_fdr=('fdr', 'min'), max_rate_difference=('rate_difference', 'max'), max_odds_ratio=('odds_ratio', 'max'), max_phase_resultant_delta=('phase_resultant_delta', 'max')).reset_index()
        out = out.merge(contrast, on=['comparison', 'event_kind', 'rbp_symbol'], how='left')
    if not apa_utr_bias.empty:
        apa_bias = apa_utr_bias.copy()
        apa_bias['event_kind'] = 'APA'
        out = out.merge(apa_bias[['comparison', 'event_kind', 'rbp_symbol', 'n_bound_apa_events', 'median_bound_utr_quantile', 'delta_median_utr_quantile', 'delta_long_utr_fraction', 'empirical_p_value', 'empirical_fdr', 'utr_bias_direction', 'top_long_utr_examples']], on=['comparison', 'event_kind', 'rbp_symbol'], how='left')
    if not se_region_bias.empty:
        se_bias = se_region_bias.copy()
        se_bias['event_kind'] = 'SE'
        out = out.merge(se_bias[['comparison', 'event_kind', 'rbp_symbol', 'flank_fraction', 'dominant_region', 'dominant_region_fraction', 'n_total_region_links']], on=['comparison', 'event_kind', 'rbp_symbol'], how='left')
    out['enrichment_score'] = out.get('best_enrichment_fdr', np.nan).map(deep__minus_log10)
    out['contrast_score'] = out.get('best_contrast_fdr', np.nan).map(deep__minus_log10) if 'best_contrast_fdr' in out else 0.0
    out['lag_coherence_score'] = out.get('best_lag_coherence_fdr', np.nan).map(deep__minus_log10)
    out['phase_module_score'] = deep__safe_numeric(out['rbp_to_event_resultant_length']).fillna(0) + deep__safe_numeric(out['event_to_mrna_resultant_length']).fillna(0)
    out['replication_score'] = np.log10(deep__safe_numeric(out.get('n_validation_replicated_events', 0)).fillna(0) + 1.0)
    out['triplet_scale_score'] = np.log10(deep__safe_numeric(out['n_events']).fillna(0) + 1.0)
    out['utr_or_region_score'] = 0.0
    if 'delta_median_utr_quantile' in out:
        out['utr_or_region_score'] = out['utr_or_region_score'] + deep__safe_numeric(out['delta_median_utr_quantile']).abs().fillna(0)
    if 'dominant_region_fraction' in out:
        out['utr_or_region_score'] = out['utr_or_region_score'] + deep__safe_numeric(out['dominant_region_fraction']).fillna(0)
    out['deep_priority_score'] = out['enrichment_score'].clip(0, 10) + out['contrast_score'].clip(0, 10) + out['lag_coherence_score'].clip(0, 8) + 2.0 * out['phase_module_score'].clip(0, 2) + out['replication_score'].clip(0, 3) + out['triplet_scale_score'].clip(0, 4) + 2.0 * out['utr_or_region_score'].clip(0, 2)
    out['mechanistic_axis'] = ''
    if 'utr_bias_direction' in out:
        out.loc[out['event_kind'].eq('APA'), 'mechanistic_axis'] = out.loc[out['event_kind'].eq('APA'), 'utr_bias_direction'].fillna('no_utr_rank_bias')
    if 'dominant_region' in out:
        out.loc[out['event_kind'].eq('SE'), 'mechanistic_axis'] = 'SE_' + out.loc[out['event_kind'].eq('SE'), 'dominant_region'].fillna('no_region_bias')
    out['suggested_panel_use'] = 'backup'
    out.loc[out.get('evidence_tier', '').astype(str).eq('main_candidate') | out['deep_priority_score'].ge(out['deep_priority_score'].quantile(0.85)), 'suggested_panel_use'] = 'main_or_supplement'
    out.loc[out['event_kind'].eq('SE') & out['deep_priority_score'].ge(out['deep_priority_score'].quantile(0.75)), 'suggested_panel_use'] = 'splicing_supplement'
    return out.sort_values(['deep_priority_score', 'n_events'], ascending=[False, False])

def deep__load_font(size: int):
    from PIL import ImageFont
    for path in ['/System/Library/Fonts/Supplemental/Arial.ttf', '/System/Library/Fonts/Supplemental/Helvetica.ttf', '/Library/Fonts/Arial.ttf']:
        if Path(path).exists():
            return ImageFont.truetype(path, size=size)
    return ImageFont.load_default()

def deep__save_png_pdf(image, figures_dir: Path, name: str) -> list[Path]:
    figures_dir.mkdir(parents=True, exist_ok=True)
    png_path = figures_dir / f'{name}.png'
    pdf_path = figures_dir / f'{name}.pdf'
    image.save(png_path)
    try:
        from reportlab.lib.utils import ImageReader
        from reportlab.pdfgen import canvas
        width_px, height_px = image.size
        width_pt = width_px * 0.72
        height_pt = height_px * 0.72
        c = canvas.Canvas(str(pdf_path), pagesize=(width_pt, height_pt))
        c.drawImage(ImageReader(str(png_path)), 0, 0, width=width_pt, height=height_pt)
        c.showPage()
        c.save()
    except ModuleNotFoundError:
        image.save(pdf_path, 'PDF', resolution=300.0)
    return [pdf_path, png_path]

def deep__hex_to_rgb(color: str) -> tuple[int, int, int]:
    color = color.lstrip('#')
    return tuple((int(color[i:i + 2], 16) for i in (0, 2, 4)))

def deep__blend(c1: str, c2: str, t: float) -> tuple[int, int, int]:
    a = np.array(deep__hex_to_rgb(c1), dtype=float)
    b = np.array(deep__hex_to_rgb(c2), dtype=float)
    return tuple(np.clip(a * (1 - t) + b * t, 0, 255).astype(int))

def deep__draw_text(draw, xy, text: str, font, fill=None, anchor=None):
    draw.text(xy, str(text), font=font, fill=fill or deep__hex_to_rgb(deep_PALETTE['dark']), anchor=anchor)

def deep_plot_apa_utr_bias(apa_utr_bias: pd.DataFrame, figures_dir: Path) -> list[Path]:
    if apa_utr_bias.empty:
        return []
    from PIL import Image, ImageDraw
    plot = apa_utr_bias.copy()
    plot['abs_delta'] = deep__safe_numeric(plot['delta_median_utr_quantile']).abs()
    plot = plot.sort_values(['empirical_fdr', 'abs_delta', 'n_bound_apa_events'], ascending=[True, False, False]).head(18)
    row_h = 34
    width = 1200
    height = 150 + row_h * len(plot)
    img = Image.new('RGB', (width, height), 'white')
    draw = ImageDraw.Draw(img)
    font = deep__load_font(18)
    small = deep__load_font(14)
    bold = deep__load_font(22)
    left = 310
    right = width - 85
    top = 95
    axis_y = top + row_h * len(plot) + 15
    deep__draw_text(draw, (30, 28), "APA events bound by rhythmic RBPs are stratified by annotated 3'UTR isoform rank", bold)
    deep__draw_text(draw, (30, 58), "Points show median within-gene 3'UTR length quantile: gray=all rhythmic APA, color=RBP-bound APA", small, deep_PALETTE['gray'])
    draw.line((left, axis_y, right, axis_y), fill=deep__hex_to_rgb(deep_PALETTE['dark']), width=2)
    for tick in [0, 0.25, 0.5, 0.75, 1.0]:
        x = int(left + tick * (right - left))
        draw.line((x, axis_y - 6, x, axis_y + 6), fill=deep__hex_to_rgb(deep_PALETTE['dark']), width=1)
        deep__draw_text(draw, (x, axis_y + 12), f'{tick:.2g}', small, anchor='ma')
    deep__draw_text(draw, ((left + right) // 2, axis_y + 44), "Within-gene 3'UTR length quantile", font, anchor='ma')
    for i, row in enumerate(plot.itertuples(index=False)):
        y = top + i * row_h
        label = f"{row.comparison.replace('_vs_', ' vs ')}  {row.rbp_symbol}  n={row.n_bound_apa_events}"
        deep__draw_text(draw, (30, y - 7), label, small)
        bg = float(row.median_background_utr_quantile)
        bd = float(row.median_bound_utr_quantile)
        x_bg = int(left + np.clip(bg, 0, 1) * (right - left))
        x_bd = int(left + np.clip(bd, 0, 1) * (right - left))
        color = deep_PALETTE['orange'] if bd >= bg else deep_PALETTE['blue']
        draw.line((x_bg, y, x_bd, y), fill=deep__hex_to_rgb(deep_PALETTE['light_gray']), width=5)
        draw.ellipse((x_bg - 6, y - 6, x_bg + 6, y + 6), outline=deep__hex_to_rgb(deep_PALETTE['gray']), width=2)
        draw.ellipse((x_bd - 8, y - 8, x_bd + 8, y + 8), fill=deep__hex_to_rgb(color))
        fdr = row.empirical_fdr
        fdr_label = f'FDR={fdr:.3g}' if pd.notna(fdr) else 'FDR=NA'
        deep__draw_text(draw, (right + 15, y - 7), fdr_label, small, deep_PALETTE['gray'])
    return deep__save_png_pdf(img, figures_dir, 'rbp_apa_utr_rank_bias')

def deep_plot_phase_modules(phase_modules: pd.DataFrame, candidate_priority: pd.DataFrame, figures_dir: Path) -> list[Path]:
    if phase_modules.empty:
        return []
    from PIL import Image, ImageDraw
    if not candidate_priority.empty:
        keys = candidate_priority.head(24)[['comparison', 'event_kind', 'rbp_symbol']]
        plot = phase_modules.merge(keys, on=['comparison', 'event_kind', 'rbp_symbol'], how='inner')
    else:
        plot = phase_modules.sort_values(['n_events', 'rbp_to_event_resultant_length'], ascending=[False, False]).head(24)
    plot = plot.drop_duplicates(['comparison', 'event_kind', 'rbp_symbol']).head(24)
    row_h = 34
    width = 1250
    height = 155 + row_h * len(plot)
    img = Image.new('RGB', (width, height), 'white')
    draw = ImageDraw.Draw(img)
    font = deep__load_font(18)
    small = deep__load_font(14)
    bold = deep__load_font(22)
    left = 370
    right = width - 125
    top = 98
    axis_y = top + row_h * len(plot) + 16
    deep__draw_text(draw, (30, 28), 'RBP-event-mRNA phase modules', bold)
    deep__draw_text(draw, (30, 58), 'Positive lag means the downstream layer peaks after the upstream layer', small, deep_PALETTE['gray'])
    draw.line((left, axis_y, right, axis_y), fill=deep__hex_to_rgb(deep_PALETTE['dark']), width=2)
    for tick in [-12, -6, 0, 6, 12]:
        x = int(left + (tick + 12) / 24 * (right - left))
        draw.line((x, top - 18, x, axis_y + 6), fill=deep__hex_to_rgb(deep_PALETTE['light_gray'] if tick else deep_PALETTE['gray']), width=1)
        deep__draw_text(draw, (x, axis_y + 12), str(tick), small, anchor='ma')
    deep__draw_text(draw, ((left + right) // 2, axis_y + 44), 'Mean circular lag (hours)', font, anchor='ma')
    legend_x = right - 265
    draw.ellipse((legend_x, 28, legend_x + 14, 42), fill=deep__hex_to_rgb(deep_PALETTE['orange']))
    deep__draw_text(draw, (legend_x + 22, 25), 'RBP -> event', small)
    draw.ellipse((legend_x + 135, 28, legend_x + 149, 42), fill=deep__hex_to_rgb(deep_PALETTE['green']))
    deep__draw_text(draw, (legend_x + 157, 25), 'event -> mRNA', small)
    for i, row in enumerate(plot.itertuples(index=False)):
        y = top + i * row_h
        label = f"{row.event_kind}  {row.rbp_symbol}  {row.comparison.replace('_vs_', ' vs ')}  events={row.n_events}"
        deep__draw_text(draw, (30, y - 7), label, small)
        lag1 = row.rbp_to_event_mean_lag_hours
        lag2 = row.event_to_mrna_mean_lag_hours
        x1 = int(left + (np.clip(float(lag1), -12, 12) + 12) / 24 * (right - left)) if pd.notna(lag1) else None
        x2 = int(left + (np.clip(float(lag2), -12, 12) + 12) / 24 * (right - left)) if pd.notna(lag2) else None
        if x1 is not None and x2 is not None:
            draw.line((x1, y, x2, y), fill=deep__hex_to_rgb(deep_PALETTE['light_gray']), width=3)
        if x1 is not None:
            draw.ellipse((x1 - 8, y - 8, x1 + 8, y + 8), fill=deep__hex_to_rgb(deep_PALETTE['orange']))
        if x2 is not None:
            draw.rectangle((x2 - 7, y - 7, x2 + 7, y + 7), fill=deep__hex_to_rgb(deep_PALETTE['green']))
    return deep__save_png_pdf(img, figures_dir, 'rbp_phase_module_cascade')

def deep_plot_se_region_distribution(se_region_distribution: pd.DataFrame, se_region_bias: pd.DataFrame, figures_dir: Path) -> list[Path]:
    if se_region_distribution.empty or se_region_bias.empty:
        return []
    from PIL import Image, ImageDraw
    top_keys = se_region_bias.head(16)[['comparison', 'rbp_symbol']]
    plot = se_region_distribution.merge(top_keys, on=['comparison', 'rbp_symbol'], how='inner')
    row_h = 34
    width = 1100
    height = 140 + row_h * len(top_keys)
    img = Image.new('RGB', (width, height), 'white')
    draw = ImageDraw.Draw(img)
    small = deep__load_font(14)
    font = deep__load_font(18)
    bold = deep__load_font(22)
    left = 310
    right = width - 80
    top = 88
    colors = {'upstream_flank': deep_PALETTE['blue'], 'exon_body': deep_PALETTE['gold'], 'downstream_flank': deep_PALETTE['purple']}
    deep__draw_text(draw, (30, 28), 'CLIP positional support around rhythmic skipped exons', bold)
    legend_x = left
    for region in ['upstream_flank', 'exon_body', 'downstream_flank']:
        draw.rectangle((legend_x, 56, legend_x + 18, 70), fill=deep__hex_to_rgb(colors[region]))
        deep__draw_text(draw, (legend_x + 25, 53), region.replace('_', ' '), small)
        legend_x += 185
    for i, key in enumerate(top_keys.itertuples(index=False)):
        y = top + i * row_h
        sub = plot[plot['comparison'].eq(key.comparison) & plot['rbp_symbol'].eq(key.rbp_symbol)].set_index('region_group')
        label = f"{key.comparison.replace('_vs_', ' vs ')}  {key.rbp_symbol}"
        deep__draw_text(draw, (30, y - 7), label, small)
        x0 = left
        for region in ['upstream_flank', 'exon_body', 'downstream_flank']:
            frac = float(sub.loc[region, 'region_fraction']) if region in sub.index else 0.0
            x1 = x0 + int(frac * (right - left))
            draw.rectangle((x0, y - 10, x1, y + 10), fill=deep__hex_to_rgb(colors[region]))
            x0 = x1
        draw.rectangle((left, y - 10, right, y + 10), outline=deep__hex_to_rgb(deep_PALETTE['dark']), width=1)
    deep__draw_text(draw, ((left + right) // 2, height - 35), 'Fraction of rhythmic SE CLIP links', font, anchor='ma')
    return deep__save_png_pdf(img, figures_dir, 'rbp_se_region_distribution')

def deep_plot_candidate_matrix(candidate_priority: pd.DataFrame, figures_dir: Path) -> list[Path]:
    if candidate_priority.empty:
        return []
    from PIL import Image, ImageDraw
    features = ['enrichment_score', 'contrast_score', 'lag_coherence_score', 'phase_module_score', 'replication_score', 'triplet_scale_score', 'utr_or_region_score']
    plot = candidate_priority.head(22).copy()
    width = 1180
    cell_w = 105
    row_h = 31
    left = 355
    top = 105
    height = top + row_h * len(plot) + 95
    img = Image.new('RGB', (width, height), 'white')
    draw = ImageDraw.Draw(img)
    small = deep__load_font(13)
    font = deep__load_font(16)
    bold = deep__load_font(22)
    deep__draw_text(draw, (30, 28), 'Deep RBP-processing candidate evidence matrix', bold)
    deep__draw_text(draw, (30, 58), 'Rows are ranked by integrated enrichment, phase coherence, replication, and event-annotation evidence', small, deep_PALETTE['gray'])
    labels = {'enrichment_score': 'enrich', 'contrast_score': 'target bg', 'lag_coherence_score': 'lag coh', 'phase_module_score': 'phase R', 'replication_score': 'replicate', 'triplet_scale_score': 'events', 'utr_or_region_score': 'UTR/SE pos'}
    maxima = {}
    for feat in features:
        values = deep__safe_numeric(plot[feat]).fillna(0)
        maxima[feat] = max(float(values.quantile(0.95)), 1e-09)
        x = left + features.index(feat) * cell_w + cell_w // 2
        deep__draw_text(draw, (x, top - 28), labels[feat], small, anchor='ma')
    for i, row in enumerate(plot.itertuples(index=False)):
        y = top + i * row_h
        label = f"{row.event_kind} {row.rbp_symbol} {row.comparison.replace('_vs_', ' vs ')}"
        deep__draw_text(draw, (30, y + 3), label, small)
        for j, feat in enumerate(features):
            value = getattr(row, feat)
            value = 0.0 if pd.isna(value) else float(value)
            t = float(np.clip(value / maxima[feat], 0, 1))
            color = deep__blend('#F3F3F3', deep_PALETTE['blue'], t)
            x0 = left + j * cell_w
            draw.rectangle((x0, y - 10, x0 + cell_w - 5, y + 15), fill=color, outline=(255, 255, 255))
        score_x = left + len(features) * cell_w + 12
        deep__draw_text(draw, (score_x, y + 3), f'{row.deep_priority_score:.1f}', small, deep_PALETTE['dark'])
    deep__draw_text(draw, (left + len(features) * cell_w + 12, top - 28), 'score', small)
    return deep__save_png_pdf(img, figures_dir, 'rbp_deep_candidate_evidence_matrix')

def deep_make_figures(apa_utr_bias: pd.DataFrame, phase_modules: pd.DataFrame, se_region_distribution: pd.DataFrame, se_region_bias: pd.DataFrame, candidate_priority: pd.DataFrame, figures_dir: Path) -> list[Path]:
    saved: list[Path] = []
    try:
        saved.extend(deep_plot_apa_utr_bias(apa_utr_bias, figures_dir))
        saved.extend(deep_plot_phase_modules(phase_modules, candidate_priority, figures_dir))
        saved.extend(deep_plot_se_region_distribution(se_region_distribution, se_region_bias, figures_dir))
        saved.extend(deep_plot_candidate_matrix(candidate_priority, figures_dir))
    except Exception as exc:
        print(f'Figure generation skipped: {type(exc).__name__}: {exc}')
    return saved

def deep_run_deep_dive(project_root: str | os.PathLike[str] | None=None, analysis: str=deep_ANALYSIS, n_permutations: int=2000, make_plots: bool=True) -> dict[str, pd.DataFrame | list[Path] | deep_DeepDiveOutputs]:
    project_root = deep_resolve_project_root(project_root)
    output_dir = project_root / 'data' / analysis
    figures_dir = project_root / 'data' / 'figures' / analysis
    output_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)
    apa_phase = deep__read_table(output_dir / 'apa_expression_phase_lag_inputs.tsv')
    apa_lags = deep__read_table(output_dir / 'rbp_apa_phase_lags.tsv')
    se_lags = deep__read_table(output_dir / 'rbp_se_phase_lags.tsv')
    triplets = deep__read_table(output_dir / 'rbp_event_gene_triplets.tsv')
    clip_links = deep__read_table(output_dir / 'clip_supported_rbp_target_links_consensus.tsv')
    candidate_summary = deep__read_table(output_dir / 'rbp_main_figure_candidate_summary.tsv')
    target_contrast = deep__read_table(output_dir / 'rbp_target_background_contrast.tsv')
    utr = deep_load_utr_annotation(project_root)
    annotated_apa = deep_annotate_apa_utr_rank(apa_phase, utr)
    apa_utr_bias = deep_compute_apa_utr_bias(annotated_apa, apa_lags, n_permutations=n_permutations)
    phase_modules = deep_summarize_phase_modules(triplets)
    se_region_distribution, se_region_bias = deep_summarize_se_region_bias(se_lags, clip_links)
    candidate_priority = deep_build_candidate_priority(candidate_summary=candidate_summary, phase_modules=phase_modules, apa_utr_bias=apa_utr_bias, se_region_bias=se_region_bias, target_contrast=target_contrast)
    paths = deep_DeepDiveOutputs(annotated_apa=deep__write_tsv(annotated_apa, output_dir / 'rbp_apa_utr_rank_annotation.tsv'), apa_utr_bias=deep__write_tsv(apa_utr_bias, output_dir / 'rbp_apa_utr_rank_bias.tsv'), phase_modules=deep__write_tsv(phase_modules, output_dir / 'rbp_event_phase_module_summary.tsv'), se_region_distribution=deep__write_tsv(se_region_distribution, output_dir / 'rbp_se_region_distribution.tsv'), se_region_bias=deep__write_tsv(se_region_bias, output_dir / 'rbp_se_region_bias.tsv'), candidate_priority=deep__write_tsv(candidate_priority, output_dir / 'rbp_deep_candidate_priority.tsv'), manifest=output_dir / 'rbp_deep_dive_manifest.tsv')
    saved_figures = []
    if make_plots:
        saved_figures = deep_make_figures(apa_utr_bias=apa_utr_bias, phase_modules=phase_modules, se_region_distribution=se_region_distribution, se_region_bias=se_region_bias, candidate_priority=candidate_priority, figures_dir=figures_dir)
    manifest = pd.DataFrame([{'metric': 'annotated_apa_rows', 'value': len(annotated_apa), 'detail': deep__relative_path(paths.annotated_apa, project_root)}, {'metric': 'annotated_apa_utr_available_fraction', 'value': float(annotated_apa.get('utr_annotation_available', pd.Series(dtype=bool)).mean()) if len(annotated_apa) else np.nan, 'detail': "Fraction of APA phase rows with matching transcript-level 3'UTR annotation."}, {'metric': 'apa_utr_bias_rows', 'value': len(apa_utr_bias), 'detail': deep__relative_path(paths.apa_utr_bias, project_root)}, {'metric': 'phase_module_rows', 'value': len(phase_modules), 'detail': deep__relative_path(paths.phase_modules, project_root)}, {'metric': 'se_region_bias_rows', 'value': len(se_region_bias), 'detail': deep__relative_path(paths.se_region_bias, project_root)}, {'metric': 'candidate_priority_rows', 'value': len(candidate_priority), 'detail': deep__relative_path(paths.candidate_priority, project_root)}, {'metric': 'saved_deep_dive_figures', 'value': len(saved_figures), 'detail': ';'.join((deep__relative_path(Path(path), project_root) for path in saved_figures))}, {'metric': 'apa_utr_rank_caveat', 'value': 'annotation_proxy', 'detail': "3'UTR directionality is inferred from annotated transcript-level 3'UTR length rank, not direct PAS coordinates."}])
    deep__write_tsv(manifest, paths.manifest)
    return {'paths': paths, 'annotated_apa': annotated_apa, 'apa_utr_bias': apa_utr_bias, 'phase_modules': phase_modules, 'se_region_distribution': se_region_distribution, 'se_region_bias': se_region_bias, 'candidate_priority': candidate_priority, 'manifest': manifest, 'saved_figures': saved_figures}


In [ ]:
# Cell 8: Activate the visible deep-dive namespace.
for _name, _value in list(globals().items()):
    if _name.startswith('deep_'):
        setattr(deep, _name.removeprefix('deep_'), _value)


In [ ]:
# Cell 9: Embedded and namespace-isolated from scripts/rbp_rna_processing_site_modeling.py
"""Regulatory-site availability and sample-level usage models for RBP modules.

This module extends the RBP-centered APA/SE analysis with two mechanistic
checks:

1. APA regulatory-site availability proxies from annotated 3'UTR length rank,
   TargetScan miRNA site burden, and broad 3'UTR CLIP burden.
2. Exploratory sample-level PDUI/PSI models asking whether RBP expression adds
   explanatory power beyond host-gene expression in a within-event framework.

The sample-level models are prioritization screens. Observations share samples
and events, so p-values should not be read as fully independent tests.
"""
from __future__ import annotations
from dataclasses import dataclass
import math
import os
from pathlib import Path
import re
from typing import Iterable
import numpy as np
import pandas as pd
from scipy import stats
site_ANALYSIS = 'rbp_rna_processing_links'
site_FOCUS_RBPS = {'DDX3X', 'MBNL2', 'RBFOX2', 'PRPF8', 'NUDT21', 'CPSF6', 'FIP1L1'}
site_APA_USAGE_FILES = {'TCGA_Tumor': 'data/polyAPA/TCGA/filtered_apa/TCGA_variable_apa_events.csv', 'CPTAC_Tumor': 'data/polyAPA/CPTAC/filtered_apa/CPTAC_variable_apa_events.csv', 'GTEx_Normal': 'data/polyAPA/GTEx/filtered_apa/GTEx_variable_apa_events.csv'}
site_SE_USAGE_FILES = {'TCGA_Tumor': 'data/AS/TCGA/filtered_events/TCGA_SE_variable_AS.csv', 'CPTAC_Tumor': 'data/AS/CPTAC/filtered_events/CPTAC_SE_variable_AS.csv', 'GTEx_Normal': 'data/AS/GTEx/post/SE.MATS.JC.txt'}
site_SE_SAMPLE_FILES = {'TCGA_Tumor': 'data/AS/TCGA/all_samples.txt', 'CPTAC_Tumor': 'data/AS/CPTAC/all_samples.txt', 'GTEx_Normal': 'data/AS/GTEx/all_samples.txt'}
site_EXPRESSION_FILES = {'TCGA_Tumor': 'data/Merged/Group_TPM_TCGA-PAAD_Tumor.csv', 'CPTAC_Tumor': 'data/Merged/Group_TPM_CPTAC-3_Tumor.csv', 'GTEx_Normal': 'data/Merged/Group_TPM_GTEx_Normal.csv'}
site_PALETTE = {'blue': '#0072B2', 'orange': '#D55E00', 'green': '#009E73', 'purple': '#CC79A7', 'gold': '#E69F00', 'sky': '#56B4E9', 'gray': '#7A7A7A', 'light_gray': '#E6E6E6', 'dark': '#222222', 'red': '#B2182B', 'navy': '#2166AC'}

@dataclass(frozen=True)
class site_SiteAvailabilityOutputs:
    site_availability: Path
    site_bias: Path
    site_manifest: Path

@dataclass(frozen=True)
class site_SampleModelOutputs:
    model_summary: Path
    event_effects: Path
    model_manifest: Path

def site_resolve_project_root(project_root: str | os.PathLike[str] | None=None) -> Path:
    if project_root is not None:
        return Path(project_root).expanduser().resolve()
    workspace_root = Path(os.environ.get('WORKSPACE_ROOT', '/home/wdeng3/workspace/Codespace'))
    candidates = [workspace_root / 'PDAC_Rhythmicity', Path.cwd() / 'PDAC_Rhythmicity', Path.cwd(), Path(__file__).resolve().parents[1]]
    for candidate in candidates:
        if candidate.name == 'PDAC_Rhythmicity' and (candidate / 'data').exists():
            return candidate.resolve()
    return Path(__file__).resolve().parents[1]

def site__read_table(path: Path, **kwargs) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    sep = '\t' if path.suffix.lower() in {'.tsv', '.txt', '.bed'} else ','
    return pd.read_csv(path, sep=sep, low_memory=False, **kwargs)

def site__write_tsv(df: pd.DataFrame, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, sep='\t', index=False)
    return path

def site__relative_path(path: Path, root: Path) -> str:
    try:
        return str(path.relative_to(root))
    except ValueError:
        return str(path)

def site__safe_numeric(series: pd.Series | Iterable[object]) -> pd.Series:
    return pd.to_numeric(pd.Series(series), errors='coerce')

def site__bh_fdr(p_values: Iterable[object]) -> np.ndarray:
    values = pd.to_numeric(pd.Series(list(p_values)), errors='coerce').to_numpy(dtype=float)
    out = np.full(values.shape, np.nan, dtype=float)
    mask = np.isfinite(values)
    if not mask.any():
        return out
    finite = values[mask]
    order = np.argsort(finite)
    ranked = finite[order]
    n = len(ranked)
    adjusted = ranked * n / np.arange(1, n + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    q = np.empty_like(finite)
    q[order] = np.clip(adjusted, 0.0, 1.0)
    out[mask] = q
    return out

def site__zscore(values: pd.Series | np.ndarray) -> pd.Series:
    series = site__safe_numeric(values)
    mean = series.mean(skipna=True)
    sd = series.std(skipna=True)
    if pd.isna(sd) or sd == 0:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - mean) / sd

def site__event_key(value: object) -> str:
    return base.standardize_event_id(value)

def site__transcript_core(value: object) -> str:
    text = '' if pd.isna(value) else str(value).strip()
    transcript = text.split('|')[0] if text else ''
    return re.sub('\\.\\d+$', '', transcript)

def site__transcript_id(value: object) -> str:
    text = '' if pd.isna(value) else str(value).strip()
    return text.split('|')[0] if text else ''

def site__standardize_sample_id(value: object) -> str:
    text = '' if pd.isna(value) else str(value).strip().strip('"')
    if not text:
        return ''
    text = Path(text).name
    for suffix in ['.Aligned.sortedByCoord.out.patched.md.bam', '.Aligned.sortedByCoord.out.bam', '.patched.md.bam', '.bam']:
        if text.endswith(suffix):
            text = text[:-len(suffix)]
    text = text.replace(', ', '_').replace(',', '_').replace(' ', '')
    text = re.sub('_+', '_', text)
    return text.strip('_')

def site__load_font(size: int):
    from PIL import ImageFont
    for path in ['/System/Library/Fonts/Supplemental/Arial.ttf', '/System/Library/Fonts/Supplemental/Helvetica.ttf', '/Library/Fonts/Arial.ttf']:
        if Path(path).exists():
            return ImageFont.truetype(path, size=size)
    return ImageFont.load_default()

def site__hex_to_rgb(color: str) -> tuple[int, int, int]:
    color = color.lstrip('#')
    return tuple((int(color[i:i + 2], 16) for i in (0, 2, 4)))

def site__blend(c1: str, c2: str, t: float) -> tuple[int, int, int]:
    a = np.array(site__hex_to_rgb(c1), dtype=float)
    b = np.array(site__hex_to_rgb(c2), dtype=float)
    return tuple(np.clip(a * (1 - t) + b * t, 0, 255).astype(int))

def site__draw_text(draw, xy, text: str, font, fill=None, anchor=None):
    draw.text(xy, str(text), font=font, fill=fill or site__hex_to_rgb(site_PALETTE['dark']), anchor=anchor)

def site__save_png_pdf(image, figures_dir: Path, name: str) -> list[Path]:
    figures_dir.mkdir(parents=True, exist_ok=True)
    png_path = figures_dir / f'{name}.png'
    pdf_path = figures_dir / f'{name}.pdf'
    image.save(png_path)
    try:
        from reportlab.lib.utils import ImageReader
        from reportlab.pdfgen import canvas
        width_px, height_px = image.size
        width_pt = width_px * 0.72
        height_pt = height_px * 0.72
        c = canvas.Canvas(str(pdf_path), pagesize=(width_pt, height_pt))
        c.drawImage(ImageReader(str(png_path)), 0, 0, width=width_pt, height=height_pt)
        c.showPage()
        c.save()
    except ModuleNotFoundError:
        image.save(pdf_path, 'PDF', resolution=300.0)
    return [pdf_path, png_path]

def site_load_mirna_site_burden(project_root: Path) -> pd.DataFrame:
    path = project_root / 'data' / 'apa_mirna_rhythmic_connection' / 'apa_mirna_gene_triads.tsv'
    columns = ['apa_event_key', 'transcript_id', 'transcript_core', 'host_gene_symbol', 'n_mirna_ids', 'n_seed_families', 'n_targetscan_transcripts', 'total_conserved_sites', 'total_nonconserved_sites', 'total_mirna_sites']
    if not path.exists():
        return pd.DataFrame(columns=columns)
    header = pd.read_csv(path, sep='\t', nrows=0).columns
    wanted = ['mirna_id_target_db', 'mirna_id', 'seed_family', 'n_targetscan_transcripts', 'targetscan_total_conserved_sites', 'targetscan_total_nonconserved_sites', 'transcript_id', 'apa_event_id', 'host_gene_symbol', 'target_gene_symbol', 'apa_gene_symbol']
    usecols = [col for col in wanted if col in header]
    df = pd.read_csv(path, sep='\t', usecols=usecols, low_memory=False)
    if df.empty or 'apa_event_id' not in df.columns:
        return pd.DataFrame(columns=columns)
    df['apa_event_key'] = df['apa_event_id'].map(site__event_key)
    df['transcript_id'] = df.get('transcript_id', df['apa_event_id']).fillna(df['apa_event_id']).map(site__transcript_id)
    df['transcript_core'] = df['transcript_id'].map(site__transcript_core)
    if 'host_gene_symbol' not in df.columns:
        df['host_gene_symbol'] = ''
    fallback_symbol = df.get('apa_gene_symbol', df.get('target_gene_symbol', pd.Series('', index=df.index)))
    df['host_gene_symbol'] = df['host_gene_symbol'].fillna(fallback_symbol).map(base.standardize_symbol)
    df['mirna_key'] = df.get('mirna_id_target_db', df.get('mirna_id', pd.Series('', index=df.index))).fillna('').astype(str)
    df['seed_family'] = df.get('seed_family', pd.Series('', index=df.index)).fillna('').astype(str)
    for col in ['n_targetscan_transcripts', 'targetscan_total_conserved_sites', 'targetscan_total_nonconserved_sites']:
        if col not in df.columns:
            df[col] = 0
        df[col] = site__safe_numeric(df[col]).fillna(0)
    df = df[df['apa_event_key'].ne('')].copy()
    if df.empty:
        return pd.DataFrame(columns=columns)
    dedup = df.drop_duplicates(['apa_event_key', 'mirna_key', 'seed_family']).copy()
    out = dedup.groupby(['apa_event_key', 'transcript_id', 'transcript_core', 'host_gene_symbol'], dropna=False).agg(n_mirna_ids=('mirna_key', lambda x: int(pd.Series(x).replace('', np.nan).nunique())), n_seed_families=('seed_family', lambda x: int(pd.Series(x).replace('', np.nan).nunique())), n_targetscan_transcripts=('n_targetscan_transcripts', 'max'), total_conserved_sites=('targetscan_total_conserved_sites', 'sum'), total_nonconserved_sites=('targetscan_total_nonconserved_sites', 'sum')).reset_index()
    out['total_mirna_sites'] = out['total_conserved_sites'] + out['total_nonconserved_sites']
    return out[columns].copy()

def site_load_clip_utr_site_burden(project_root: Path, analysis: str=site_ANALYSIS) -> pd.DataFrame:
    path = project_root / 'data' / analysis / 'clip_supported_rbp_target_links_consensus.tsv'
    columns = ['host_gene_symbol', 'n_3utr_clip_rbps', 'n_3utr_clip_links', 'n_3utr_clip_accessions', 'n_3utr_clip_methods', 'max_3utr_clip_score', 'median_3utr_clip_score']
    if not path.exists():
        return pd.DataFrame(columns=columns)
    header = pd.read_csv(path, sep='\t', nrows=0).columns
    wanted = ['target_gene_symbol', 'region', 'rbp_symbol', 'n_accessions', 'n_methods', 'max_score', 'median_score']
    usecols = [col for col in wanted if col in header]
    df = pd.read_csv(path, sep='\t', usecols=usecols, low_memory=False)
    if df.empty:
        return pd.DataFrame(columns=columns)
    df['region'] = df.get('region', pd.Series('', index=df.index)).astype(str)
    df = df[df['region'].str.contains('3UTR', case=False, na=False)].copy()
    if df.empty:
        return pd.DataFrame(columns=columns)
    df['host_gene_symbol'] = df.get('target_gene_symbol', pd.Series('', index=df.index)).map(base.standardize_symbol)
    df = df[df['host_gene_symbol'].ne('')]
    for col in ['n_accessions', 'n_methods', 'max_score', 'median_score']:
        if col not in df.columns:
            df[col] = 0
        df[col] = site__safe_numeric(df[col]).fillna(0)
    out = df.groupby('host_gene_symbol', dropna=False).agg(n_3utr_clip_rbps=('rbp_symbol', lambda x: int(pd.Series(x).map(base.standardize_symbol).replace('', np.nan).nunique())), n_3utr_clip_links=('host_gene_symbol', 'size'), n_3utr_clip_accessions=('n_accessions', 'sum'), n_3utr_clip_methods=('n_methods', 'sum'), max_3utr_clip_score=('max_score', 'max'), median_3utr_clip_score=('median_score', 'median')).reset_index()
    return out[columns].copy()

def site_build_site_availability(project_root: Path, analysis: str=site_ANALYSIS) -> pd.DataFrame:
    output_dir = project_root / 'data' / analysis
    apa_phase = site__read_table(output_dir / 'apa_expression_phase_lag_inputs.tsv')
    if apa_phase.empty:
        apa_phase = base.load_apa_phase_tables(project_root)
    if apa_phase.empty:
        return pd.DataFrame()
    utr = deep.load_utr_annotation(project_root)
    site = deep.annotate_apa_utr_rank(apa_phase, utr)
    site['apa_event_key'] = site['apa_event_id'].map(site__event_key)
    site['transcript_id'] = site.get('transcript_id', site['apa_event_id']).fillna(site['apa_event_id']).map(site__transcript_id)
    site['transcript_core'] = site['transcript_id'].map(site__transcript_core)
    site['host_gene_symbol'] = site.get('host_gene_symbol', pd.Series('', index=site.index)).map(base.standardize_symbol)
    mirna = site_load_mirna_site_burden(project_root)
    if not mirna.empty:
        merge_cols = ['apa_event_key', 'n_mirna_ids', 'n_seed_families', 'n_targetscan_transcripts', 'total_conserved_sites', 'total_nonconserved_sites', 'total_mirna_sites']
        site = site.merge(mirna[merge_cols].drop_duplicates('apa_event_key'), on='apa_event_key', how='left')
    for col in ['n_mirna_ids', 'n_seed_families', 'n_targetscan_transcripts', 'total_conserved_sites', 'total_nonconserved_sites', 'total_mirna_sites']:
        if col not in site.columns:
            site[col] = 0
        site[col] = site__safe_numeric(site[col]).fillna(0)
    clip_burden = site_load_clip_utr_site_burden(project_root, analysis)
    if not clip_burden.empty:
        site = site.merge(clip_burden, on='host_gene_symbol', how='left')
    for col in ['n_3utr_clip_rbps', 'n_3utr_clip_links', 'n_3utr_clip_accessions', 'n_3utr_clip_methods', 'max_3utr_clip_score', 'median_3utr_clip_score']:
        if col not in site.columns:
            site[col] = 0
        site[col] = site__safe_numeric(site[col]).fillna(0)
    components = pd.DataFrame({'mirna_sites': np.log1p(site['total_mirna_sites']), 'seed_families': np.log1p(site['n_seed_families']), 'clip_rbps': np.log1p(site['n_3utr_clip_rbps']), 'clip_links': np.log1p(site['n_3utr_clip_links']), 'utr_length': np.log1p(site__safe_numeric(site.get('utr_length_bp', 0)).fillna(0))}, index=site.index)
    z_components = components.apply(site__zscore)
    site['regulatory_site_availability_score'] = z_components.mean(axis=1)
    site['has_mirna_site_proxy'] = site['total_mirna_sites'].gt(0)
    site['has_3utr_clip_site_proxy'] = site['n_3utr_clip_rbps'].gt(0)
    site['site_availability_proxy_note'] = "TargetScan transcript/gene miRNA burden plus broad 3'UTR CLIP burden; not direct PAS-gain/loss interval counting."
    return site

def site_compute_site_availability_bias(site: pd.DataFrame, apa_lags: pd.DataFrame) -> pd.DataFrame:
    if site.empty or apa_lags.empty:
        return pd.DataFrame()
    bg = site.copy()
    bg['apa_event_key'] = bg['apa_event_key'].map(site__event_key)
    bg = bg.drop_duplicates(['comparison', 'apa_event_key'])
    lag = apa_lags.copy()
    lag['apa_event_key'] = lag['apa_event_id'].map(site__event_key)
    lag = lag.drop_duplicates(['comparison', 'rbp_symbol', 'apa_event_key'])
    metrics = ['n_seed_families', 'total_mirna_sites', 'n_3utr_clip_rbps', 'n_3utr_clip_links', 'regulatory_site_availability_score', 'utr_length_quantile']
    rows = []
    for (comparison, rbp_symbol), grp in lag.groupby(['comparison', 'rbp_symbol'], dropna=False):
        background = bg[bg['comparison'].eq(comparison)].copy()
        if len(background) < 20:
            continue
        bound_keys = set(grp['apa_event_key'])
        bound = background[background['apa_event_key'].isin(bound_keys)].copy()
        if len(bound) < 8:
            continue
        row = {'comparison': comparison, 'rbp_symbol': rbp_symbol, 'n_bound_apa_events': int(len(bound)), 'n_background_apa_events': int(len(background))}
        for metric in metrics:
            b = site__safe_numeric(bound.get(metric, pd.Series(dtype=float))).dropna()
            a = site__safe_numeric(background.get(metric, pd.Series(dtype=float))).dropna()
            if len(b) and len(a):
                row[f'median_bound_{metric}'] = float(b.median())
                row[f'median_background_{metric}'] = float(a.median())
                row[f'delta_{metric}'] = float(b.median() - a.median())
                try:
                    row[f'p_value_{metric}'] = float(stats.mannwhitneyu(b, a, alternative='two-sided').pvalue)
                except ValueError:
                    row[f'p_value_{metric}'] = np.nan
            else:
                row[f'median_bound_{metric}'] = np.nan
                row[f'median_background_{metric}'] = np.nan
                row[f'delta_{metric}'] = np.nan
                row[f'p_value_{metric}'] = np.nan
        examples = bound.assign(example_score=lambda x: site__safe_numeric(x['total_mirna_sites']).fillna(0) + site__safe_numeric(x['n_3utr_clip_links']).fillna(0) + site__safe_numeric(x['utr_length_quantile']).fillna(0)).sort_values(['example_score', 'fdr'], ascending=[False, True]).get('host_gene_symbol', pd.Series(dtype=str)).dropna().astype(str).head(8).tolist()
        row['top_site_available_examples'] = ','.join(dict.fromkeys(examples))
        rows.append(row)
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out['availability_bias_p_value'] = out['p_value_regulatory_site_availability_score']
    out['availability_bias_fdr'] = site__bh_fdr(out['availability_bias_p_value'])
    out['availability_bias_direction'] = 'mixed_or_flat'
    out.loc[out['delta_regulatory_site_availability_score'].ge(0.15), 'availability_bias_direction'] = 'higher_site_availability'
    out.loc[out['delta_regulatory_site_availability_score'].le(-0.15), 'availability_bias_direction'] = 'lower_site_availability'
    return out.sort_values(['availability_bias_fdr', 'n_bound_apa_events', 'delta_regulatory_site_availability_score'], ascending=[True, False, False])

def site__delta_color(value: float, scale: float) -> tuple[int, int, int]:
    if pd.isna(value):
        return site__hex_to_rgb('#F3F3F3')
    t = float(np.clip(abs(value) / max(scale, 1e-09), 0, 1))
    if value >= 0:
        return site__blend('#F7F7F7', site_PALETTE['red'], t)
    return site__blend('#F7F7F7', site_PALETTE['navy'], t)

def site_plot_site_availability_bias(site_bias: pd.DataFrame, figures_dir: Path) -> list[Path]:
    if site_bias.empty:
        return []
    from PIL import Image, ImageDraw
    metrics = [('delta_n_seed_families', 'seed fam'), ('delta_total_mirna_sites', 'miRNA sites'), ('delta_n_3utr_clip_rbps', "3'UTR RBPs"), ('delta_n_3utr_clip_links', "3'UTR links"), ('delta_regulatory_site_availability_score', 'site score')]
    plot = site_bias.copy()
    plot['rank_score'] = site__safe_numeric(plot['delta_regulatory_site_availability_score']).abs().fillna(0)
    plot = plot.sort_values(['availability_bias_fdr', 'rank_score', 'n_bound_apa_events'], ascending=[True, False, False]).head(22)
    row_h = 31
    cell_w = 108
    left = 335
    top = 108
    width = 1120
    height = top + row_h * len(plot) + 105
    img = Image.new('RGB', (width, height), 'white')
    draw = ImageDraw.Draw(img)
    small = site__load_font(13)
    font = site__load_font(16)
    bold = site__load_font(22)
    site__draw_text(draw, (30, 28), 'APA regulatory-site availability around RBP-bound isoforms', bold)
    site__draw_text(draw, (30, 58), 'Cells show bound minus background medians; red=higher availability in bound APA, blue=lower', small, site_PALETTE['gray'])
    for j, (_, label) in enumerate(metrics):
        x = left + j * cell_w + cell_w // 2
        site__draw_text(draw, (x, top - 28), label, small, anchor='ma')
    scales = {}
    for metric, _ in metrics:
        values = site__safe_numeric(plot[metric]).abs().dropna()
        scales[metric] = max(float(values.quantile(0.9)) if len(values) else 1.0, 1e-06)
    for i, row in enumerate(plot.itertuples(index=False)):
        y = top + i * row_h
        label = f"{row.comparison.replace('_vs_', ' vs ')}  {row.rbp_symbol}  n={row.n_bound_apa_events}"
        site__draw_text(draw, (30, y + 2), label, small)
        for j, (metric, _) in enumerate(metrics):
            value = getattr(row, metric)
            x0 = left + j * cell_w
            draw.rectangle((x0, y - 11, x0 + cell_w - 5, y + 15), fill=site__delta_color(value, scales[metric]), outline=(255, 255, 255))
            if pd.notna(value):
                site__draw_text(draw, (x0 + cell_w // 2, y + 2), f'{float(value):.2g}', small, anchor='ma')
        fdr = row.availability_bias_fdr
        fdr_label = f'FDR={fdr:.2g}' if pd.notna(fdr) else 'FDR=NA'
        site__draw_text(draw, (left + len(metrics) * cell_w + 20, y + 2), fdr_label, small, site_PALETTE['gray'])
    site__draw_text(draw, (left + len(metrics) * cell_w + 20, top - 28), 'site score', small)
    site__draw_text(draw, (left + 230, height - 35), "Availability proxy: TargetScan miRNA burden + broad 3'UTR CLIP burden + annotated 3'UTR rank", font, site_PALETTE['gray'], anchor='ma')
    return site__save_png_pdf(img, figures_dir, 'rbp_apa_regulatory_site_availability')

def site_run_site_availability(project_root: str | os.PathLike[str] | None=None, analysis: str=site_ANALYSIS, make_plots: bool=True) -> dict[str, pd.DataFrame | list[Path]]:
    project_root = site_resolve_project_root(project_root)
    output_dir = project_root / 'data' / analysis
    figures_dir = project_root / 'data' / 'figures' / analysis
    output_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)
    site = site_build_site_availability(project_root, analysis)
    apa_lags = site__read_table(output_dir / 'rbp_apa_phase_lags.tsv')
    site_bias = site_compute_site_availability_bias(site, apa_lags)
    site_path = site__write_tsv(site, output_dir / 'rbp_apa_regulatory_site_availability.tsv')
    site_bias_path = site__write_tsv(site_bias, output_dir / 'rbp_apa_regulatory_site_bias.tsv')
    figures = site_plot_site_availability_bias(site_bias, figures_dir) if make_plots else []
    manifest = pd.DataFrame([{'key': 'site_availability_rows', 'value': len(site), 'detail': site__relative_path(site_path, project_root)}, {'key': 'site_bias_rows', 'value': len(site_bias), 'detail': site__relative_path(site_bias_path, project_root)}, {'key': 'availability_proxy_caveat', 'value': 'proxy', 'detail': 'Uses transcript-level TargetScan/CLIP burden and UTR rank; direct PAS-specific gained/lost motif intervals remain future work.'}, {'key': 'figures', 'value': len(figures), 'detail': ';'.join((site__relative_path(p, project_root) for p in figures))}])
    manifest_path = site__write_tsv(manifest, output_dir / 'rbp_apa_regulatory_site_manifest.tsv')
    return {'site_availability': site, 'site_bias': site_bias, 'site_manifest': manifest, 'figures': figures, 'paths': site_SiteAvailabilityOutputs(site_path, site_bias_path, manifest_path)}

def site_load_expression_matrix(project_root: Path, cohort_key: str) -> pd.DataFrame:
    rel = site_EXPRESSION_FILES.get(cohort_key)
    if rel is None:
        return pd.DataFrame()
    path = project_root / rel
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path, sep='\t', low_memory=False)
    if df.empty or 'Gene' not in df.columns:
        return pd.DataFrame()
    df['gene_symbol'] = df['Gene'].map(base.standardize_symbol)
    df = df[df['gene_symbol'].ne('')].drop(columns=['Gene'])
    sample_cols = [col for col in df.columns if col != 'gene_symbol']
    rename = {col: site__standardize_sample_id(col) for col in sample_cols}
    df = df.rename(columns=rename)
    value_cols = [col for col in df.columns if col != 'gene_symbol' and col]
    values = df[value_cols].apply(pd.to_numeric, errors='coerce')
    values.insert(0, 'gene_symbol', df['gene_symbol'].values)
    values = values.groupby('gene_symbol', dropna=False).mean()
    return np.log2(values + 1.0)

def site_load_apa_usage_matrix(project_root: Path, cohort_key: str, event_ids: set[str] | None=None) -> pd.DataFrame:
    rel = site_APA_USAGE_FILES.get(cohort_key)
    if rel is None:
        return pd.DataFrame()
    path = project_root / rel
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path, low_memory=False)
    if df.empty:
        return pd.DataFrame()
    first = df.columns[0]
    df = df.rename(columns={first: 'event_id'})
    df['event_id'] = df['event_id'].map(site__event_key)
    df = df[df['event_id'].ne('')]
    if event_ids:
        df = df[df['event_id'].isin(event_ids)]
    if df.empty:
        return pd.DataFrame()
    sample_cols = [col for col in df.columns if col != 'event_id']
    rename = {col: site__standardize_sample_id(col) for col in sample_cols}
    df = df.rename(columns=rename)
    value_cols = [col for col in df.columns if col != 'event_id' and col]
    values = df[value_cols].apply(pd.to_numeric, errors='coerce')
    values.insert(0, 'event_id', df['event_id'].values)
    return values.drop_duplicates('event_id').set_index('event_id')

def site__read_rmats_samples(project_root: Path, cohort_key: str) -> list[str]:
    rel = site_SE_SAMPLE_FILES.get(cohort_key)
    if rel is None:
        return []
    path = project_root / rel
    if not path.exists():
        return []
    samples = []
    for item in path.read_text().strip().split(','):
        sample = site__standardize_sample_id(item)
        if sample:
            samples.append(sample)
    return samples

def site_load_se_usage_matrix(project_root: Path, cohort_key: str, event_ids: set[str] | None=None) -> tuple[pd.DataFrame, pd.DataFrame]:
    rel = site_SE_USAGE_FILES.get(cohort_key)
    if rel is None:
        return (pd.DataFrame(), pd.DataFrame())
    path = project_root / rel
    if not path.exists():
        return (pd.DataFrame(), pd.DataFrame())
    header = pd.read_csv(path, nrows=0).columns
    id_col = 'ID' if 'ID' in header else header[0]
    gene_col = 'geneSymbol' if 'geneSymbol' in header else 'GeneID' if 'GeneID' in header else id_col
    inc_col = 'IncLevel1' if 'IncLevel1' in header else ''
    if not inc_col:
        return (pd.DataFrame(), pd.DataFrame())
    df = pd.read_csv(path, usecols=[id_col, gene_col, inc_col], low_memory=False)
    df['event_id'] = df[id_col].map(site__event_key)
    df['gene_symbol'] = df[gene_col].map(base.standardize_symbol)
    if event_ids:
        df = df[df['event_id'].isin(event_ids)]
    df = df[df['event_id'].ne('')]
    samples = site__read_rmats_samples(project_root, cohort_key)
    if df.empty or not samples:
        return (pd.DataFrame(), pd.DataFrame())
    rows = []
    index = []
    genes = []
    for row in df.itertuples(index=False):
        event_id = getattr(row, 'event_id')
        values = str(getattr(row, inc_col)).split(',')
        if len(values) != len(samples):
            continue
        numeric = pd.to_numeric(pd.Series(values), errors='coerce').to_numpy(dtype=float)
        rows.append(numeric)
        index.append(event_id)
        genes.append(getattr(row, 'gene_symbol'))
    if not rows:
        return (pd.DataFrame(), pd.DataFrame())
    usage = pd.DataFrame(rows, index=index, columns=samples)
    usage = usage[~usage.index.duplicated(keep='first')]
    meta = pd.DataFrame({'event_id': index, 'gene_symbol': genes}).drop_duplicates('event_id').set_index('event_id')
    return (usage, meta)

def site_select_model_candidates(project_root: Path, analysis: str=site_ANALYSIS, max_models: int=48) -> pd.DataFrame:
    path = project_root / 'data' / analysis / 'rbp_deep_candidate_priority.tsv'
    candidates = site__read_table(path)
    if candidates.empty:
        rows = []
        for rbp_symbol in sorted(site_FOCUS_RBPS):
            for comparison, cfg in base.COMPARISONS.items():
                rows.append({'comparison': comparison, 'cohort_key': cfg['tumor_key'], 'event_kind': 'APA', 'rbp_symbol': rbp_symbol, 'deep_priority_score': 0, 'candidate_reason': 'focus_rbp_fallback'})
        return pd.DataFrame(rows)
    candidates['rbp_symbol'] = candidates['rbp_symbol'].map(base.standardize_symbol)
    candidates['deep_priority_score'] = site__safe_numeric(candidates.get('deep_priority_score', 0)).fillna(0)
    candidates['candidate_reason'] = 'deep_priority'
    keep = candidates[candidates['suggested_panel_use'].astype(str).ne('backup') | candidates['rbp_symbol'].isin(site_FOCUS_RBPS) | candidates['deep_priority_score'].ge(candidates['deep_priority_score'].quantile(0.7))].copy()
    keep = keep.sort_values(['deep_priority_score', 'n_events'], ascending=[False, False])
    if len(keep) < max_models:
        keep = pd.concat([keep, candidates.sort_values('deep_priority_score', ascending=False).head(max_models)], ignore_index=True)
    keep = keep.drop_duplicates(['comparison', 'event_kind', 'rbp_symbol']).head(max_models)
    cohort_map = {comparison: cfg['tumor_key'] for comparison, cfg in base.COMPARISONS.items()}
    keep['cohort_key'] = keep['comparison'].map(cohort_map)
    return keep[['comparison', 'cohort_key', 'event_kind', 'rbp_symbol', 'deep_priority_score', 'candidate_reason']].copy()

def site__aggregate_lag_events(lag_group: pd.DataFrame, event_kind: str, site_availability: pd.DataFrame | None=None) -> pd.DataFrame:
    if lag_group.empty:
        return pd.DataFrame()
    lag = lag_group.copy()
    event_col = 'apa_event_id' if event_kind == 'APA' else 'event_id'
    gene_col = 'host_gene_symbol' if event_kind == 'APA' else 'gene_symbol'
    lag['event_id'] = lag[event_col].map(site__event_key)
    lag['gene_symbol_model'] = lag[gene_col].map(base.standardize_symbol)
    for col in ['n_merged_peaks', 'n_accessions', 'n_methods', 'n_samples', 'max_score', 'median_score']:
        if col not in lag.columns:
            lag[col] = 0
        lag[col] = site__safe_numeric(lag[col]).fillna(0)
    agg = lag.groupby('event_id', dropna=False).agg(gene_symbol=('gene_symbol_model', 'first'), n_clip_rows=('event_id', 'size'), n_merged_peaks=('n_merged_peaks', 'max'), n_accessions=('n_accessions', 'max'), n_methods=('n_methods', 'max'), n_samples=('n_samples', 'max'), max_score=('max_score', 'max'), median_score=('median_score', 'median')).reset_index()
    agg = agg[agg['event_id'].ne('') & agg['gene_symbol'].ne('')]
    if event_kind == 'APA' and site_availability is not None and (not site_availability.empty):
        site_cols = ['comparison', 'apa_event_key', 'n_seed_families', 'total_mirna_sites', 'n_3utr_clip_rbps', 'n_3utr_clip_links', 'utr_length_quantile', 'regulatory_site_availability_score']
        available_cols = [col for col in site_cols if col in site_availability.columns]
        site = site_availability[available_cols].drop_duplicates(['comparison', 'apa_event_key'])
        comparison = lag_group['comparison'].iloc[0] if 'comparison' in lag_group else ''
        site = site[site['comparison'].eq(comparison)].rename(columns={'apa_event_key': 'event_id'})
        agg = agg.merge(site.drop(columns=['comparison']), on='event_id', how='left')
    for col in ['n_seed_families', 'total_mirna_sites', 'n_3utr_clip_rbps', 'n_3utr_clip_links', 'utr_length_quantile']:
        if col not in agg.columns:
            agg[col] = 0
        agg[col] = site__safe_numeric(agg[col]).fillna(0)
    return agg.sort_values(['event_id', 'gene_symbol'], ascending=[True, True])

def site__fit_ols(y: pd.Series, x: pd.DataFrame) -> dict[str, float]:
    data = pd.concat([y.rename('y'), x], axis=1).dropna()
    if len(data) <= x.shape[1] + 2:
        return {'n': len(data), 'r2': np.nan, 'adj_r2': np.nan, 'coef_last': np.nan, 'p_last': np.nan}
    yv = data['y'].to_numpy(dtype=float)
    xv = data.drop(columns=['y']).to_numpy(dtype=float)
    xv = np.column_stack([np.ones(len(xv)), xv])
    coef = np.linalg.pinv(xv) @ yv
    fitted = xv @ coef
    resid = yv - fitted
    sse = float(np.sum(resid ** 2))
    sst = float(np.sum((yv - np.mean(yv)) ** 2))
    r2 = 1.0 - sse / sst if sst > 0 else np.nan
    p = xv.shape[1]
    dof = len(yv) - p
    adj_r2 = 1.0 - (1.0 - r2) * (len(yv) - 1) / dof if pd.notna(r2) and dof > 0 else np.nan
    p_last = np.nan
    if dof > 0:
        sigma2 = sse / dof
        cov = sigma2 * np.linalg.pinv(xv.T @ xv)
        se = np.sqrt(np.clip(np.diag(cov), 0, np.inf))
        if se[-1] > 0:
            t_stat = coef[-1] / se[-1]
            p_last = float(2 * stats.t.sf(abs(t_stat), dof))
    return {'n': int(len(data)), 'r2': float(r2), 'adj_r2': float(adj_r2) if pd.notna(adj_r2) else np.nan, 'coef_last': float(coef[-1]), 'p_last': p_last}

def site__fit_model_table(model_df: pd.DataFrame) -> dict[str, float]:
    df = model_df.copy()
    required = ['event_id', 'sample_id', 'usage', 'host_expr', 'rbp_expr']
    df = df.dropna(subset=required)
    if len(df) < 100 or df['event_id'].nunique() < 3 or df['sample_id'].nunique() < 10:
        return {'n_observations': int(len(df)), 'n_events_modeled': int(df['event_id'].nunique()), 'n_samples_overlap': int(df['sample_id'].nunique()), 'model_type': 'within_event_centered_ols', 'base_r2': np.nan, 'full_r2': np.nan, 'delta_r2': np.nan, 'beta_rbp_expr_z': np.nan, 'p_value': np.nan}
    event_groups = df.groupby('event_id', dropna=False)
    df['usage_within_event'] = df['usage'] - event_groups['usage'].transform('mean')
    df['host_expr_within_event_z'] = site__zscore(df['host_expr'] - event_groups['host_expr'].transform('mean'))
    df['rbp_expr_within_event_z'] = site__zscore(df['rbp_expr'] - event_groups['rbp_expr'].transform('mean'))
    df = df.dropna(subset=['usage_within_event', 'host_expr_within_event_z', 'rbp_expr_within_event_z'])
    if len(df) < 100 or df['event_id'].nunique() < 3 or df['sample_id'].nunique() < 10:
        return {'n_observations': int(len(df)), 'n_events_modeled': int(df['event_id'].nunique()), 'n_samples_overlap': int(df['sample_id'].nunique()), 'model_type': 'within_event_centered_ols', 'base_r2': np.nan, 'full_r2': np.nan, 'delta_r2': np.nan, 'beta_rbp_expr_z': np.nan, 'p_value': np.nan}
    base = site__fit_ols(df['usage_within_event'], df[['host_expr_within_event_z']])
    full = site__fit_ols(df['usage_within_event'], df[['host_expr_within_event_z', 'rbp_expr_within_event_z']])
    return {'n_observations': int(len(df)), 'n_events_modeled': int(df['event_id'].nunique()), 'n_samples_overlap': int(df['sample_id'].nunique()), 'model_type': 'within_event_centered_ols', 'base_r2': base['r2'], 'full_r2': full['r2'], 'delta_r2': float(full['r2'] - base['r2']) if pd.notna(base['r2']) and pd.notna(full['r2']) else np.nan, 'beta_rbp_expr_z': full['coef_last'], 'p_value': full['p_last']}

def site__event_level_effects(model_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for event_id, grp in model_df.groupby('event_id', dropna=False):
        df = grp.copy()
        if len(df) < 20:
            continue
        df['host_expr_z'] = site__zscore(df['host_expr'])
        df['rbp_expr_z'] = site__zscore(df['rbp_expr'])
        df = df.dropna(subset=['usage', 'host_expr_z', 'rbp_expr_z'])
        if len(df) < 20:
            continue
        base = site__fit_ols(df['usage'], df[['host_expr_z']])
        full = site__fit_ols(df['usage'], df[['host_expr_z', 'rbp_expr_z']])
        rows.append({'event_id': event_id, 'host_gene_symbol': df['host_gene_symbol'].iloc[0], 'n_samples': int(len(df)), 'event_mean_usage': float(df['usage'].mean()), 'event_base_r2': base['r2'], 'event_full_r2': full['r2'], 'event_delta_r2': float(full['r2'] - base['r2']) if pd.notna(base['r2']) and pd.notna(full['r2']) else np.nan, 'event_beta_rbp_expr_z': full['coef_last'], 'event_p_value': full['p_last']})
    out = pd.DataFrame(rows)
    if not out.empty:
        out['event_fdr'] = site__bh_fdr(out['event_p_value'])
    return out

def site__build_model_dataframe(usage: pd.DataFrame, expression: pd.DataFrame, event_info: pd.DataFrame, rbp_symbol: str, event_kind: str, max_events: int | None) -> tuple[pd.DataFrame, dict[str, int]]:
    if usage.empty or expression.empty or event_info.empty or (rbp_symbol not in expression.index):
        return (pd.DataFrame(), {'candidate_events': int(len(event_info)), 'events_with_usage': 0, 'events_with_host_expression': 0, 'overlap_samples': 0})
    samples = [sample for sample in usage.columns if sample in expression.columns]
    if not samples:
        return (pd.DataFrame(), {'candidate_events': int(len(event_info)), 'events_with_usage': 0, 'events_with_host_expression': 0, 'overlap_samples': 0})
    event_info = event_info[event_info['event_id'].isin(usage.index)].copy()
    if max_events is not None and max_events > 0 and (len(event_info) > max_events):
        event_info = event_info.sort_values(['event_id', 'gene_symbol']).head(max_events).copy()
    event_info = event_info[event_info['gene_symbol'].isin(expression.index)].copy()
    if event_info.empty:
        return (pd.DataFrame(), {'candidate_events': int(len(event_info)), 'events_with_usage': 0, 'events_with_host_expression': 0, 'overlap_samples': int(len(samples))})
    rbp_expr = expression.loc[rbp_symbol, samples].astype(float)
    rows = []
    for event in event_info.itertuples(index=False):
        event_id = event.event_id
        host = event.gene_symbol
        usage_values = usage.loc[event_id, samples].astype(float)
        host_expr = expression.loc[host, samples].astype(float)
        tmp = pd.DataFrame({'event_id': event_id, 'event_kind': event_kind, 'host_gene_symbol': host, 'sample_id': samples, 'usage': usage_values.to_numpy(dtype=float), 'host_expr': host_expr.to_numpy(dtype=float), 'rbp_expr': rbp_expr.to_numpy(dtype=float)})
        tmp['event_mean_usage'] = float(tmp['usage'].mean(skipna=True))
        rows.append(tmp)
    if not rows:
        return (pd.DataFrame(), {'candidate_events': int(len(event_info)), 'events_with_usage': 0, 'events_with_host_expression': 0, 'overlap_samples': int(len(samples))})
    model_df = pd.concat(rows, ignore_index=True, sort=False)
    model_df = model_df.dropna(subset=['usage', 'host_expr', 'rbp_expr'])
    return (model_df, {'candidate_events': int(len(event_info)), 'events_with_usage': int(event_info['event_id'].nunique()), 'events_with_host_expression': int(event_info['gene_symbol'].nunique()), 'overlap_samples': int(len(samples))})

def site_plot_sample_model_delta_r2(model_summary: pd.DataFrame, figures_dir: Path) -> list[Path]:
    if model_summary.empty:
        return []
    from PIL import Image, ImageDraw
    plot = model_summary.copy()
    plot['delta_r2_rank'] = site__safe_numeric(plot['delta_r2']).fillna(-np.inf)
    plot = plot.sort_values(['delta_r2_rank', 'n_events_modeled'], ascending=[False, False]).head(24)
    if plot.empty:
        return []
    row_h = 32
    left = 405
    right = 1040
    top = 100
    width = 1220
    height = top + row_h * len(plot) + 105
    img = Image.new('RGB', (width, height), 'white')
    draw = ImageDraw.Draw(img)
    small = site__load_font(13)
    font = site__load_font(16)
    bold = site__load_font(22)
    site__draw_text(draw, (30, 28), 'Within-event PDUI/PSI models: added explanatory power of RBP expression', bold)
    site__draw_text(draw, (30, 58), 'Delta R2 compares full model to within-event host-expression baseline; p-values are exploratory', small, site_PALETTE['gray'])
    max_x = max(float(site__safe_numeric(plot['delta_r2']).max()), 0.01)
    max_x = min(max_x * 1.15, max(max_x, 0.05))
    draw.line((left, top - 10, left, top + row_h * len(plot) + 12), fill=site__hex_to_rgb(site_PALETTE['gray']), width=1)
    for tick in np.linspace(0, max_x, 5):
        x = int(left + tick / max_x * (right - left))
        draw.line((x, top + row_h * len(plot) + 12, x, top + row_h * len(plot) + 18), fill=site__hex_to_rgb(site_PALETTE['dark']), width=1)
        site__draw_text(draw, (x, top + row_h * len(plot) + 25), f'{tick:.3f}', small, anchor='ma')
    for i, row in enumerate(plot.itertuples(index=False)):
        y = top + i * row_h
        label = f"{row.event_kind}  {row.rbp_symbol}  {row.comparison.replace('_vs_', ' vs ')}  events={row.n_events_modeled}"
        site__draw_text(draw, (30, y - 4), label, small)
        delta = 0.0 if pd.isna(row.delta_r2) else max(float(row.delta_r2), 0.0)
        x = int(left + min(delta, max_x) / max_x * (right - left))
        color = site_PALETTE['blue'] if row.event_kind == 'APA' else site_PALETTE['orange']
        draw.rectangle((left, y - 9, x, y + 11), fill=site__hex_to_rgb(color))
        direction = '+' if pd.notna(row.beta_rbp_expr_z) and row.beta_rbp_expr_z >= 0 else '-'
        fdr = row.model_fdr
        fdr_label = f'FDR={fdr:.2g}' if pd.notna(fdr) else 'FDR=NA'
        site__draw_text(draw, (right + 18, y - 4), f'{direction} {fdr_label}', small, site_PALETTE['gray'])
    site__draw_text(draw, ((left + right) // 2, height - 35), 'Incremental R2 from sample-level RBP expression', font, anchor='ma')
    return site__save_png_pdf(img, figures_dir, 'rbp_sample_level_usage_model_delta_r2')

def site_run_sample_level_models(project_root: str | os.PathLike[str] | None=None, analysis: str=site_ANALYSIS, make_plots: bool=True, max_models: int=48, max_events_per_model: int | None=None) -> dict[str, pd.DataFrame | list[Path]]:
    project_root = site_resolve_project_root(project_root)
    output_dir = project_root / 'data' / analysis
    figures_dir = project_root / 'data' / 'figures' / analysis
    output_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)
    site_path = output_dir / 'rbp_apa_regulatory_site_availability.tsv'
    site_availability = site__read_table(site_path) if site_path.exists() else site_build_site_availability(project_root, analysis)
    apa_lags = site__read_table(output_dir / 'rbp_apa_phase_lags.tsv')
    se_lags = site__read_table(output_dir / 'rbp_se_phase_lags.tsv')
    candidates = site_select_model_candidates(project_root, analysis, max_models=max_models)
    expression_cache: dict[str, pd.DataFrame] = {}
    usage_cache: dict[tuple[str, str], tuple[pd.DataFrame, pd.DataFrame]] = {}
    summary_rows = []
    event_effect_frames = []
    manifest_rows = []
    for candidate in candidates.itertuples(index=False):
        comparison = candidate.comparison
        cohort_key = candidate.cohort_key
        event_kind = candidate.event_kind
        rbp_symbol = base.standardize_symbol(candidate.rbp_symbol)
        lag_source = apa_lags if event_kind == 'APA' else se_lags
        if lag_source.empty:
            continue
        lag_group = lag_source[lag_source['comparison'].eq(comparison) & lag_source['cohort_key'].eq(cohort_key) & lag_source['rbp_symbol'].map(base.standardize_symbol).eq(rbp_symbol)].copy()
        if lag_group.empty:
            continue
        event_info = site__aggregate_lag_events(lag_group, event_kind, site_availability)
        if event_info.empty:
            continue
        event_ids = set(event_info['event_id'].astype(str))
        if cohort_key not in expression_cache:
            expression_cache[cohort_key] = site_load_expression_matrix(project_root, cohort_key)
        expression = expression_cache[cohort_key]
        usage_key = (cohort_key, event_kind)
        if usage_key not in usage_cache:
            if event_kind == 'APA':
                usage_cache[usage_key] = (site_load_apa_usage_matrix(project_root, cohort_key, event_ids=None), pd.DataFrame())
            else:
                usage_cache[usage_key] = site_load_se_usage_matrix(project_root, cohort_key, event_ids=None)
        usage, se_meta = usage_cache[usage_key]
        if event_kind == 'SE' and (not se_meta.empty):
            event_info = event_info.merge(se_meta[['gene_symbol']].rename(columns={'gene_symbol': 'se_usage_gene_symbol'}), left_on='event_id', right_index=True, how='left')
            event_info['gene_symbol'] = event_info['gene_symbol'].where(event_info['gene_symbol'].ne(''), event_info['se_usage_gene_symbol'].fillna(''))
            event_info = event_info.drop(columns=['se_usage_gene_symbol'])
        model_df, counts = site__build_model_dataframe(usage=usage, expression=expression, event_info=event_info, rbp_symbol=rbp_symbol, event_kind=event_kind, max_events=max_events_per_model)
        fit = site__fit_model_table(model_df)
        summary_row = {'comparison': comparison, 'cohort_key': cohort_key, 'event_kind': event_kind, 'usage_metric': 'PDUI' if event_kind == 'APA' else 'PSI', 'rbp_symbol': rbp_symbol, 'candidate_reason': candidate.candidate_reason, 'deep_priority_score': float(candidate.deep_priority_score), 'n_candidate_events': counts['candidate_events'], 'n_events_with_usage': counts['events_with_usage'], 'n_events_with_host_expression': counts['events_with_host_expression'], 'n_overlap_samples_available': counts['overlap_samples'], **fit}
        summary_rows.append(summary_row)
        event_effects = site__event_level_effects(model_df)
        if not event_effects.empty:
            event_effects.insert(0, 'rbp_symbol', rbp_symbol)
            event_effects.insert(0, 'usage_metric', 'PDUI' if event_kind == 'APA' else 'PSI')
            event_effects.insert(0, 'event_kind', event_kind)
            event_effects.insert(0, 'cohort_key', cohort_key)
            event_effects.insert(0, 'comparison', comparison)
            event_effect_frames.append(event_effects)
        manifest_rows.append({'comparison': comparison, 'cohort_key': cohort_key, 'event_kind': event_kind, 'rbp_symbol': rbp_symbol, 'candidate_events': counts['candidate_events'], 'events_modeled': fit['n_events_modeled'], 'samples_modeled': fit['n_samples_overlap'], 'observations_modeled': fit['n_observations']})
    model_summary = pd.DataFrame(summary_rows)
    if not model_summary.empty:
        model_summary['model_fdr'] = site__bh_fdr(model_summary['p_value'])
        model_summary['rbp_effect_direction'] = 'flat_or_unstable'
        model_summary.loc[model_summary['beta_rbp_expr_z'].ge(0), 'rbp_effect_direction'] = 'higher_rbp_higher_usage'
        model_summary.loc[model_summary['beta_rbp_expr_z'].lt(0), 'rbp_effect_direction'] = 'higher_rbp_lower_usage'
        model_summary = model_summary.sort_values(['delta_r2', 'n_events_modeled'], ascending=[False, False])
    event_effects = pd.concat(event_effect_frames, ignore_index=True, sort=False) if event_effect_frames else pd.DataFrame()
    if not event_effects.empty:
        event_effects['global_event_fdr'] = site__bh_fdr(event_effects['event_p_value'])
        event_effects = event_effects.sort_values(['event_delta_r2', 'n_samples'], ascending=[False, False])
    summary_path = site__write_tsv(model_summary, output_dir / 'rbp_sample_level_usage_model_summary.tsv')
    event_path = site__write_tsv(event_effects, output_dir / 'rbp_sample_level_usage_model_event_effects.tsv')
    figures = site_plot_sample_model_delta_r2(model_summary, figures_dir) if make_plots else []
    manifest = pd.DataFrame(manifest_rows)
    caveat = pd.DataFrame([{'comparison': 'all', 'cohort_key': 'tumor_cohorts_only', 'event_kind': 'APA_SE', 'rbp_symbol': 'model_caveat', 'candidate_events': len(candidates), 'events_modeled': int(model_summary['n_events_modeled'].sum()) if not model_summary.empty else 0, 'samples_modeled': int(model_summary['n_samples_overlap'].max()) if not model_summary.empty else 0, 'observations_modeled': int(model_summary['n_observations'].sum()) if not model_summary.empty else 0, 'detail': 'Exploratory within-event centered OLS; event-sample observations are not independent. Event-level targetability/mean-usage effects are absorbed by within-event centering. Normal-cohort sample-level modeling is not run because current CLIP-supported phase-lag candidates are tumor-cohort keyed.'}])
    manifest = pd.concat([manifest, caveat], ignore_index=True, sort=False)
    manifest_path = site__write_tsv(manifest, output_dir / 'rbp_sample_level_usage_model_manifest.tsv')
    return {'model_summary': model_summary, 'event_effects': event_effects, 'model_manifest': manifest, 'figures': figures, 'paths': site_SampleModelOutputs(summary_path, event_path, manifest_path)}


In [ ]:
# Cell 10: Activate the visible site-model namespace and publication output mapping.
for _name, _value in list(globals().items()):
    if _name.startswith('site_'):
        setattr(site, _name.removeprefix('site_'), _value)

def _publication_dirs(project_root: Path, analysis: str = ANALYSIS_SLUG) -> dict[str, Path]:
    return {'output': TABLE_DIR, 'figures': FIGURE_DIR, 'logs': LOG_DIR}
base.ensure_output_dirs = _publication_dirs


In [ ]:
# Cell 11: Input provenance and availability audit
REQUIRED_INPUTS = {
    'expression_regression': [
        INPUT_ROOT / 'Merged' / 'regression_windowed' / f'all_genes_regression_periodic_{cohort}.txt'
        for cohort in ['TCGA-PAAD_Tumor', 'CPTAC-3_Tumor', 'GTEx_Normal']
    ],
    'apa_phase_inputs': [
        INPUT_ROOT / 'polyAPA' / 'comparison' / 'apa_expression_phase_lag_tcga_vs_gtex.tsv',
        INPUT_ROOT / 'polyAPA' / 'comparison' / 'apa_expression_phase_lag_cptac_vs_gtex.tsv',
    ],
    'apa_membership_inputs': [
        INPUT_ROOT / 'polyAPA' / 'comparison' / 'apa_host_gene_set_membership_tcga_vs_gtex.tsv',
        INPUT_ROOT / 'polyAPA' / 'comparison' / 'apa_host_gene_set_membership_cptac_vs_gtex.tsv',
    ],
    'se_variable_inputs': [
        INPUT_ROOT / 'AS' / cohort / 'filtered_events' / f'{cohort}_SE_variable_AS.csv'
        for cohort in ['TCGA', 'CPTAC']
    ],
    'se_canonical_inputs': [
        INPUT_ROOT / 'AS' / cohort_dir / 'rhythmic_results' / cohort_key / 'SE' / 'all_results.csv'
        for cohort_dir, cohort_key in [('TCGA', 'TCGA_Tumor'), ('CPTAC', 'CPTAC_Tumor')]
    ],
    'utr_reference': [INPUT_ROOT / 'Ref' / 'hg38_extracted_3UTR.bed'],
    'reactome_reference': [FIG05_REACTOME_FILE],
}
REQUIRED_INPUTS.update({
    'expression_tpm': [INPUT_ROOT / 'Merged' / f'Group_TPM_{cohort}.csv' for cohort in ['TCGA-PAAD_Tumor', 'CPTAC-3_Tumor']],
    'apa_usage': [INPUT_ROOT / 'polyAPA' / cohort / 'filtered_apa' / f'{cohort}_variable_apa_events.csv' for cohort in ['TCGA', 'CPTAC']],
    'se_sample_order': [INPUT_ROOT / 'AS' / cohort / 'all_samples.txt' for cohort in ['TCGA', 'CPTAC']],
    'mirna_site_burden': [INPUT_ROOT / 'apa_mirna_rhythmic_connection' / 'apa_mirna_gene_triads.tsv'],
})
CLIP_SNAPSHOT = INPUT_ROOT / 'rbp_rna_processing_links' / 'clip_supported_rbp_target_links_consensus.tsv'
REQUIRED_INPUTS['clip_input'] = [CLIP_SNAPSHOT] if USE_REVIEWED_CLIP_SNAPSHOT else [INPUT_ROOT / 'Ref' / 'clip_rbp' / 'POSTAR3_human_hg38.txt.gz']
def input_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
def portable_input_path(path):
    return str(path.relative_to(PROJECT_ROOT)) if path.is_relative_to(PROJECT_ROOT) else str(path)
input_audit = pd.DataFrame([
    {'input_class': group, 'path': portable_input_path(path), 'exists': path.exists(),
     'bytes': path.stat().st_size if path.exists() and path.is_file() else np.nan}
    for group, paths in REQUIRED_INPUTS.items() for path in paths
])
input_audit.to_csv(TABLE_DIR / 'fig05_input_availability.tsv', sep='\t', index=False)
display(input_audit)
assert input_audit['exists'].all(), 'Resolve missing primary inputs before regeneration.'

# Fingerprint exactly the direct inputs used by this configured run.
input_hashes = pd.DataFrame([
    {'input_class': group, 'path': portable_input_path(path), 'bytes': path.stat().st_size, 'sha256': input_sha256(path)}
    for group, paths in REQUIRED_INPUTS.items() for path in paths
])
input_hashes.to_csv(TABLE_DIR / 'fig05_input_sha256.tsv', sep='\t', index=False)
clip_snapshot_manifest = input_hashes.query("input_class == 'clip_input'").copy()
clip_snapshot_manifest['mode'] = 'processed_snapshot' if USE_REVIEWED_CLIP_SNAPSHOT else 'raw_rebuild'
clip_snapshot_manifest['interpretation'] = 'Snapshot freshness relative to raw POSTAR3 is not asserted.' if USE_REVIEWED_CLIP_SNAPSHOT else 'Consensus links regenerated from raw reference.'
clip_snapshot_manifest.to_csv(TABLE_DIR / 'fig05_clip_snapshot_manifest.tsv', sep='\t', index=False)


In [ ]:
# Cell 12: Regenerate RBP, APA, SE, CLIP, and Reactome layers.
# Copy the reviewed processed consensus snapshot into this publication namespace.
# This is explicit preservation of the reviewed mapping. To rebuild raw POSTAR3 overlaps, set
# USE_REVIEWED_CLIP_SNAPSHOT = False; that route can take substantial time and writes its own source manifest.
if USE_REVIEWED_CLIP_SNAPSHOT:
    (TABLE_DIR / CLIP_SNAPSHOT.name).write_bytes(CLIP_SNAPSHOT.read_bytes())

_stage_started = time.perf_counter()
source_results = source.generate_current_source_tables(
    project_root=PROJECT_ROOT, analysis=ANALYSIS_SLUG,
    n_lag_permutations=500, use_clip_cache=USE_REVIEWED_CLIP_SNAPSHOT,
)
source_manifest = source_results['manifest']
pd.DataFrame([{'stage': 'source_tables', 'seconds': time.perf_counter() - _stage_started,
               'clip_mode': 'processed_snapshot' if USE_REVIEWED_CLIP_SNAPSHOT else 'raw_rebuild'}]).to_csv(
    LOG_DIR / 'fig05_analysis_stage_timing.tsv', sep='\t', index=False
)
display(source_manifest.groupby('stage', dropna=False).agg(n_tables=('table', 'nunique'), rows=('rows', 'sum')))


In [ ]:
# Cell 13: Regenerate UTR-rank, phase-module, regulatory-site, and exploratory sample-level-model tables.
_timing_rows = []
_stage_started = time.perf_counter()
deep_results = deep.run_deep_dive(
    project_root=PROJECT_ROOT, analysis=ANALYSIS_SLUG, n_permutations=2000, make_plots=False
)
_timing_rows.append({'stage': 'deep_dive', 'seconds': time.perf_counter() - _stage_started})
_stage_started = time.perf_counter()
site_results = site.run_site_availability(
    project_root=PROJECT_ROOT, analysis=ANALYSIS_SLUG, make_plots=False
)
_timing_rows.append({'stage': 'site_availability', 'seconds': time.perf_counter() - _stage_started})
# The within-event OLS screen is exploratory; 48 candidates match the reviewed workflow.
_stage_started = time.perf_counter()
usage_results = site.run_sample_level_models(
    project_root=PROJECT_ROOT, analysis=ANALYSIS_SLUG, make_plots=False,
    max_models=48, max_events_per_model=None
)
_timing_rows.append({'stage': 'sample_level_models_48', 'seconds': time.perf_counter() - _stage_started})
pd.concat([pd.read_csv(LOG_DIR / 'fig05_analysis_stage_timing.tsv', sep='\t'), pd.DataFrame(_timing_rows)], ignore_index=True).to_csv(
    LOG_DIR / 'fig05_analysis_stage_timing.tsv', sep='\t', index=False
)
display(usage_results['model_summary'].head(12))


In [ ]:
# Cell 14: Canonical source-table audit. Figure-specific plotting cells consume these tables.
CANONICAL_TABLES = sorted(set(source.CURRENT_PANEL_SOURCE_TABLES + [
    'rbp_processing_source_table_generation_manifest.tsv',
    'rbp_apa_utr_rank_annotation.tsv', 'rbp_apa_utr_rank_bias.tsv',
    'rbp_event_phase_module_summary.tsv', 'rbp_se_region_distribution.tsv',
    'rbp_se_region_bias.tsv', 'rbp_deep_candidate_priority.tsv',
    'rbp_deep_dive_manifest.tsv', 'rbp_apa_regulatory_site_availability.tsv',
    'rbp_apa_regulatory_site_bias.tsv', 'rbp_apa_regulatory_site_manifest.tsv',
    'rbp_sample_level_usage_model_summary.tsv', 'rbp_sample_level_usage_model_event_effects.tsv',
    'rbp_sample_level_usage_model_manifest.tsv', 'fig05_input_availability.tsv',
    'fig05_clip_snapshot_manifest.tsv'
]))
canonical_audit = pd.DataFrame([{
    'table': name, 'exists': (TABLE_DIR / name).exists(),
    'bytes': (TABLE_DIR / name).stat().st_size if (TABLE_DIR / name).exists() else 0
} for name in CANONICAL_TABLES])
canonical_audit.to_csv(TABLE_DIR / 'fig05_canonical_source_table_audit.tsv', sep='\t', index=False)
display(canonical_audit)
assert canonical_audit['exists'].all() and canonical_audit['bytes'].gt(0).all(), canonical_audit.query('not exists or bytes == 0')


## Figure 5 and Supplementary Figures S14–S15: RBP RNA-processing links

The following figures redraw the audited, panel-ready analysis products in `TABLE_DIR`. Figure 5 is the TCGA-primary story; S14 is descriptive Reactome context (nominal p values, with no FDR-significant term); S15 maps CPTAC panels D, F, H, I, J, K, L, and N into panels A–H. The regulatory-site score is a proxy from miRNA burden, broad UTR CLIP burden, and annotated UTR rank; it is not a direct accessibility measurement.

In [ ]:
# Cell 15: Shared publication panel definitions
from pathlib import Path
from IPython.display import Image, display
from matplotlib.text import Text
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import textwrap
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': FONT['base'], 'axes.titlesize': FONT['title'], 'axes.labelsize': FONT['axis_title'], 'xtick.labelsize': FONT['tick'], 'ytick.labelsize': FONT['tick'], 'legend.fontsize': FONT['legend'], 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'savefig.dpi': 300})
COL = dict(blue='#0072B2', orange='#D55E00', green='#009E73', purple='#CC79A7', gold='#E69F00', sky='#56B4E9', gray='#7A7A7A', light_gray='#D9D9D9', dark='#222222')
COHORT_LABEL = {'TCGA_Tumor': 'TCGA Tumor', 'GTEx_Normal': 'GTEx Normal', 'CPTAC_Tumor': 'CPTAC Tumor'}
COHORT_COLOR = {'TCGA_Tumor': COL['blue'], 'GTEx_Normal': COL['green'], 'CPTAC_Tumor': COL['orange']}
COMP_LABEL = {'TCGA_vs_GTEx': 'TCGA vs GTEx', 'CPTAC_vs_GTEx': 'CPTAC vs GTEx'}
CRITERIA = [('clip_enrichment', 'CLIP\nenrich'), ('target_contrast', 'target\n> bg'), ('lag_coherence', 'lag\ncoherent'), ('cross_cohort', 'cross-\ncohort'), ('mechanistic_feature', 'APA/SE\nfeature'), ('sample_model', 'usage\nmodel')]

def _read(name):
    path = Path(TABLE_DIR) / name
    if not path.is_file():
        raise FileNotFoundError(f'Required Figure 5 table missing: {path}')
    return pd.read_csv(path, sep='\t')

def _num(x):
    return pd.to_numeric(x, errors='coerce')

def _mlog(x, cap=20):
    z = _num(x).replace([np.inf, -np.inf], np.nan)
    return (-np.log10(z.clip(lower=np.nextafter(0, 1)))).where(z.gt(0), 0).clip(0, cap).fillna(0)

def _fdr(x):
    x = float(x) if pd.notna(x) else np.nan
    return '$q$=NA' if not np.isfinite(x) else '$q$<1e-300' if x == 0 else f'$q$={x:.1e}' if x < 0.001 else f'$q$={x:.3f}'

def _short(x, width=38):
    return textwrap.shorten(str(x), width=width, placeholder='...')

def _overlap_color(cat, comparison):
    return (COL['orange'] if comparison == 'CPTAC_vs_GTEx' else COL['blue']) if cat == 'tumor_only' else COL['green'] if cat == 'shared' else COL['sky'] if cat == 'normal_only' else COL['light_gray']

def _style(ax, title, xlabel=None, ylabel=None):
    width_in = ax.figure.bbox.width / ax.figure.dpi
    chars = 23 if width_in < 3 else 38 if width_in < 6 else 68
    ax.set_title('\n'.join(textwrap.fill(line, chars) for line in title.splitlines()), fontsize=FONT['title'], fontweight='bold', pad=4)
    if xlabel:
        ax.set_xlabel(textwrap.fill(xlabel, chars + 3), fontsize=FONT['axis_title'])
    if ylabel:
        ax.set_ylabel(textwrap.fill(ylabel, 38), fontsize=FONT['axis_title'])
    ax.tick_params(labelsize=FONT['tick'])

def _no_data(ax, title, message):
    _style(ax, title)
    ax.axis('off')
    ax.text(0.5, 0.5, message, ha='center', va='center', fontsize=FONT['annotation'], transform=ax.transAxes)
F5 = {'rhythmicity': _read('rbp_rhythmicity_summary.tsv'), 'density': _read('rbp_phase_circular_density.tsv'), 'polarization': _read('rbp_phase_polarization_summary.tsv'), 'expression': _read('rbp_expression_rhythmicity.tsv'), 'overlap': _read('rbp_tumor_normal_rhythmic_overlap.tsv'), 'apa': _read('rbp_apa_clip_enrichment.tsv'), 'se': _read('rbp_se_clip_enrichment.tsv'), 'contrast': _read('rbp_target_background_contrast.tsv'), 'coherence': _read('rbp_phase_lag_coherence.tsv'), 'sample': _read('rbp_sample_level_usage_model_summary.tsv'), 'candidate': _read('rbp_deep_candidate_priority.tsv'), 'region': _read('rbp_se_region_bias.tsv'), 'site': _read('rbp_apa_regulatory_site_bias.tsv'), 'reactome': _read('rbp_overlap_reactome_enrichment_top.tsv')}
for key, frame in F5.items():
    assert not frame.empty, f'Figure 5 source {key} is empty'

def _evidence(comparison):
    cand = F5['candidate'].query('comparison == @comparison').copy()
    assert not cand.empty
    site = F5['site'].query('comparison == @comparison')
    sample = F5['sample'].query('comparison == @comparison')
    cand['rate_difference_for_rank'] = _num(cand['max_rate_difference_y']).fillna(_num(cand['max_rate_difference_x'])).fillna(0)
    cand['clip_enrichment'] = _num(cand['best_enrichment_fdr']).lt(0.05)
    cand['target_contrast'] = _num(cand['best_contrast_fdr']).lt(0.05) & cand['rate_difference_for_rank'].gt(0)
    cand['lag_coherence'] = _num(cand['best_lag_coherence_fdr']).lt(0.05)
    cand['cross_cohort'] = _num(cand['n_validation_replicated_events']).fillna(0).gt(0) | _num(cand['n_sig_enrichment_comparisons']).fillna(0).ge(2)
    site_sig = set(site.loc[_num(site['availability_bias_fdr']).lt(0.05), 'rbp_symbol'])
    cand['mechanistic_feature'] = (cand.event_kind == 'APA') & cand.rbp_symbol.isin(site_sig) | (cand.event_kind == 'SE') & _num(cand['dominant_region_fraction']).ge(0.45) & _num(cand['n_total_region_links']).fillna(0).ge(3)
    smask = _num(sample['model_fdr']).lt(0.05) & _num(sample['delta_r2']).ge(0.001)
    sample_sig = set(zip(sample.loc[smask, 'event_kind'], sample.loc[smask, 'rbp_symbol']))
    cand['sample_model'] = [(r.event_kind, r.rbp_symbol) in sample_sig for r in cand.itertuples()]
    cc = [x[0] for x in CRITERIA]
    cand['evidence_count'] = cand[cc].sum(axis=1)
    cand = cand.sort_values(['evidence_count', 'n_validation_replicated_events', 'n_events', 'rate_difference_for_rank'], ascending=False)
    cand = pd.concat([cand[cand.event_kind == 'APA'].head(7), cand[cand.event_kind == 'SE'].head(5)], ignore_index=True)
    cand = cand.sort_values(['evidence_count', 'event_kind', 'n_validation_replicated_events', 'n_events', 'rate_difference_for_rank'], ascending=[False, True, False, False, False]).reset_index(drop=True)
    cand['display_rank'] = np.arange(1, len(cand) + 1)
    cand['module'] = cand.event_kind + ' ' + cand.rbp_symbol
    return cand
EVIDENCE = {'TCGA_vs_GTEx': _evidence('TCGA_vs_GTEx'), 'CPTAC_vs_GTEx': _evidence('CPTAC_vs_GTEx')}
assert len(EVIDENCE['TCGA_vs_GTEx']) <= 12 and len(EVIDENCE['CPTAC_vs_GTEx']) <= 12

def draw_landscape(sf):
    ax = sf.add_subplot()
    d = F5['rhythmicity'].query("summary_type == 'cohort'").drop_duplicates('cohort_key').set_index('cohort_key').loc[['TCGA_Tumor', 'GTEx_Normal', 'CPTAC_Tumor']].reset_index()
    bars = ax.bar([COHORT_LABEL[x].replace(' ', '\n') for x in d.cohort_key], d.n_rhythmic, color=[COHORT_COLOR[x] for x in d.cohort_key])
    _style(ax, 'Rhythmic RBPs\nacross cohorts', ylabel='Rhythmic RBPs')
    ax.set_ylim(0, max(d.n_rhythmic) * 1.27)
    for b, r in zip(bars, d.itertuples()):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 2, f'{int(r.n_rhythmic)}/{int(r.n_rbps_tested)}', ha='center', fontsize=FONT['annotation'])

def _panel_heading(ax, title, subtitle, polar=False):
    """A bold title with a separate regular-weight, globally sized annotation."""
    from matplotlib.offsetbox import AnchoredOffsetbox, TextArea, VPacker
    width_in = ax.figure.bbox.width / ax.figure.dpi
    chars = 23 if width_in < 3 else 38 if width_in < 6 else 68
    title = '\n'.join(textwrap.fill(line, chars) for line in title.splitlines())
    block = VPacker(children=[
        TextArea(title, textprops=dict(fontsize=FONT['title'], fontweight='bold', ha='center')),
        TextArea(subtitle, textprops=dict(fontsize=FONT['annotation'], fontweight='normal', ha='center')),
    ], align='center', pad=0, sep=2)
    heading = AnchoredOffsetbox(loc='lower center', child=block, frameon=False,
        bbox_to_anchor=(0.5, 1.20 if polar else 1.02), bbox_transform=ax.transAxes,
        pad=0, borderpad=0)
    ax.add_artist(heading)
    return heading

def draw_phase(sf, cohort):
    ax = sf.add_subplot(projection='polar')
    den = F5['density'].query('cohort_key == @cohort')
    summ = F5['polarization'].query('cohort_key == @cohort').iloc[0]
    theta = _num(den.theta)
    density = _num(den.density)
    rmax = max(float(density.max()) * 1.18, 0.1)
    color = COHORT_COLOR[cohort]
    ax.plot(theta, density, color=color, lw=1.4)
    ax.fill_between(theta, density, color=color, alpha=0.22)
    pts = F5['expression'].query('cohort_key == @cohort and rhythmic == True')
    t = _num(pts.acrophase_hours).dropna().to_numpy() / 24 * 2 * np.pi
    ax.scatter(t, np.full(len(t), rmax * 0.04), s=7, color=color, alpha=0.55)
    for h in (summ.axis_hours, summ.opposite_axis_hours):
        ax.plot([float(h) / 24 * 2 * np.pi] * 2, [0, rmax], color='.25', ls='--', lw=0.8)
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.set_xticks(np.arange(0, 24, 4) / 24 * 2 * np.pi)
    ax.set_xticklabels(['0/24', '4', '8', '12', '16', '20'], fontsize=FONT['tick'])
    ax.set_yticklabels([])
    ax.set_ylim(0, rmax)
    ax.grid(color='.88', lw=0.6)
    _panel_heading(ax, f'{COHORT_LABEL[cohort]} RBP phase', f'n={int(summ.n)}, R₂={summ.axial_r2:.2f}', polar=True)

def draw_overlap(sf):
    ax = sf.add_subplot()
    order = ['tumor_only', 'shared', 'normal_only', 'not_rhythmic']
    labels = {'shared': 'shared', 'normal_only': 'GTEx only', 'not_rhythmic': 'not rhythmic'}
    for y, comp in enumerate(['TCGA_vs_GTEx', 'CPTAC_vs_GTEx']):
        left = 0
        for cat in order:
            z = F5['overlap'].query('comparison == @comp and category == @cat')
            v = int(z.n_rbps.iloc[0]) if len(z) else 0
            ax.barh(y, v, left=left, color=_overlap_color(cat, comp), ec='white')
            if v > 7:
                ax.text(left + v / 2, y, str(v), ha='center', va='center', fontsize=FONT['annotation'], color='white' if cat in ('tumor_only', 'shared') else COL['dark'])
            left += v
    _style(ax, 'Tumor-normal rhythmicity classes', xlabel='RBPs tested')
    ax.set_yticks([0, 1], ['TCGA vs GTEx', 'CPTAC vs GTEx'])
    ax.invert_yaxis()
    ax.legend(handles=[Patch(fc=COL['blue'], label='TCGA only'), Patch(fc=COL['orange'], label='CPTAC only'), Patch(fc=COL['green'], label='shared'), Patch(fc=COL['sky'], label='GTEx only'), Patch(fc=COL['light_gray'], label='not rhythmic')], ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.22), fontsize=FONT['legend'])

def draw_enrich(sf, comparison, kind):
    ax = sf.add_subplot()
    d = (F5['apa'] if kind == 'APA' else F5['se']).query('comparison == @comparison').copy()
    d['neglog'] = _mlog(d.fdr)
    d = d[_num(d.fdr).le(0.05) & _num(d.odds_ratio).gt(0)].copy()
    title = f"{('TCGA' if comparison.startswith('TCGA') else 'CPTAC')} {kind}: CLIP-supported RBP enrichment"
    if kind == 'SE':
        title = f"{('TCGA' if comparison.startswith('TCGA') else 'CPTAC')} SE:\nCLIP-supported RBP\nenrichment"
    if d.empty:
        return _no_data(ax, title, 'No positive enrichment at $q$ ≤ 0.05.')
    finite = _num(d.odds_ratio).replace([np.inf, -np.inf], np.nan).dropna()
    cap = max(float(finite.max()) * 1.15, 1.0) if len(finite) else 1.0
    d['or_infinite'] = ~np.isfinite(_num(d.odds_ratio))
    d['or_display'] = _num(d.odds_ratio).replace([np.inf, -np.inf], cap)
    d = d.sort_values(['or_display', 'neglog'], ascending=False).head(10)
    y = np.arange(len(d))
    norm = Normalize(0, max(float(d.neglog.max()), 1))
    ax.barh(y, d.or_display, color=plt.get_cmap('YlOrRd')(norm(d.neglog)))
    ax.set_yticks(y, d.rbp_symbol)
    ax.invert_yaxis()
    xmax = max(float(d.or_display.max()), 0.1)
    ax.set_xlim(0, xmax * 1.32)
    _style(ax, '', xlabel='Fisher exact-test odds ratio')
    _panel_heading(ax, title, '($q$ ≤ 0.05)')
    for yi, r in enumerate(d.itertuples()):
        ax.text(xmax * 1.04, yi, f"{r.neglog:.1f}{('  OR=∞' if r.or_infinite else '')}", fontsize=FONT['annotation'], va='center', color=COL['gray'])
    sm = ScalarMappable(norm=norm, cmap='YlOrRd')
    sm.set_array([])
    cb = plt.colorbar(sm, ax=ax, fraction=0.045, pad=0.02)
    cb.set_label('-log10($q$)', fontsize=FONT['legend'])
    cb.ax.tick_params(labelsize=FONT['tick'])
    ax.grid(axis='x', color='#EEE')
    ax.set_axisbelow(True)

def draw_contrast(sf, comparison):
    ax = sf.add_subplot()
    d = F5['contrast'].query('comparison == @comparison').copy()
    d['neglog'] = _mlog(d.fdr)
    d = d.sort_values(['rate_difference', 'neglog'], ascending=False).head(12)
    y = np.arange(len(d))
    ax.scatter(d.rate_difference, y, s=30 + 30 * d.neglog.clip(0, 4), c=d.event_kind.map({'APA': COL['blue'], 'SE': COL['orange']}))
    ax.axvline(0, color=COL['gray'])
    ax.set_yticks(y, [f'{r.event_kind} | {r.rbp_symbol}' for r in d.itertuples()])
    ax.invert_yaxis()
    _style(ax, f"{('TCGA' if comparison.startswith('TCGA') else 'CPTAC')} target-vs-background rhythmicity contrast", xlabel='Rhythmic-rate difference (CLIP targets - background)')
    ax.legend(handles=[Line2D([0], [0], marker='o', color='w', markerfacecolor=COL['blue'], label='APA'), Line2D([0], [0], marker='o', color='w', markerfacecolor=COL['orange'], label='SE')], fontsize=FONT['legend'])
    ax.grid(axis='x', color='#EEE')

def draw_coherence(sf, comparison):
    ax = sf.add_subplot()
    d = F5['coherence'].query('comparison == @comparison').head(14).copy()
    d['delta_R'] = _num(d['delta_R']) if 'delta_R' in d else _num(d.lag_resultant_length) - _num(d.null_mean_resultant_length)
    d['mlog'] = _num(d['minus_log10_empirical_fdr']) if 'minus_log10_empirical_fdr' in d else _mlog(d.empirical_fdr)
    d['mlog'] = d.mlog.fillna(0)
    ax.scatter(d.n_lags, d.delta_R, s=30 + 35 * d.mlog.clip(0, 4), c=d.event_kind.map({'APA': COL['blue'], 'SE': COL['orange']}), alpha=0.9)
    ax._publication_labels = []
    for r in d.sort_values(['mlog', 'delta_R'], ascending=False).head(8).itertuples():
        label = ax.annotate(f'{r.event_kind} {r.rbp_symbol}', (r.n_lags, r.delta_R), xytext=(3, 4), textcoords='offset points', fontsize=FONT['annotation'], arrowprops=dict(arrowstyle='-', color='#888888', lw=0.4), annotation_clip=True)
        ax._publication_labels.append(label)
    ax.axhline(0, color=COL['gray'])
    ax.set_xscale('log')
    ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda value, position: f'{value:g}'))
    ax.margins(x=0.22, y=0.2)
    _style(ax, f"{('TCGA' if comparison.startswith('TCGA') else 'CPTAC')} phase-lag coherence", xlabel='RBP-event lag count (log scale)', ylabel='Circular concentration over null')
    hs = [ax.scatter([], [], s=30 + 35 * v, color=COL['gray'], label=str(v)) for v in [1, 2, 3]]
    ax.legend(handles=hs, title='-log10($q$)', title_fontsize=FONT['legend'], fontsize=FONT['legend'], loc='upper right')
    ax.grid(color='#EEE')

def draw_sample(sf, comparison, show_legend=True):
    ax = sf.add_subplot()
    d = F5['sample'].query('comparison == @comparison').copy()
    d['delta'] = _num(d.delta_r2).fillna(0)
    d['mlog'] = _mlog(d.model_fdr, cap=10)
    d = d.sort_values('delta', ascending=False).head(10)
    y = np.arange(len(d))
    ax.barh(y, d.delta, color=d.event_kind.map({'APA': COL['blue'], 'SE': COL['orange']}))
    ax.set_yticks(y, [f'{r.event_kind} {r.rbp_symbol}' for r in d.itertuples()])
    ax.invert_yaxis()
    xmax = max(float(d.delta.max()), 0.001)
    ax.set_xlim(0, xmax * 1.55)
    _style(ax, f"{('TCGA' if comparison.startswith('TCGA') else 'CPTAC')} sample-level PDUI/PSI usage models", xlabel='Incremental R² from RBP expression')
    for yi, r in enumerate(d.itertuples()):
        sign = '+' if pd.notna(r.beta_rbp_expr_z) and r.beta_rbp_expr_z >= 0 else '-'
        ax.text(xmax * 1.04, yi, f'{sign} {_fdr(r.model_fdr)}', fontsize=FONT['annotation'], va='center', color=COL['gray'])
        ax.scatter(r.delta, yi, s=22 + 20 * min(float(r.mlog), 5), color='white', edgecolor=COL['dark'], linewidth=0.6, zorder=3)
    if show_legend:
        sf.legend(handles=[Patch(fc=COL['blue'], label='APA PDUI'), Patch(fc=COL['orange'], label='SE PSI')], loc='outside lower center', ncol=2, fontsize=FONT['legend'], frameon=False)
    ax.grid(axis='x', color='#EEE')

def draw_evidence(sf, comparison):
    ax = sf.add_subplot()
    d = EVIDENCE[comparison].sort_values('display_rank')
    y = np.arange(len(d))
    n = len(CRITERIA)
    ax.set_xlim(-0.5, n + 2.4)
    ax.set_ylim(-0.8, len(d) - 0.2)
    ax.invert_yaxis()
    ax.set_yticks(y, d.module)
    ax.set_xticks(range(n), [b for a, b in CRITERIA])
    ax.tick_params(axis='both', length=0)
    _style(ax, f"{('TCGA' if comparison.startswith('TCGA') else 'CPTAC')} RBP-module evidence matrix")
    for yi, r in enumerate(d.itertuples()):
        for xi, (c, _) in enumerate(CRITERIA):
            ax.scatter(xi, yi, s=72 if getattr(r, c) else 40, c=COL['blue'] if getattr(r, c) else 'white', ec='white' if getattr(r, c) else COL['light_gray'], lw=0.8)
        ax.text(n + 0.22, yi, str(int(r.evidence_count)), ha='center', va='center', fontsize=FONT['annotation'])
        ax.text(n + 1.12, yi, str(int(r.n_events)), va='center', fontsize=FONT['annotation'])
    ax.text(n + 0.22, -0.56, 'met', ha='center', fontsize=FONT['annotation'], fontweight='bold')
    ax.text(n + 1.1, -0.56, 'events', fontsize=FONT['annotation'], fontweight='bold')
    [s.set_visible(False) for s in ax.spines.values()]

def draw_modules(sf, comparison):
    ax = sf.add_subplot()
    ev = EVIDENCE[comparison][['event_kind', 'rbp_symbol', 'display_rank']]
    d = F5['candidate'].query('comparison == @comparison').merge(ev, on=['event_kind', 'rbp_symbol']).sort_values('display_rank').head(10)
    y = np.arange(len(d))
    ax.axvline(0, color=COL['gray'])
    for yi, r in enumerate(d.itertuples()):
        a, b = np.clip([r.rbp_to_event_mean_lag_hours, r.event_to_mrna_mean_lag_hours], -12, 12)
        ax.plot([a, b], [yi, yi], color=COL['light_gray'], lw=1.6)
        ax.scatter(a, yi, c=COL['orange'], s=28)
        ax.scatter(b, yi, c=COL['green'], s=28, marker='s')
    ax.set(xlim=(-12, 12), yticks=y, yticklabels=[f'{r.event_kind} {r.rbp_symbol}' for r in d.itertuples()])
    ax.invert_yaxis()
    _style(ax, f"{('TCGA' if comparison.startswith('TCGA') else 'CPTAC')} RBP-event-mRNA phase modules", xlabel='Mean circular lag (h)')
    sf.legend(handles=[Line2D([0], [0], marker='o', color='w', markerfacecolor=COL['orange'], label='RBP → event'), Line2D([0], [0], marker='s', color='w', markerfacecolor=COL['green'], label='event → mRNA')], loc='outside lower center', ncol=2, fontsize=FONT['legend'], frameon=False)
    ax.grid(axis='x', color='#EEE')

def draw_region(sf, comparison):
    ax = sf.add_subplot()
    d = F5['region'].query('comparison == @comparison').sort_values(['dominant_region_fraction', 'n_total_region_links'], ascending=False).head(10)
    title = f"{('TCGA' if comparison.startswith('TCGA') else 'CPTAC')} SE positional CLIP support"
    if d.empty:
        return _no_data(ax, title, 'No positional SE table after rhythmic-event filtering.')
    left = np.zeros(len(d))
    y = np.arange(len(d))
    for c, l, color in [('upstream_flank', 'upstream', COL['blue']), ('exon_body', 'exon', COL['gold']), ('downstream_flank', 'downstream', COL['purple'])]:
        v = _num(d[c]).fillna(0).to_numpy()
        ax.barh(y, v, left=left, color=color, label=l)
        left += v
    ax.set_yticks(y, d.rbp_symbol)
    ax.invert_yaxis()
    ax.set_xlim(0, 1)
    _style(ax, title, xlabel='Fraction of rhythmic SE CLIP links')
    sf.legend(*ax.get_legend_handles_labels(), ncol=3, loc='outside lower center', fontsize=FONT['legend'], frameon=False)

def draw_site(sf, comparison):
    ax = sf.add_subplot()
    d = F5['site'].query('comparison == @comparison').copy()
    d = d.assign(fdr=_num(d.availability_bias_fdr).fillna(1)).sort_values(['fdr', 'delta_regulatory_site_availability_score'], ascending=[True, False]).head(10)
    y = np.arange(len(d))
    ax.barh(y, d.delta_regulatory_site_availability_score, color=COL['blue'])
    ax.set_yticks(y, d.rbp_symbol)
    ax.invert_yaxis()
    _style(ax, f"{('TCGA' if comparison.startswith('TCGA') else 'CPTAC')} APA regulatory-site availability", xlabel='Bound - background site-availability score')
    xmax = max(float(_num(d.delta_regulatory_site_availability_score).max()), 0.05)
    ax.set_xlim(0, xmax * 1.35)
    for yi, r in enumerate(d.itertuples()):
        ax.text(r.delta_regulatory_site_availability_score + xmax * 0.02, yi, _fdr(r.availability_bias_fdr), fontsize=FONT['annotation'], va='center', color=COL['gray'])
    ax.grid(axis='x', color='#EEE')

def draw_reactome(sf, comparison):
    ax = sf.add_subplot()
    d = F5['reactome'].query('comparison == @comparison').copy()
    d['mlog'] = _mlog(d.p_value, 8)
    d['orx'] = _num(d.odds_ratio).fillna(0)
    d['n'] = _num(d.overlap_count).fillna(0)
    d['fdr'] = _num(d.fdr).fillna(1)
    d = d.sort_values(['p_value', 'orx'], ascending=[True, False]).head(10).iloc[::-1].reset_index(drop=True)
    y = np.arange(len(d))
    ax.scatter(d.mlog, y, s=28 + 22 * d.n.clip(0, 6), c=[_overlap_color(r.category, comparison) for r in d.itertuples()], ec='white', lw=0.6)
    ax.set_yticks(y, [f'{r.category_label}: {_short(r.pathway_name)}' for r in d.itertuples()])
    _style(ax, f"Reactome context: {('TCGA' if comparison.startswith('TCGA') else 'CPTAC')} overlap classes", xlabel='-log10(nominal $p$)')
    ax.axvline(-np.log10(0.05), color=COL['gray'], ls='--', lw=0.8)
    ax.text(-np.log10(0.05) + 0.012, 0.98, '$p$ = 0.05', transform=ax.get_xaxis_transform(), ha='left', va='top', fontsize=FONT['annotation'], color=COL['gray'])
    ax.grid(axis='x', color='#EEE')
PANELS = {'Figure5A': (1, 1, draw_landscape, 'rhythmic_rbp_landscape', 'rhythmicity'), 'Figure5B': (1, 1, lambda sf: draw_phase(sf, 'TCGA_Tumor'), 'tcga_rbp_phase', 'density; polarization; expression'), 'Figure5C': (1, 1, lambda sf: draw_phase(sf, 'GTEx_Normal'), 'gtex_rbp_phase', 'density; polarization; expression'), 'Figure5D': (1, 1, lambda sf: draw_phase(sf, 'CPTAC_Tumor'), 'cptac_rbp_phase', 'density; polarization; expression'), 'Figure5E': (2, 1, draw_overlap, 'tumor_normal_rhythmicity_overlap', 'overlap'), 'Figure5F': (1, 1, lambda sf: draw_enrich(sf, 'TCGA_vs_GTEx', 'APA'), 'tcga_apa_clip_enrichment', 'apa'), 'Figure5G': (1, 1, lambda sf: draw_enrich(sf, 'TCGA_vs_GTEx', 'SE'), 'tcga_se_clip_enrichment', 'se'), 'Figure5H': (2, 1, lambda sf: draw_contrast(sf, 'TCGA_vs_GTEx'), 'tcga_target_background_contrast', 'contrast'), 'Figure5I': (2, 1, lambda sf: draw_coherence(sf, 'TCGA_vs_GTEx'), 'tcga_phase_lag_coherence', 'coherence'), 'Figure5J': (2, 1, lambda sf: draw_sample(sf, 'TCGA_vs_GTEx'), 'tcga_usage_models', 'sample'), 'Figure5K': (2, 2, lambda sf: draw_evidence(sf, 'TCGA_vs_GTEx'), 'tcga_evidence_matrix', 'candidate; site; sample'), 'Figure5L': (2, 1, lambda sf: draw_modules(sf, 'TCGA_vs_GTEx'), 'tcga_phase_modules', 'candidate; evidence'), 'Figure5M': (2, 1, lambda sf: draw_region(sf, 'TCGA_vs_GTEx'), 'tcga_se_positional_clip', 'region'), 'Figure5N': (2, 1, lambda sf: draw_site(sf, 'TCGA_vs_GTEx'), 'tcga_regulatory_site_availability', 'site'), 'FigureS14A': (4, 1, lambda sf: draw_reactome(sf, 'TCGA_vs_GTEx'), 'tcga_reactome_overlap_context', 'reactome'), 'FigureS14B': (4, 1, lambda sf: draw_reactome(sf, 'CPTAC_vs_GTEx'), 'cptac_reactome_overlap_context', 'reactome'), 'FigureS15A': (1, 1, lambda sf: draw_phase(sf, 'CPTAC_Tumor'), 'cptac_rbp_phase', 'density; polarization; expression'), 'FigureS15B': (2, 1, lambda sf: draw_enrich(sf, 'CPTAC_vs_GTEx', 'APA'), 'cptac_apa_clip_enrichment', 'apa'), 'FigureS15C': (2, 1, lambda sf: draw_contrast(sf, 'CPTAC_vs_GTEx'), 'cptac_target_background_contrast', 'contrast'), 'FigureS15D': (2, 1, lambda sf: draw_coherence(sf, 'CPTAC_vs_GTEx'), 'cptac_phase_lag_coherence', 'coherence'), 'FigureS15E': (2, 1, lambda sf: draw_sample(sf, 'CPTAC_vs_GTEx', show_legend=False), 'cptac_usage_models', 'sample'), 'FigureS15F': (2, 2, lambda sf: draw_evidence(sf, 'CPTAC_vs_GTEx'), 'cptac_evidence_matrix', 'candidate; site; sample'), 'FigureS15G': (2, 1, lambda sf: draw_modules(sf, 'CPTAC_vs_GTEx'), 'cptac_phase_modules', 'candidate; evidence'), 'FigureS15H': (2, 1, lambda sf: draw_site(sf, 'CPTAC_vs_GTEx'), 'cptac_regulatory_site_availability', 'site')}
assert len(PANELS) == 24
assert all(w * h in (1, 2, 3, 4) for w, h, *_ in PANELS.values())


In [ ]:
# Cell 16: Individual panels, assembled figures, and inline display
EXPORT_MANIFEST = []

def _tag(sf, letter):
    """Place a PDF-only letter near the upper-left without covering existing text."""
    renderer = sf.figure.canvas.get_renderer()
    occupied = [t.get_window_extent(renderer).expanded(1.03, 1.08) for t in sf.findobj(Text)
                if t.get_visible() and t.get_text().strip()]
    tag = sf.text(0.012, 0.992, letter[-1], transform=sf.transSubfigure,
                  fontsize=FONT['tag'], fontweight='bold', ha='left', va='top')
    candidates = []
    for x in (0.012, 0.045, 0.08, 0.12, 0.16, 0.22):
        for y in (0.992, 0.955, 0.918, 0.88, 0.84):
            tag.set_position((x, y))
            box = tag.get_window_extent(renderer)
            overlap = sum(max(0, min(box.x1, b.x1)-max(box.x0, b.x0)) *
                          max(0, min(box.y1, b.y1)-max(box.y0, b.y0)) for b in occupied)
            distance = ((x-0.012)*sf.bbox.width)**2 + ((y-0.992)*sf.bbox.height)**2
            candidates.append((overlap*1e6+distance, overlap, x, y))
    _, overlap, x, y = min(candidates)
    assert overlap < 0.01, f'No collision-free panel-letter position for {letter}'
    tag.set_position((x, y))
    return tag


def _place_point_labels(fig):
    """Place point labels deterministically after final physical layout is known."""
    renderer = fig.canvas.get_renderer()
    for ax in fig.axes:
        labels = getattr(ax, '_publication_labels', [])
        occupied = []
        if labels and ax.collections:
            points = ax.collections[0]
            for point, size in zip(points.get_offsets(), points.get_sizes()):
                px, py = ax.transData.transform(point)
                radius = np.sqrt(size) * fig.dpi / 144 + 2
                occupied.append(mpl.transforms.Bbox.from_extents(px-radius, py-radius, px+radius, py+radius))
        legend = ax.get_legend()
        if legend is not None:
            occupied.append(legend.get_window_extent(renderer).expanded(1.04, 1.04))
        bounds = ax.get_window_extent(renderer)
        for label in labels:
            choices = []
            for dx in (3, -3, 15, -15, 30, -30):
                for dy in (5, -12, 16, -23, 27, -34):
                    label.set_position((dx, dy))
                    label.set_ha('left' if dx >= 0 else 'right')
                    box = Text.get_window_extent(label, renderer).expanded(1.04, 1.15)
                    outside = max(0, bounds.x0-box.x0)+max(0, box.x1-bounds.x1)+max(0, bounds.y0-box.y0)+max(0, box.y1-bounds.y1)
                    overlap = sum(max(0, min(box.x1,b.x1)-max(box.x0,b.x0))*max(0,min(box.y1,b.y1)-max(box.y0,b.y0)) for b in occupied)
                    choices.append((outside*10000 + overlap*100 + dx*dx+dy*dy, dx, dy))
            _, dx, dy = min(choices)
            label.set_position((dx, dy))
            label.set_ha('left' if dx >= 0 else 'right')
            occupied.append(Text.get_window_extent(label, renderer).expanded(1.04, 1.15))
    fig.canvas.draw()

def _audit(fig, stem):
    fig.canvas.draw()
    _place_point_labels(fig)
    renderer = fig.canvas.get_renderer()
    bad = []
    for t in fig.findobj(Text):
        if t.get_visible() and t.get_text().strip() and (t.get_fontsize() < 7 - 1e-06):
            bad.append((t.get_text(), t.get_fontsize()))
    if bad:
        raise AssertionError(f'{stem}: text below 7 pt: {bad[:3]}')
    return min((t.get_fontsize() for t in fig.findobj(Text) if t.get_visible() and t.get_text().strip()), default=FONT['base'])

def _export(fig, stem, w, h, tags):
    assert w in (1, 2, 3, 4) and int(h) == h and (h >= 1)
    out = Path(FIGURE_DIR) / stem
    out.parent.mkdir(parents=True, exist_ok=True)
    minpt = _audit(fig, stem)
    fig.savefig(out.with_suffix('.png'), dpi=300, facecolor='white')
    fig.canvas.draw()
    handles = [_tag(s, l) for s, l in tags]
    _audit(fig, stem)
    fig.savefig(out.with_suffix('.pdf'), facecolor='white')
    [x.remove() for x in handles]
    EXPORT_MANIFEST.append(dict(stem=stem, width_modules=w, height_modules=h, width_mm=PANEL_MM * w, height_mm=PANEL_MM * h, minimum_text_pt=minpt, png_has_panel_labels=False, pdf_has_panel_labels=True, png=str(out.with_suffix('.png').relative_to(FIGURE_DIR)), pdf=str(out.with_suffix('.pdf').relative_to(FIGURE_DIR))))
    display(Image(filename=str(out.with_suffix('.png')), width=min(1100, round(PANEL_MM * w / 25.4 * 115))))
    plt.close(fig)

def _panel_source(key):
    _, _, _, content, sources = PANELS[key]
    frames = []
    for name in [x.strip() for x in sources.split(';')]:
        if name == 'evidence':
            frames.append(EVIDENCE['CPTAC_vs_GTEx' if key.startswith('FigureS15') else 'TCGA_vs_GTEx'].assign(source_table='in_memory_evidence_selection'))
        else:
            frames.append(F5[name].assign(source_table=name))
    pd.concat(frames, ignore_index=True, sort=False).to_csv(Path(TABLE_DIR) / (key + '_' + content + '_source.tsv'), sep='\t', index=False)

def export_panel(key):
    w, h, draw, content, _ = PANELS[key]
    fig = plt.figure(figsize=(PANEL_MM * w / 25.4, PANEL_MM * h / 25.4), layout='constrained')
    sf = fig.subfigures(1, 1)
    draw(sf)
    _panel_source(key)
    _export(fig, key + '_' + content, w, h, [(sf, key)])

def export_assembly(figure, rows, content):
    widths = [sum((PANELS[k][0] for k in row)) for row in rows]
    assert all((x <= 4 for x in widths)), widths
    heights = [max((PANELS[k][1] for k in row)) for row in rows]
    fig = plt.figure(figsize=(PANEL_MM * 4 / 25.4, PANEL_MM * sum(heights) / 25.4), layout='constrained')
    grid = fig.add_gridspec(sum(heights), 4, wspace=0, hspace=0)
    tags = []
    start = 0
    for row, rh in zip(rows, heights):
        col = 0
        for key in row:
            w, h, draw, _, _ = PANELS[key]
            sf = fig.add_subfigure(grid[start:start + h, col:col + w])
            draw(sf)
            tags.append((sf, key))
            col += w
        start += rh
    _export(fig, figure + '_assembled_' + content, 4, sum(heights), tags)
def export_grid(figure, positions, height, content):
    fig = plt.figure(figsize=(PANEL_MM * 4 / 25.4, PANEL_MM * height / 25.4), layout='constrained')
    grid = fig.add_gridspec(height, 4, wspace=0, hspace=0)
    tags = []
    for key, row, col in positions:
        width, panel_height, draw, _, _ = PANELS[key]
        sf = fig.add_subfigure(grid[row:row + panel_height, col:col + width])
        draw(sf)
        tags.append((sf, key))
    _export(fig, figure + '_assembled_' + content, 4, height, tags)

for key in PANELS:
    export_panel(key)
export_grid('Figure5', [('Figure5A',0,0),('Figure5B',0,1),('Figure5C',0,2),('Figure5D',0,3),
    ('Figure5E',1,0),('Figure5F',1,2),('Figure5G',1,3),('Figure5H',2,0),('Figure5I',2,2),
    ('Figure5J',3,0),('Figure5L',4,0),('Figure5K',3,2),('Figure5M',5,0),('Figure5N',5,2)], 6, 'rbp_rna_processing')
export_assembly('FigureS14', [['FigureS14A'],['FigureS14B']], 'reactome_overlap_context')
export_grid('FigureS15', [('FigureS15A',0,0),('FigureS15B',0,2),('FigureS15C',1,0),('FigureS15D',1,2),
    ('FigureS15E',2,0),('FigureS15G',3,0),('FigureS15F',2,2),('FigureS15H',4,0)], 5, 'cptac_rbp_validation')
manifest = pd.DataFrame(EXPORT_MANIFEST)
assert len(manifest) == 27 and not manifest.stem.duplicated().any() and manifest.minimum_text_pt.ge(7).all()
manifest.to_csv(Path(TABLE_DIR) / 'figure5_s14_s15_export_manifest.tsv', sep='\t', index=False)
display(manifest[['stem','width_mm','height_mm','minimum_text_pt']])
print('Exported 24 panels and 3 assemblies as PNG/PDF.')


## Validation and reproducibility record


In [ ]:
# Cell 17: Export validation, compact analysis QC, and software provenance
expected_panels = {f'Figure5{x}' for x in 'ABCDEFGHIJKLMN'} | {f'FigureS14{x}' for x in 'AB'} | {f'FigureS15{x}' for x in 'ABCDEFGH'}
export_manifest = pd.DataFrame(EXPORT_MANIFEST)
assert len(export_manifest) == 27 and not export_manifest['stem'].duplicated().any()
assert set(export_manifest.loc[~export_manifest.stem.str.contains('assembled'), 'stem'].map(lambda s: Path(s).name.split('_')[0])) == expected_panels
assert export_manifest.minimum_text_pt.ge(7).all()
for field in ['png', 'pdf']:
    assert all((FIGURE_DIR / p).is_file() and (FIGURE_DIR / p).stat().st_size > 0 for p in export_manifest[field])

rhythmicity_qc = pd.read_csv(TABLE_DIR / 'rbp_rhythmicity_summary.tsv', sep='\t').query("summary_type == 'cohort'")
assert set(rhythmicity_qc.cohort_key) == {'TCGA_Tumor', 'GTEx_Normal', 'CPTAC_Tumor'}
assert rhythmicity_qc.n_rhythmic.le(rhythmicity_qc.n_rbps_tested).all()
display(rhythmicity_qc[['cohort_key', 'n_rbps_tested', 'n_rhythmic', 'rhythmic_fraction']])
phase_qc = pd.read_csv(TABLE_DIR / 'rbp_phase_polarization_summary.tsv', sep='\t')
assert phase_qc.axial_r2.between(0, 1).all()
assert phase_qc.axis_hours.between(0, 12, inclusive='left').all()
display(phase_qc)
model_qc = pd.read_csv(TABLE_DIR / 'rbp_sample_level_usage_model_summary.tsv', sep='\t')
assert 0 < len(model_qc) <= 48
assert model_qc['model_fdr'].dropna().between(0, 1).all()
assert model_qc['delta_r2'].dropna().between(-1e-10, 1).all()
display(model_qc.groupby(['comparison', 'event_kind']).size().rename('n_models').reset_index())
qc_columns = [x for x in ['comparison', 'event_kind', 'rbp_symbol', 'n_events_modeled', 'n_samples_overlap', 'n_observations', 'n_overlap_samples_available', 'delta_r2', 'model_fdr'] if x in model_qc.columns]
display(model_qc[qc_columns].head(12))
font_table = pd.DataFrame({'role': list(FONT), 'font_size_pt': list(FONT.values())})
font_table.to_csv(TABLE_DIR / 'publication_font_configuration.tsv', sep='\t', index=False)
parameters = {
    'seed': PUBLICATION_SEED, 'rbp_expression_nominal_p': 0.05, 'rbp_expression_bh_q': 0.05,
    'canonical_se_event_nominal_p': 0.05, 'lag_permutations': 500, 'utr_rank_permutations': 2000,
    'sample_model_max_candidates': 48, 'module_mm': PANEL_MM,
    'clip_snapshot_default': bool(USE_REVIEWED_CLIP_SNAPSHOT),
    'usage_model_interpretation': 'Exploratory within-event OLS; shared samples/events are not independent observations.',
    'site_availability_interpretation': 'Annotation-derived miRNA/CLIP burden and UTR-rank proxy.',
}
(TABLE_DIR / 'analysis_parameters.json').write_text(json.dumps(parameters, indent=2) + '\n')
session = {'python': sys.version, 'platform': platform.platform(), 'elapsed_seconds_this_kernel': round(time.perf_counter() - _started_at, 2), 'analysis_stages': pd.read_csv(LOG_DIR / 'fig05_analysis_stage_timing.tsv', sep='\t').to_dict('records'), 'packages': {x: importlib.metadata.version(x) for x in ['numpy', 'pandas', 'scipy', 'matplotlib', 'statsmodels']}}
(TABLE_DIR / 'session_info.json').write_text(json.dumps(session, indent=2) + '\n')
print('Verified 24 individual panels plus 3 assemblies, in PNG/PDF; all text ≥7 pt.')
print('Source tables:', TABLE_DIR.relative_to(PROJECT_ROOT))
print('Figures:', FIGURE_DIR.relative_to(PROJECT_ROOT))
